## 以上为organ_classification暂时可用版本，确实没问题
### 采用的是软路由，目前最大的误差来源是leaf和flower
### 但是基于现实，leaf和flower是现实图片最大的可能，因此种类分类器采用软路由即可大幅度提高模型精度
### 以下是species_classification 的训练情况，当前误差较大，但记忆中有80%左右，但现在找不到模型结果了
### 全流程尝试1111

In [1]:
# =======================
# Step 0: 配置 & 生成映射
# =======================
import json
from pathlib import Path
import pandas as pd

# === 路径与列名（与你的数据匹配）===
DATA_CSV = "/mnt/e/code/plants-classification-conda/real_data/plant_Brazil.csv"
OUT_DIR  = Path("./real_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_IMAGE     = "image_path"   # 图片路径
COL_ORGAN_TXT = "organ"        # 器官文字：leaf/flower/fruit/bark
COL_ORGAN_ID  = "organ_id"     # 器官数值：如 0/1/2/3（或 1/2/3/4）
COL_LABEL     = "label"        # 植物拉丁学名（文本）
COL_SPECIESID = "species"      # 植物全局 ID（数值，label 的数值化）

# 读取 & 规范化（仅对文字列做清洗；数值列保持数值类型）
df = pd.read_csv(DATA_CSV)
assert all(c in df.columns for c in [COL_IMAGE, COL_ORGAN_TXT, COL_ORGAN_ID, COL_LABEL, COL_SPECIESID]), \
    f"CSV 至少需要列: {COL_IMAGE}, {COL_ORGAN_TXT}, {COL_ORGAN_ID}, {COL_LABEL}, {COL_SPECIESID}"

df[COL_ORGAN_TXT] = df[COL_ORGAN_TXT].astype(str).str.strip().str.lower()
df[COL_LABEL]     = df[COL_LABEL].astype(str).str.strip()

# 1) organ_classes.json（基于 organ↔organ_id 一致性）
pairs = df[[COL_ORGAN_TXT, COL_ORGAN_ID]].drop_duplicates()
cnt1 = pairs.groupby(COL_ORGAN_TXT)[COL_ORGAN_ID].nunique()
cnt2 = pairs.groupby(COL_ORGAN_ID)[COL_ORGAN_TXT].nunique()
if (cnt1 > 1).any() or (cnt2 > 1).any():
    raise ValueError("检测到 organ 与 organ_id 不是一一对应，请检查数据。")

organ_classes = dict(pairs.sort_values(COL_ORGAN_ID).values)  # 文本→id
with open(OUT_DIR / "organ_classes.json", "w", encoding="utf-8") as f:
    json.dump(organ_classes, f, ensure_ascii=False, indent=2)
print("✅ organ_classes.json 已保存：", organ_classes)

# 2) 每器官生成局部映射（训练 head 用）
for organ_txt in organ_classes.keys():
    sub = df[df[COL_ORGAN_TXT] == organ_txt].copy()
    uniq = sub[[COL_LABEL, COL_SPECIESID]].drop_duplicates()
    uniq = uniq.sort_values([COL_SPECIESID, COL_LABEL]).reset_index(drop=True)
    uniq["local_id"] = range(len(uniq))

    label2local = {row[COL_LABEL]: int(row["local_id"]) for _, row in uniq.iterrows()}
    gid2local   = {str(int(row[COL_SPECIESID])): int(row["local_id"]) for _, row in uniq.iterrows()}
    local2gid   = {str(int(row["local_id"])): int(row[COL_SPECIESID]) for _, row in uniq.iterrows()}

    with open(OUT_DIR / f"species_local_map_{organ_txt}.json", "w", encoding="utf-8") as f:
        json.dump(label2local, f, ensure_ascii=False, indent=2)
    with open(OUT_DIR / f"species_global2local_{organ_txt}.json", "w", encoding="utf-8") as f:
        json.dump(gid2local, f, ensure_ascii=False, indent=2)
    with open(OUT_DIR / f"species_local2global_{organ_txt}.json", "w", encoding="utf-8") as f:
        json.dump(local2gid, f, ensure_ascii=False, indent=2)

    print(f"✅ [{organ_txt}] 局部映射已保存：labels={len(label2local)}")


✅ organ_classes.json 已保存： {'bark': 0, 'flower': 1, 'fruit': 2, 'leaf': 3}
✅ [bark] 局部映射已保存：labels=345
✅ [flower] 局部映射已保存：labels=422
✅ [fruit] 局部映射已保存：labels=430
✅ [leaf] 局部映射已保存：labels=444


In [2]:
# =======================
# Step 1: 依赖 & Dataset
# =======================
import os, math
from typing import List, Dict, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from PIL import Image
from torchvision import transforms, models
from torchvision.models import (
    ResNet18_Weights,
    ResNet50_Weights,
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device =", DEVICE)

class BasicImageDataset(Dataset):
    """
    通用图像数据集：通过 label_lookup 决定监督标签的来源（organ 或 species local_id）
    - label_col 可是 organ_id 或 "__species_local__"
    - label_lookup 把【真实值】映射到【训练用的连续id】，这里通常是恒等映射
    """
    def __init__(self, df: pd.DataFrame, image_col: str, label_col: str,
                 label_lookup: Dict[int, int], tfm: transforms.Compose):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.label_lookup = label_lookup
        self.tfm = tfm

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row[self.image_col]).convert("RGB")
        img = self.tfm(img)
        raw_label = int(row[self.label_col])
        label = self.label_lookup[raw_label]
        return img, label


Device = cuda


In [3]:
from torchvision.transforms import AutoAugment, AutoAugmentPolicy, RandomErasing

def make_species_transforms(organ: str, img_size: int):
    organ = organ.lower().strip()

    # 骨架：val 始终 CenterCrop；train 使用 RandomResizedCrop + 适度颜色/几何扰动
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])

    if organ == "flower":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.15)),
            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0), ratio=(0.80, 1.20)),
            transforms.RandomHorizontalFlip(p=0.5),
            AutoAugment(AutoAugmentPolicy.IMAGENET),          # 提升泛化（花色/形状）
            transforms.ColorJitter(0.20, 0.20, 0.12, 0.04),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            RandomErasing(p=0.20, scale=(0.02, 0.10), value='random'),
        ])

    elif organ == "leaf":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0), ratio=(0.85, 1.15)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.18, 0.18, 0.10, 0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            RandomErasing(p=0.25, scale=(0.02, 0.12), value='random'),  # 叶脉鲁棒
        ])

    elif organ == "fruit":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.65, 1.0), ratio=(0.90, 1.10)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),            # ✅ 果实形变/朝向
            transforms.ColorJitter(0.15, 0.15, 0.10, 0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])

    else:  # bark
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.60, 1.0), ratio=(0.95, 1.05)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.10, 0.10, 0.06, 0.02),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])

    # ======= bark 训练增强（更接近评估视野，抑制形变） =======


    return train_tf, val_tf


In [4]:
# =======================
# Step 2: 器官分类器
# =======================

# ================ Cell 1: Imports & Utils ================
import os
import json
import math
import time
import random
from dataclasses import dataclass
from typing import Tuple, List, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report

import timm
from tqdm import tqdm
from timm.utils import ModelEmaV2


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True  # 更快


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


# ================ Cell 2: Config ================
@dataclass
class TrainConfig:
    csv: str = "/mnt/e/code/plants-classification-conda/real_data/plant_Brazil.csv"  # 改成你的 CSV 路径
    data_root: str = "/"                                 # 若 image_path 为绝对路径，保持 "/" 即可
    out: str = "./real_data"          # 输出目录

    backbone: str = 'convnext_small'                        # timm 任意骨干
    pretrained: bool = True
    #img_size: int = 256 原设置为256
    img_size: int = 384
    batch_size: int = 64
    epochs: int = 100
    lr: float = 2e-4
    wd: float = 5e-2
    warmup_epochs: float = 3.0
    num_workers: int = 8
    amp: bool = True
    grad_accum: int = 1
    aux_weight: float = 0.35
    label_smoothing: float = 0.05
    seed: int = 42
    train_split: float = 0.9        # 训练集比例，分层划分
    patience: int = 20              # 早停
    save_every: int = 0             # >0 表示每 N 轮额外保存一次



    # 采样配置（缓解类不平衡）
    use_weighted_sampler: bool = True
    sampler_power: float = 0.7      # 1.0=完全按 1/freq；<1 软化

    # 温度标定（验证集）
    do_temperature_scaling: bool = True

    # 推理用阈值（训练不使用，仅保留）
    confidence_tau: float = 0.55


CFG = TrainConfig()
set_seed(CFG.seed)
ensure_dir(CFG.out)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


# ================ Cell 3: Dataset ================
ORGANS_STD = ["bark", "flower", "fruit", "leaf"]
ORGAN2ID = {k: i for i, k in enumerate(ORGANS_STD)}

# 常见同义归一，可按需扩展
ORGAN_NORMALIZE = {
    'flower': 'flower', 'flw': 'flower', 'flo': 'flower', 'flor': 'flower',
    'leaf': 'leaf', 'leaves': 'leaf', 'foliage': 'leaf',
    'fruit': 'fruit', 'frt': 'fruit', 'seedpod': 'fruit',
    'bark': 'bark', 'trunk': 'bark', 'stem': 'bark', 'branch': 'bark'
}


def normalize_organ(name: str) -> str:
    if name is None:
        return None
    key = str(name).strip().lower()
    return ORGAN_NORMALIZE.get(key, key)


class PlantOrganDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_root: str, img_size: int, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.img_size = img_size
        self.train = train

        '''if train:
            self.tf = transforms.Compose([
                transforms.Resize(int(img_size * 1.15)),
                transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)], p=0.5),
                transforms.ToTensor(),
                transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3)),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
    
            self.tf = transforms.Compose([
                transforms.Resize(int(img_size * 1.05)),
                transforms.CenterCrop(img_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])'''

        if train:
            self.tf = transforms.Compose([
                transforms.Resize(int(img_size * 1.10)),
                transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.85, 1.15)),  # ↑ 提高下限，保细节
                transforms.RandomHorizontalFlip(),
                transforms.RandomApply([
                    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)
                ], p=0.5),
                transforms.RandomApply([
                    transforms.RandomPerspective(distortion_scale=0.2)
                ], p=0.2),
                transforms.RandomGrayscale(p=0.1),
                transforms.ToTensor(),
                transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3)),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
    
            self.tf = transforms.Compose([
                transforms.Resize(int(img_size * 1.05)),
                transforms.CenterCrop(img_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['image_path']
        if not os.path.isabs(path):
            path = os.path.join(self.data_root, path)
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            # 若图片损坏，使用全黑占位，保证训练不中断
            img = Image.fromarray(np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8))
        x = self.tf(img)
        y = int(row['label_id'])
        return x, y


# ================ Cell 4: Model ================
class OrgansHier(nn.Module):
    """共享骨干 + gate/head，输出 4 类概率（软组合）。"""
    def __init__(self, backbone: str = 'convnext_tiny', pretrained: bool = True, drop_path_rate: float = 0.3):
        super().__init__()
        # 可调 drop_path 有助于泛化
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.gate = nn.Linear(feat_dim, 2)     # 0: FL (flower+leaf), 1: FB (fruit+bark)
        self.head_fl = nn.Linear(feat_dim, 2)  # [flower, leaf]
        self.head_fb = nn.Linear(feat_dim, 2)  # [fruit, bark]

    def forward(
        self, x, targets=None, aux_weight: float = 0.3, label_smoothing: float = 0.0, temperature: float = 1.0,
        w_main: torch.Tensor | None = None,
        w_gate: torch.Tensor | None = None,
        w_fl_head: torch.Tensor | None = None,
        w_fb_head: torch.Tensor | None = None,
        gate_margin_weight: float = 0.08, gate_hi: float = 0.7, gate_lo: float = 0.3
    ):
        feats = self.backbone(x)
        gate_logits = self.gate(feats) / temperature
        fl_logits = self.head_fl(feats) / temperature
        fb_logits = self.head_fb(feats) / temperature

        g = F.softmax(gate_logits, dim=1)[:, 0]  # P(FL)
        p_fl = F.softmax(fl_logits, dim=1)       # [flower, leaf]
        p_fb = F.softmax(fb_logits, dim=1)       # [fruit, bark]

        P = torch.stack([
            g * p_fl[:, 0],          # flower
            g * p_fl[:, 1],          # leaf
            (1 - g) * p_fb[:, 0],    # fruit
            (1 - g) * p_fb[:, 1],    # bark
        ], dim=1).clamp_min(1e-8)

        outputs = {
            'probs': P,
            'gate': g,
            'gate_logits': gate_logits,
            'fl_logits': fl_logits,
            'fb_logits': fb_logits,
        }

        if targets is not None:
            # 主损失（四类）— 带类权重
            loss_main = nll_loss_with_label_smoothing(P.log(), targets, smoothing=label_smoothing, weight=w_main)

            # gate 损失（FL/FB）— 带权重
            is_FL = (targets == 0) | (targets == 1)
            gate_t = torch.where(is_FL, torch.zeros_like(targets), torch.ones_like(targets))  # 0:FL,1:FB
            loss_gate = F.cross_entropy(gate_logits, gate_t, weight=w_gate)

            # 子头损失（各自两类）— 带权重
            fl_mask = is_FL
            fb_mask = ~is_FL
            loss_fl = (F.cross_entropy(fl_logits[fl_mask], (targets[fl_mask] % 2), weight=w_fl_head)
                       if fl_mask.any() else torch.tensor(0., device=x.device))
            loss_fb = (F.cross_entropy(fb_logits[fb_mask], ((targets[fb_mask] - 2) % 2), weight=w_fb_head)
                       if fb_mask.any() else torch.tensor(0., device=x.device))

            total_loss = loss_main + aux_weight * (loss_gate + loss_fl + loss_fb)

            # gate margin 正则：让 FL 样本 g 更大、FB 样本 g 更小（轻量）
            if gate_margin_weight > 0:
                loss_gate_margin = 0.
                if fl_mask.any():
                    loss_gate_margin = loss_gate_margin + F.relu(gate_hi - g[fl_mask]).mean()
                if fb_mask.any():
                    loss_gate_margin = loss_gate_margin + F.relu(g[fb_mask] - gate_lo).mean()
                total_loss = total_loss + gate_margin_weight * loss_gate_margin

            outputs['loss'] = total_loss

        return outputs


def nll_loss_with_label_smoothing(
    log_probs: torch.Tensor, targets: torch.Tensor, smoothing: float = 0.0, weight: torch.Tensor | None = None
) -> torch.Tensor:
    """
    log_probs: [B, C] 的对数概率
    weight:   [C] 的类权重（可为 None）
    """
    if smoothing <= 0:
        return F.nll_loss(log_probs, targets, weight=weight)

    n_classes = log_probs.size(-1)
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(smoothing / (n_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)

    per_class_loss = -true_dist * log_probs  # [B,C]
    if weight is not None:
        per_class_loss = per_class_loss * weight.view(1, -1)
    return per_class_loss.sum(dim=1).mean()



# ================ Cell 5: Data Loading Helpers ================

def load_and_split(csv_path: str, train_ratio: float, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(csv_path)
    # 归一化器官
    df['organ_norm'] = df['organ'].apply(normalize_organ)
    df = df[df['organ_norm'].isin(ORGANS_STD)].copy()
    df['label_id'] = df['organ_norm'].map(ORGAN2ID)

    # 分层划分
    train_df, val_df = train_test_split(
        df, test_size=1.0 - train_ratio, random_state=seed, stratify=df['label_id']
    )
    return train_df, val_df


def build_dataloaders(cfg: TrainConfig) -> Tuple[DataLoader, DataLoader, Dict]:
    train_df, val_df = load_and_split(cfg.csv, cfg.train_split, cfg.seed)

    stats = {
        'train_counts': train_df['label_id'].value_counts().to_dict(),
        'val_counts': val_df['label_id'].value_counts().to_dict(),
    }

    train_ds = PlantOrganDataset(train_df, cfg.data_root, cfg.img_size, train=True)
    val_ds = PlantOrganDataset(val_df, cfg.data_root, cfg.img_size, train=False)

    if cfg.use_weighted_sampler:
        counts = train_df['label_id'].value_counts().sort_index().values.astype(float)
        inv = (1.0 / (counts + 1e-6)) ** cfg.sampler_power
        class_weights = inv / inv.sum() * len(counts)
        sample_weights = train_df['label_id'].map(lambda c: class_weights[c]).values
        sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.num_workers, pin_memory=True)

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size*2, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    return train_loader, val_loader, stats


# ================ Cell 6: Optimizer & Scheduler ================

def build_optimizer(model: nn.Module, cfg: TrainConfig):
    # norm/bias 不做 weight decay
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if p.ndimension() == 1 or name.endswith('.bias'):
            no_decay.append(p)
    
            decay.append(p)
    params = [
        {'params': decay, 'weight_decay': cfg.wd},
        {'params': no_decay, 'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(params, lr=cfg.lr)
    return opt


def build_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = int(cfg.warmup_epochs * steps_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return scheduler


# ================ Cell 7: Train & Eval ================

def train_one_epoch(
    model, loader, optimizer, scaler, device, cfg: TrainConfig, ema=None,
    w_main=None, w_gate=None, w_fl_head=None, w_fb_head=None,
    gate_margin_weight: float = 0.05, gate_hi: float = 0.6, gate_lo: float = 0.4
):
    model.train()
    total_loss = 0.0
    n = 0
    pbar = tqdm(loader, desc='train', leave=False)
    optimizer.zero_grad(set_to_none=True)

    for i, (x, y) in enumerate(pbar):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=cfg.amp):
            out = model(
                x, targets=y,
                aux_weight=cfg.aux_weight,
                label_smoothing=cfg.label_smoothing,
                w_main=w_main, w_gate=w_gate, w_fl_head=w_fl_head, w_fb_head=w_fb_head,
                gate_margin_weight=gate_margin_weight, gate_hi=gate_hi, gate_lo=gate_lo
            )
            loss = out['loss'] / cfg.grad_accum

        if cfg.amp:
            scaler.scale(loss).backward()
    
            loss.backward()

        if (i + 1) % cfg.grad_accum == 0:
            if cfg.amp:
                scaler.step(optimizer)
                scaler.update()
        
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            if ema is not None:
                ema.update(model)

        total_loss += float(loss.item()) * x.size(0) * cfg.grad_accum
        n += x.size(0)
        pbar.set_postfix(loss=f"{total_loss / max(1, n):.4f}")

    return total_loss / max(1, n)


def evaluate(model, loader, device, cfg: TrainConfig) -> Dict:
    model.eval()
    all_probs, all_targets, all_logits = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # TTA: 原图
            out1 = model(x, targets=None)
            probs1 = out1['probs']

            # TTA: 水平翻转
            x_flip = torch.flip(x, dims=[3])
            out2 = model(x_flip, targets=None)
            probs2 = out2['probs']

            probs = ((probs1 + probs2) * 0.5).clamp_min(1e-8)
            all_probs.append(probs.detach().cpu())
            all_targets.append(y.detach().cpu())
            all_logits.append(probs.log().detach().cpu())  # “伪 logits”供温度标定

    probs = torch.cat(all_probs, dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy()
    logits = torch.cat(all_logits, dim=0).numpy()

    preds = probs.argmax(axis=1)
    acc = (preds == targets).mean()
    bal_acc = balanced_accuracy_score(targets, preds)
    macro_f1 = f1_score(targets, preds, average='macro')
    cm = confusion_matrix(targets, preds, labels=[0,1,2,3])
    rep = classification_report(targets, preds, target_names=ORGANS_STD, digits=4)

    return {
        'acc': float(acc),
        'bal_acc': float(bal_acc),
        'macro_f1': float(macro_f1),
        'cm': cm.tolist(),
        'report': rep,
        'probs': probs,
        'targets': targets,
        'logits': logits,
    }


def make_weights(counts, epoch, total_epochs, cap_ratio=2.0, alpha_final=0.8, warmup_ep=5):
    # 1) 退火：前 warmup_ep 轮从 0 -> alpha_final 线性增长
    if epoch <= warmup_ep:
        alpha = alpha_final * (epoch / max(1, warmup_ep))

        alpha = alpha_final
    inv = 1.0 / (counts + 1e-6)
    w = (inv ** alpha)
    # 2) 限幅：少数类/多数类的比值不超过 cap_ratio（例如 2 倍）
    w = w / w.min()
    w = np.clip(w, 1.0, cap_ratio)
    # 3) 归一到均值=1，数值更稳
    w = w / w.mean()
    # gate 的 FL/FB 权重也限幅
    n_FL = counts[0] + counts[1]
    n_FB = counts[2] + counts[3]
    wg = np.array([1.0/max(n_FL,1e-6), 1.0/max(n_FB,1e-6)], dtype=np.float32)
    wg = wg / wg.min()
    wg = np.clip(wg, 1.0, cap_ratio)
    wg = wg / wg.mean()
    return w, wg



# ================ Cell 8: Temperature Scaling (optional) ================
class _TempScalingModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature.clamp_min(1e-3)


def fit_temperature(logits: np.ndarray, targets: np.ndarray, max_iter: int = 200, device: str = 'cuda') -> float:
    """在验证集 logits 上拟合标量温度，最小化 CE。"""
    model = _TempScalingModule().to(device)
    opt = torch.optim.LBFGS(model.parameters(), lr=0.01, max_iter=100)

    x = torch.from_numpy(logits).to(device=device, dtype=torch.float32)
    y = torch.from_numpy(targets).to(device=device, dtype=torch.long)

    def _closure():
        opt.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        return loss

    for _ in range(max_iter):
        opt.step(_closure)
    T = float(model.temperature.detach().cpu().item())
    return T




Device: cuda


In [12]:
# ================ Cell 9: Build Everything & Train (save-best) ================
import os, time, copy
import numpy as np
import torch

train_loader, val_loader, stats = build_dataloaders(CFG)
print("Class counts (train):", stats['train_counts'])
print("Class counts (val): ", stats['val_counts'])

# 准备 counts（np.array），供 make_weights 使用
counts_np = np.array([stats['train_counts'].get(i, 1) for i in range(4)], dtype=np.float32)

model = OrgansHier(CFG.backbone, CFG.pretrained, drop_path_rate=0.2).to(DEVICE)
optimizer = build_optimizer(model, CFG)
steps_per_epoch = max(1, len(train_loader))
scheduler = build_scheduler(optimizer, CFG, steps_per_epoch)
scaler = torch.amp.GradScaler('cuda', enabled=CFG.amp)
ema = ModelEmaV2(model, decay=0.9999)

# 保存目录与文件名
save_dir = CFG.out
os.makedirs(save_dir, exist_ok=True)
best_path = os.path.join(save_dir, "organ_classification_mybest.pth")
last_path = os.path.join(save_dir, "organ_classification_last.pth")

best_metric = -1.0
best_epoch  = -1
best_state_dict = None
best_pack_meta = None
no_improve = 0

def _to_head_weights(w4):
    """
    从四类权重派生出两个子头的权重，并各自归一为均值=1（数值更稳）。
    w4: [w_flower, w_leaf, w_fruit, w_bark] 的 numpy 数组（均值=1）
    返回：w_fl_head(2,), w_fb_head(2,)
    """
    w_fl = np.array([w4[0], w4[1]], dtype=np.float32)
    w_fb = np.array([w4[2], w4[3]], dtype=np.float32)
    w_fl = w_fl / max(w_fl.mean(), 1e-6)
    w_fb = w_fb / max(w_fb.mean(), 1e-6)
    return w_fl, w_fb

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()

    # —— 每个 epoch 计算当轮的退火权重（主/子头/gate）——
    w_main_np, w_gate_np = make_weights(
        epoch=epoch,
        total_epochs=CFG.epochs,
        counts=counts_np,
        cap_ratio=1.8,       # 可调：1.5~3.0
        alpha_final=0.9,     # 可调：0.8~1.2
        warmup_ep=6          # 可调：3~8
    )
    w_fl_np, w_fb_np = _to_head_weights(w_main_np)

    # 转成 GPU tensor
    w_main    = torch.tensor(w_main_np, device=DEVICE, dtype=torch.float32)  # [4]
    w_gate    = torch.tensor(w_gate_np, device=DEVICE, dtype=torch.float32)  # [2]
    w_fl_head = torch.tensor(w_fl_np,  device=DEVICE, dtype=torch.float32)   # [2]
    w_fb_head = torch.tensor(w_fb_np,  device=DEVICE, dtype=torch.float32)   # [2]

    # gate margin 退火
    base_gate_margin = 0.08
    gm_w = base_gate_margin * min(1.0, epoch / 10.0)
    gate_hi, gate_lo = 0.7, 0.3

    # —— 训练一个 epoch —— 
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scaler, DEVICE, CFG, ema=ema,
        w_main=w_main, w_gate=w_gate, w_fl_head=w_fl_head, w_fb_head=w_fb_head,
        gate_margin_weight=gm_w, gate_hi=gate_hi, gate_lo=gate_lo
    )

    scheduler.step()

    # 用 EMA 评估（带 TTA）
    val_metrics = evaluate(ema.module, val_loader, DEVICE, CFG)
    t1 = time.time()
    print(f"Epoch {epoch:03d}: loss={train_loss:.4f} acc={val_metrics['acc']:.4f} "
          f"bal_acc={val_metrics['bal_acc']:.4f} macro_f1={val_metrics['macro_f1']:.4f} time={t1-t0:.1f}s")

    # 组合评价分（示例：macro_f1 权重更高）
    score = val_metrics['macro_f1'] * 1000 + val_metrics['bal_acc']

    # ======= ★ 出现更优就立刻保存 BEST ★ =======
    if score > best_metric:
        best_metric = score
        best_epoch  = epoch
        no_improve = 0

        # 深拷贝一份稳定的 EMA 权重
        best_state_dict = copy.deepcopy(ema.module.state_dict())

        # 先保存一个“临时无温度标定”的 best（便于断点观察/复现）
        pack = {
            "arch": "OrgansHier",
            "backbone": CFG.backbone,
            "drop_path_rate": 0.3,                  # 与模型结构保持一致
            "state_dict": best_state_dict,          # EMA 的稳定权重
            "organs": ORGANS_STD,                   # ["bark","flower","fruit","leaf"]
            "organ2id": ORGAN2ID,
            "cfg": CFG.__dict__,
            "temperature": 1.0,                     # 暂时 1.0，训练结束再回填
            "best_epoch": best_epoch,
            "best_metric": float(best_metric),
            "val_snapshot": {
                "acc": float(val_metrics['acc']),
                "bal_acc": float(val_metrics['bal_acc']),
                "macro_f1": float(val_metrics['macro_f1']),
            },
        }
        torch.save(pack, best_path)
        print(f"💾 [BEST@{best_epoch}] 临时保存到: {best_path}")
        best_pack_meta = pack  # 记录 pack 元信息


        no_improve += 1

    if no_improve >= CFG.patience:
        print(f"Early stopping at epoch {epoch} (no improvement for {CFG.patience} epochs)")
        break

# ============ 训练结束：保存“最后一次”的权重 ============
torch.save({
    "arch": "OrgansHier",
    "backbone": CFG.backbone,
    "drop_path_rate": 0.3,
    "state_dict": ema.module.state_dict(),
    "organs": ORGANS_STD,
    "organ2id": ORGAN2ID,
    "cfg": CFG.__dict__,
    "temperature": 1.0,  # 最后权重不做温标，主要用于排查
    "last_epoch": epoch,
}, last_path)
print(f"💾 已保存最后一次权重到: {last_path}")

# ============ 用“最佳权重”做最终验证 & 温度标定，并覆盖保存 BEST ============
if best_state_dict is None:
    # 极端情况：从未刷新过 best，就用最后一次
    best_state_dict = ema.module.state_dict()
    best_epoch = epoch
    best_metric = -1

# 复现一个同结构模型装载 best
best_model = OrgansHier(CFG.backbone, CFG.pretrained, drop_path_rate=0.2).to(DEVICE)
best_model.load_state_dict(best_state_dict, strict=True)
best_model.eval()

# 用最佳权重做最终评估，拿 logits/targets
final_eval = evaluate(best_model, val_loader, DEVICE, CFG)

T = 1.0
if CFG.do_temperature_scaling:
    T = fit_temperature(final_eval['logits'], final_eval['targets'], device=DEVICE)
    print(f"Fitted temperature (BEST @ epoch {best_epoch}): {T:.3f}")

# 覆盖保存“可直接上线”的 BEST 包，填入温度
final_pack = {
    "arch": "OrgansHier",
    "backbone": CFG.backbone,                # e.g. 'convnext_small'
    "drop_path_rate": 0.3,
    "state_dict": best_state_dict,           # 最优 EMA 权重
    "organs": ORGANS_STD,                    # ["bark","flower","fruit","leaf"]
    "organ2id": ORGAN2ID,
    "cfg": CFG.__dict__,
    "temperature": float(T),                 # 最优权重对应的温度标定
    "best_epoch": int(best_epoch),
    "best_metric": float(best_metric),
}
torch.save(final_pack, best_path)
print(f"✅ Saved BEST organ classifier to: {best_path}")


Class counts (train): {3: 112679, 1: 32140, 2: 22458, 0: 5967}
Class counts (val):  {3: 12521, 1: 3571, 2: 2495, 0: 663}


Epoch 001: loss=1.9569 acc=0.4809 bal_acc=0.3209 macro_f1=0.3068 time=1478.1s
💾 [BEST@1] 临时保存到: ./real_data/organ_classification_mybest.pth


Epoch 002: loss=1.4961 acc=0.5365 bal_acc=0.4066 macro_f1=0.3783 time=1529.4s
💾 [BEST@2] 临时保存到: ./real_data/organ_classification_mybest.pth


Epoch 003: loss=1.1067 acc=0.6322 bal_acc=0.5517 macro_f1=0.5018 time=2279.5s
💾 [BEST@3] 临时保存到: ./real_data/organ_classification_mybest.pth


Epoch 004: loss=0.9032 acc=0.7209 bal_acc=0.6826 macro_f1=0.6198 time=2535.0s
💾 [BEST@4] 临时保存到: ./real_data/organ_classification_mybest.pth


...

Epoch 096: loss=0.3169 acc=0.9240 bal_acc=0.8970 macro_f1=0.8833 time=2877.3s


Epoch 097: loss=0.3178 acc=0.9249 bal_acc=0.8972 macro_f1=0.8844 time=2592.5s


Epoch 098: loss=0.3158 acc=0.9248 bal_acc=0.8967 macro_f1=0.8844 time=2505.7s


Epoch 099: loss=0.3163 acc=0.9254 bal_acc=0.8970 macro_f1=0.8848 time=2711.5s
💾 [BEST@99] 临时保存到: ./real_data/organ_classification_mybest.pth


Epoch 100: loss=0.3146 acc=0.9254 bal_acc=0.8971 macro_f1=0.8851 time=2638.5s
💾 [BEST@100] 临时保存到: ./real_data/organ_classification_mybest.pth
💾 已保存最后一次权重到: ./real_data/organ_classification_last.pth
Fitted temperature (BEST @ epoch 100): 1.063
✅ Saved BEST organ classifier to: ./real_data/organ_classification_mybest.pth


In [13]:
print("\n===== FINAL VALIDATION REPORT (Current Model / EMA + TTA + Anneal) =====")
print(final_eval['report'])
print("Confusion Matrix (rows=true, cols=pred):")
for row in final_eval['cm']:
    print(row)


===== FINAL VALIDATION REPORT (Current Model / EMA + TTA + Anneal) =====
              precision    recall  f1-score   support

        bark     0.7955    0.8507    0.8222       663
      flower     0.9050    0.9339    0.9192      3571
       fruit     0.8371    0.8649    0.8508      2495
        leaf     0.9575    0.9389    0.9481     12521

    accuracy                         0.9254     19250
   macro avg     0.8738    0.8971    0.8851     19250
weighted avg     0.9266    0.9254    0.9258     19250

Confusion Matrix (rows=true, cols=pred):
[564, 1, 23, 75]
[0, 3335, 51, 185]
[23, 52, 2158, 262]
[122, 297, 346, 11756]


In [5]:
# step 2.5
# 重启内核后，加载器官分类器

import torch
import torch.nn as nn
import timm
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 定义与训练时相同的模型结构（保持一致）
class OrgansHier(nn.Module):
    def __init__(self, backbone='convnext_small', pretrained=False, drop_path_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.gate    = nn.Linear(feat_dim, 2)
        self.head_fl = nn.Linear(feat_dim, 2)
        self.head_fb = nn.Linear(feat_dim, 2)

    def forward(self, x, temperature=1.0):
        import torch.nn.functional as F
        feats = self.backbone(x)
        gate_logits = self.gate(feats) / temperature
        fl_logits   = self.head_fl(feats) / temperature
        fb_logits   = self.head_fb(feats) / temperature

        g    = F.softmax(gate_logits, dim=1)[:, 0]
        p_fl = F.softmax(fl_logits, dim=1)
        p_fb = F.softmax(fb_logits, dim=1)

        probs = torch.stack([
            g * p_fl[:, 0],          # flower
            g * p_fl[:, 1],          # leaf
            (1 - g) * p_fb[:, 0],    # fruit
            (1 - g) * p_fb[:, 1],    # bark
        ], dim=1).clamp_min(1e-8)
        return probs

# 2. 直接加载保存的 pth
ckpt_path = "./real_data/organ_classification_mybest.pth"
ckpt = torch.load(ckpt_path, map_location=DEVICE)

organ_model = OrgansHier(backbone=ckpt["backbone"],
                   pretrained=False,
                   drop_path_rate=ckpt.get("drop_path_rate", 0.3)).to(DEVICE)
organ_model.load_state_dict(ckpt["state_dict"])
organ_model.eval()

# 3. 类别映射、变换
id2organ = {v: k for k, v in ckpt["organ2id"].items()}
organ_T = ckpt.get("temperature", 1.0)

img_size = int(ckpt.get("cfg", {}).get("img_size", 384))
organ_tfm = transforms.Compose([
    transforms.Resize(int(img_size * 1.05)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

print("✅ 已加载器官分类器")
print("id2organ:", id2organ)
print("温度:", organ_T)


/tmp/ipykernel_16020/2602285453.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=DEVICE)


✅ 已加载器官分类器
id2organ: {0: 'bark', 1: 'flower', 2: 'fruit', 3: 'leaf'}
温度: 1.0633360147476196


In [6]:
# step 2.5
# 测试器官分类器
import os, math
from typing import List, Dict, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from PIL import Image
from torchvision import transforms, models
from torchvision.models import (
    ResNet18_Weights,
    ResNet50_Weights,
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
)

img = organ_tfm(Image.open("test_997_Populus_tremula3.png").convert("RGB")).unsqueeze(0).to(DEVICE)
probs = organ_model(img, temperature=organ_T)
print(probs.softmax(dim=1))  # 器官概率分布

from PIL import Image
import torch.nn.functional as F

def predict_organ(image_path, model, tfm, id2organ, device=DEVICE, temperature=1.0):
    # 读取 & 预处理
    img = Image.open(image_path).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)

    # 前向推理
    with torch.no_grad():
        probs = model(x, temperature=temperature)  # [1,4]
        probs = F.softmax(probs, dim=1).squeeze(0).cpu().numpy()

    # 取最大概率类别
    organ_id = int(probs.argmax())
    organ_name = id2organ[organ_id]
    return organ_name, probs

organ, probs = predict_organ("test_997_Populus_tremula.png", organ_model, organ_tfm, id2organ, DEVICE, organ_T)
print("预测器官:", organ)
print("各类别概率:", {id2organ[i]: round(float(p), 4) for i, p in enumerate(probs)})



tensor([[0.1817, 0.1816, 0.1823, 0.4544]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
预测器官: leaf
各类别概率: {'bark': 0.1816, 'flower': 0.1812, 'fruit': 0.1809, 'leaf': 0.4563}


### 以上是可行的器官分类器

### 重新构建

In [12]:
# ========= 从这里开始替换（训练用；不含推理） =========
from copy import deepcopy
# 记录每个器官的验证变换（评估/训练口径严格对齐）
VAL_TF_REGISTRY = {}   # organ -> val_tf
current_organ = None   # 仅用于评估函数内取 organ

def make_per_class_split(
    df: pd.DataFrame, organ: str,
    gid2local_json_dir: Path,
    COL_ORGAN_TXT: str, COL_SPECIESID: str,
    min_count_train: int = 3,
    val_ratio: float = 0.2,
    seed: int = 42
):
    organ = organ.lower().strip()
    gid2local = json.load(open(gid2local_json_dir / f"species_global2local_{organ}.json", "r", encoding="utf-8"))

    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)

    tr_list, te_list = [], []
    rng = np.random.RandomState(seed)
    for cls, g in sub.groupby("__species_local__"):
        n = len(g)
        if n < min_count_train:
            tr_list.append(g)
            continue
        n_val = max(1, int(round(n * val_ratio)))
        n_val = min(n_val, n - 1)
        idx = rng.permutation(n)
        val_idx = idx[:n_val]
        tr_idx  = idx[n_val:]
        tr_list.append(g.iloc[tr_idx]); te_list.append(g.iloc[val_idx])

    tr_df = pd.concat(tr_list).reset_index(drop=True)
    te_df = pd.concat(te_list).reset_index(drop=True) if te_list else pd.DataFrame(columns=sub.columns)

    out_dir = gid2local_json_dir / "splits"
    out_dir.mkdir(parents=True, exist_ok=True)
    tr_df.to_csv(out_dir / f"{organ}_train_split.csv", index=False)
    te_df.to_csv(out_dir / f"{organ}_val_split.csv", index=False)
    print(f"[{organ}] 保存切分：train={len(tr_df)}, val={len(te_df)}，"
          f"类覆盖(train={tr_df['__species_local__'].nunique()}, val={te_df['__species_local__'].nunique()})")
    return tr_df, te_df


def _load_fixed_split_or_fallback(df, organ, OUT_DIR, COL_ORGAN_TXT, COL_SPECIESID, gid2local):
    sp_dir = OUT_DIR / "splits"
    tr_csv = sp_dir / f"{organ}_train_split.csv"
    te_csv = sp_dir / f"{organ}_val_split.csv"
    if tr_csv.exists() and te_csv.exists():
        tr_df = pd.read_csv(tr_csv)
        te_df = pd.read_csv(te_csv)
        if "__species_local__" not in tr_df.columns:
            tr_df["__species_local__"] = tr_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        if "__species_local__" not in te_df.columns and len(te_df) > 0:
            te_df["__species_local__"] = te_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        tr_df["__species_local__"] = tr_df["__species_local__"].astype(int)
        if len(te_df) > 0:
            te_df["__species_local__"] = te_df["__species_local__"].astype(int)
        return tr_df, te_df

    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)
    counts = sub["__species_local__"].value_counts()
    if (counts.min() >= 2) and (counts.shape[0] >= 2) and (len(sub) >= 4):
        tr_df, te_df = train_test_split(sub, test_size=0.2, stratify=sub["__species_local__"], random_state=42)
    else:
        tr_df, te_df = train_test_split(sub, test_size=0.2, shuffle=True, random_state=42)
    return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)


# 器官自适应配置
ORGAN_SPEC = {
    "flower": {"img_size": 448, "epochs_head": 4, "epochs_ft": 12, "tta": True},
    "leaf":   {"img_size": 384, "epochs_head": 3, "epochs_ft": 10,  "tta": True},
    "fruit":  {"img_size": 448, "epochs_head": 5, "epochs_ft": 16,  "tta": True},
    "bark":   {"img_size": 512, "epochs_head": 5, "epochs_ft": 16,  "tta": True},
}

def make_species_transforms(organ: str, img_size: int = 512):
    organ = organ.lower()
    if organ == 'bark':
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.05)], p=0.5),
            transforms.RandomGrayscale(p=0.10),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.RandomPerspective(distortion_scale=0.05, p=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        val_tf = transforms.Compose([
            transforms.Resize(int(img_size*1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        val_tf = transforms.Compose([
            transforms.Resize(int(img_size*1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    return train_tf, val_tf

# === 关键修复：在 get_backbone_for_organ 内注册 VAL_TF_REGISTRY[organ] ===
def get_backbone_for_organ(organ: str, num_classes: int):
    organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    img_size = spec["img_size"]

    if organ == "flower":
        from torchvision.models import ConvNeXt_Small_Weights
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "leaf":
        weights = ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "fruit":
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "bark":
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_feat, num_classes))
    else:
        raise ValueError(f"未知器官: {organ}")

    train_tf, val_tf = make_species_transforms(organ, img_size)
    # ★ 注册验证变换（Warmup 与评估都会用到）
    VAL_TF_REGISTRY[organ] = val_tf
    return model.to(DEVICE), (train_tf, val_tf), img_size


# —— collate 过滤坏样本（与你原逻辑一致）——
from torch.utils.data._utils.collate import default_collate
def drop_corrupt_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), []
    return default_collate(batch)

# —— Dataset（与你原逻辑一致）——
class BasicImageDataset(Dataset):
    def __init__(self, df, image_col, label_col, label_lookup, tfm):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.label_lookup = label_lookup
        self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            return None
        img = self.tfm(img)
        label = self.label_lookup[int(row[self.label_col])]
        return img, label, path


# ============ 评估（保留，以验证训练是否正常）===========
from sklearn.metrics import accuracy_score, f1_score, classification_report
PRIOR_LOG_REGISTRY: Dict[str, torch.Tensor] = {}

@torch.no_grad()
def evaluate_with_multicrop(model, te_df, img_size, device,
                             use_tta=True, use_multicrop=True,
                             n_crops: int = 8, ratio_low: float = 0.85,
                             logit_adj_tau: float = 1.0):
    model.eval()
    organ = globals().get("current_organ", "unknown")
    val_tf = VAL_TF_REGISTRY.get(organ, None)
    assert val_tf is not None, f"[{organ}] 未找到验证变换，请检查 VAL_TF_REGISTRY 注册。"

    def _ensure_min_size(pil_img, min_side_hw):
        H, W = pil_img.size[1], pil_img.size[0]
        if H >= min_side_hw and W >= min_side_hw:
            return pil_img
        scale = max(min_side_hw / max(1, H), min_side_hw / max(1, W))
        new_w = max(min_side_hw, int(np.ceil(W * scale)))
        new_h = max(min_side_hw, int(np.ceil(H * scale)))
        return pil_img.resize((new_w, new_h), Image.BICUBIC)

    def apply_val_tf(img):
        return val_tf(img).unsqueeze(0).to(device, non_blocking=True)

    def make_crops(img):
        if not use_multicrop or n_crops <= 1:
            return []
        crops = []
        if organ in ("flower", "leaf"):
            img_big = _ensure_min_size(img, img_size)
            tc = transforms.TenCrop(img_size) if n_crops >= 10 else transforms.FiveCrop(img_size)
            out = tc(img_big)
            norm = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            ])
            for c in out:
                crops.append(norm(c if isinstance(c, Image.Image) else c).unsqueeze(0))
            return crops
        resize_side = int(img_size * 1.20)
        img_safe = _ensure_min_size(img, img_size)
        img_r = transforms.Resize(resize_side)(img_safe)
        W, H = img_r.size
        g = max(1, int(round(n_crops ** 0.5)))
        grid_rows, grid_cols = g, g
        sw = (W - img_size) // max(1, grid_cols - 1) if grid_cols > 1 else 0
        sh = (H - img_size) // max(1, grid_rows - 1) if grid_rows > 1 else 0
        norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
        for r in range(grid_rows):
            for c in range(grid_cols):
                left = min(c * sw, max(0, W - img_size))
                top  = min(r * sh, max(0, H - img_size))
                crop = img_r.crop((left, top, left + img_size, top + img_size))
                crops.append(norm(crop).unsqueeze(0))
        return crops

    log_prior = PRIOR_LOG_REGISTRY.get(organ, None)
    y_true, y_pred = [], []
    top1_hits, top3_hits = 0, 0

    for _, row in tqdm(te_df.iterrows(), total=len(te_df),
                       desc=f"[{organ}] Eval (TTA={'Y' if use_tta else 'N'}, MC={'Y' if use_multicrop else 'N'})"):
        path = row[COL_IMAGE]
        y = int(row["__species_local__"])
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            continue

        x = apply_val_tf(img)
        logits = model(x)
        if use_tta:
            logits = 0.5 * (logits + model(torch.flip(x, dims=[3])))

        crops = make_crops(img)
        if crops:
            for xx in crops:
                xx = xx.to(device, non_blocking=True)
                l = model(xx)
                if use_tta:
                    l = 0.5 * (l + model(torch.flip(xx, dims=[3])))
                logits += l
            logits = logits / (1 + len(crops))

        if log_prior is not None and logit_adj_tau is not None and logit_adj_tau > 0:
            logits = logits - logit_adj_tau * log_prior.view(1, -1).to(logits.device)

        probs = F.softmax(logits, dim=1)
        top3 = probs.topk(3, dim=1)
        pred1 = top3.indices[0, 0].item()
        top3_set = set(top3.indices[0].tolist())

        y_true.append(y); y_pred.append(pred1)
        if pred1 == y: top1_hits += 1
        if y in top3_set: top3_hits += 1

    if len(y_true) == 0:
        return 0.0, 0.0, 0.0, "N/A"

    top1 = top1_hits / len(y_true)
    top3 = top3_hits / len(y_true)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    labels_present = sorted(set(y_true) | set(y_pred))
    try:
        local2label = {int(k): v for v, k in json.load(
            open(OUT_DIR / f"species_local_map_{organ}.json", "r", encoding="utf-8")
        ).items()}
        target_names = [local2label.get(i, str(i)) for i in labels_present]
    except Exception:
        target_names = [str(i) for i in labels_present]

    report = classification_report(y_true, y_pred, labels=labels_present,
                                   target_names=target_names, zero_division=0)
    return top1, top3, macro_f1, report


# ============ 训练主函数（仅训练，不含推理）===========
class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, weight=None, reduction='mean', label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', label_smoothing=label_smoothing)
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce).clamp_min(1e-8)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss

def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0: return x, (y, y), 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, (y, y[idx]), lam

def mixup_criterion(crit, pred, targets, lam):
    y_a, y_b = targets
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

def train_one_species_model_for_organ(
    df: pd.DataFrame,
    organ: str,
    epochs_head: int = None,
    epochs_ft: int = None,
    bs: int = 64,
    base_lr_head: float = 1e-3,
    base_lr_ft_head: float = 1e-4,
    base_lr_ft_backbone: float = 5e-6,
    num_workers: int = 2,
    early_stop_patience: int = 5,
    scheduler_patience: int = 2,
    use_balanced_sampler: bool = True,
    use_multicrop_eval: bool = True,
    freeze_bn_after_warmup: bool = True,
    use_mixup: bool = True,
    mixup_alpha: float = 0.2,
    use_ema: bool = True,
    ema_m: float = 0.999,
):
    global current_organ
    current_organ = organ = organ.lower().strip()

    # 轮次
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    epochs_head = spec["epochs_head"] if epochs_head is None else epochs_head
    epochs_ft   = spec["epochs_ft"]   if epochs_ft   is None else epochs_ft

    # 映射
    gid2local = json.load(open(OUT_DIR / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    local2gid = json.load(open(OUT_DIR / f"species_local2global_{organ}.json", "r", encoding="utf-8"))
    num_classes = len(local2gid)

    # 模型 & 变换（★此处会注册 VAL_TF_REGISTRY[organ]）
    model, tf_pair, img_size = get_backbone_for_organ(organ, num_classes)
    train_tf, val_tf = tf_pair

    # 固定切分 or 回退
    tr_df, te_df = _load_fixed_split_or_fallback(
        df=df, organ=organ, OUT_DIR=OUT_DIR,
        COL_ORGAN_TXT=COL_ORGAN_TXT, COL_SPECIESID=COL_SPECIESID,
        gid2local=gid2local
    )
    # 验证集类必须包含于训练集
    assert set(te_df["__species_local__"].unique()).issubset(set(tr_df["__species_local__"].unique())), \
        f"[{organ}] Val 出现了训练缺失的类别，请清理或重建 OUT_DIR/splits。"

    # 类频与 class_w
    cnt = tr_df["__species_local__"].value_counts()
    class_w = torch.tensor(
        [1.0 / np.log(1.2 + cnt.get(i, 1)) for i in range(num_classes)],
        dtype=torch.float, device=DEVICE
    )

    # DataLoader
    local_lookup = {int(i): int(i) for i in range(num_classes)}
    tr_ds = BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", local_lookup, train_tf)
    te_ds = BasicImageDataset(te_df, COL_IMAGE, "__species_local__", local_lookup, val_tf)

    if use_balanced_sampler:
        weights = tr_df["__species_local__"].map(lambda i: class_w[int(i)].item()).astype(float).values
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True
        )
        tr_ld = DataLoader(tr_ds, batch_size=bs, sampler=sampler,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)
    else:
        tr_ld = DataLoader(tr_ds, batch_size=bs, shuffle=True,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)

    te_ld = DataLoader(te_ds, batch_size=bs, shuffle=False,
                       num_workers=max(1, num_workers//2), pin_memory=True, persistent_workers=False,
                       collate_fn=drop_corrupt_collate)

    # 损失
    if organ == "bark":
        crit = FocalLoss(gamma=1.5, label_smoothing=0.03); use_mixup = True
    elif organ == "fruit":
        crit = FocalLoss(gamma=1.6, label_smoothing=0.05); use_mixup = True
    else:
        crit = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05); use_mixup = True

    # ===== A) Warmup（只训头） =====
    for p in model.parameters(): p.requires_grad = False
    if organ == "flower":
        for p in model.classifier[2].parameters(): p.requires_grad = True
    elif organ == "leaf":
        for p in model.fc.parameters(): p.requires_grad = True
    elif organ == "fruit":
        for p in model.classifier[1].parameters(): p.requires_grad = True
    elif organ == "bark":
        for p in model.classifier[2].parameters(): p.requires_grad = True

    model = model.to(DEVICE)
    crit_warmup = nn.CrossEntropyLoss(label_smoothing=0.05)
    # Warmup 走 val_tf（稳定口径）
    warmup_loader = DataLoader(
        BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", local_lookup, VAL_TF_REGISTRY[organ]),
        batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True, collate_fn=drop_corrupt_collate
    )
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=base_lr_head)

    # Sanity check：输出维度
    tmp_loader = DataLoader(
        BasicImageDataset(tr_df.sample(min(8, len(tr_df))), COL_IMAGE, "__species_local__", local_lookup, VAL_TF_REGISTRY[organ]),
        batch_size=min(8, bs), shuffle=True, num_workers=0, collate_fn=drop_corrupt_collate
    )
    x0, y0, _ = next(iter(tmp_loader))
    x0, y0 = x0.to(DEVICE), y0.to(DEVICE)
    with torch.no_grad():
        logits0 = model(x0)
    assert logits0.shape[1] == num_classes, f"[{organ}] 头部维度不等于类数：{logits0.shape[1]} vs {num_classes}"

    for ep in range(epochs_head):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(warmup_loader, total=len(warmup_loader), desc=f"[{organ}] Warmup {ep+1}/{epochs_head}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = crit_warmup(logits, y)
            opt_head.zero_grad(); loss.backward(); opt_head.step()
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
        print(f"[{organ}] Warmup {ep+1}/{epochs_head} | Train Acc: {correct/max(1,total):.4f}")

    # BN 冻结（小 batch 推荐）
    if freeze_bn_after_warmup:
        for m in model.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    # ===== B) 解冻高层微调（LLRD/两组LR + EMA + Mixup） =====
    for p in model.parameters(): p.requires_grad = False
    if organ == "leaf":
        for name, p in model.named_parameters():
            if name.startswith(("layer3","layer4","fc")): p.requires_grad = True
    elif organ in ["flower", "bark"]:
        for name, p in model.named_parameters():
            if any(name.startswith(k) for k in ("features.7","features.8","stages.2","stages.3","classifier")):
                p.requires_grad = True
        for p in getattr(model, "classifier").parameters(): p.requires_grad = True
    elif organ == "fruit":
        for name, p in model.named_parameters():
            if name.startswith(("features.6","features.7","classifier")): p.requires_grad = True
        for p in model.classifier.parameters(): p.requires_grad = True

    params_head, params_backbone = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (params_head if any(k in name for k in ["fc","classifier"]) else params_backbone).append(p)

    def build_llrd_for_convnext(m, base_lr_bb, lr_head):
        groups = []
        for name, p in m.named_parameters():
            if not p.requires_grad: 
                continue
            if "classifier" in name or name.startswith("fc"):
                groups.append({"params": [p], "lr": lr_head, "weight_decay": 5e-4})
            else:
                lr = base_lr_bb
                if any(k in name for k in ["features.8","stages.3"]):
                    lr = max(base_lr_bb * 8.0, 8e-5)
                elif any(k in name for k in ["features.7","stages.2"]):
                    lr = max(base_lr_bb * 4.0, 4e-5)
                groups.append({"params": [p], "lr": lr, "weight_decay": 3e-4})
        return groups

    if organ in ("bark","flower"):
        param_groups = build_llrd_for_convnext(model, base_lr_ft_backbone, base_lr_ft_head)
        opt_ft = torch.optim.AdamW(param_groups)
    else:
        opt_ft = torch.optim.AdamW([
            {"params": params_backbone, "lr": base_lr_ft_backbone, "weight_decay": 3e-4},
            {"params": params_head,     "lr": base_lr_ft_head,     "weight_decay": 5e-4},
        ])

    from torch.optim.lr_scheduler import ReduceLROnPlateau
    scheduler = ReduceLROnPlateau(opt_ft, mode="max", factor=0.5, patience=max(2, scheduler_patience), verbose=True)

    ema_model = deepcopy(model).to(DEVICE) if use_ema else None
    if use_ema:
        for p in ema_model.parameters(): p.requires_grad = False

    best_state = None
    best_top1  = -1.0
    bad = 0

    for ep in range(epochs_ft):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Finetune {ep+1}/{epochs_ft}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            if use_mixup:
                x, (ya, yb), lam = mixup_data(x, y, alpha=mixup_alpha)
                logits = model(x)
                loss = mixup_criterion(crit, logits, (ya, yb), lam)
            else:
                logits = model(x)
                loss = crit(logits, y)
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()
            if use_ema:
                with torch.no_grad():
                    for p_e, p in zip(ema_model.parameters(), model.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)
            pred = logits.argmax(1)
            correct += (pred == (y if not use_mixup else ya)).sum().item()
            total += y.size(0)
        tr_acc = correct / max(1,total)

        model_for_eval = ema_model if use_ema else model
        val_top1, val_top3, val_macro_f1, _ = evaluate_with_multicrop(
            model_for_eval, te_df, img_size, DEVICE,
            use_tta=True, use_multicrop=use_multicrop_eval
        )
        print(f"[{organ}] Finetune {ep+1}/{epochs_ft} | Train Acc: {tr_acc:.4f} | "
              f"Val Top1: {val_top1:.4f} | Top3: {val_top3:.4f} | MacroF1: {val_macro_f1:.4f}")

        scheduler.step(val_top1)
        if val_top1 > best_top1:
            best_top1 = val_top1
            bad = 0
            best_state = deepcopy(model_for_eval.state_dict())
            print(f"[{organ}] 🔥 Update BEST Top1={best_top1:.4f}")
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f"[{organ}] Early stopping at epoch {ep+1}.")
                break

    # 载入 BEST，最终验证一次并保存
    if best_state is not None:
        (ema_model if use_ema else model).load_state_dict(best_state, strict=False)

    best_model = (ema_model if use_ema else model)
    top1, top3, macro_f1, report = evaluate_with_multicrop(
        best_model, te_df, img_size, DEVICE, use_tta=True, use_multicrop=use_multicrop_eval
    )
    print(f"✅ [{organ}] Final Test | Top1: {top1:.4f} | Top3: {top3:.4f} | MacroF1: {macro_f1:.4f}")
    print(report)

    torch.save(best_model.state_dict(), OUT_DIR / f"{organ}_species_model.pth")
    print(f"✅ 已保存: {OUT_DIR / f'{organ}_species_model.pth'}")

# ===== 按器官开训（保持你原有的 batch 与 lr 设置）=====
for organ in ["bark","flower","fruit","leaf"]:
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    if organ == "flower":
        bs = 48; base_lr_ft_backbone = 1e-5;  freeze_bn = True
    elif organ == "leaf":
        bs = 48; base_lr_ft_backbone = 8e-6;  freeze_bn = True
    elif organ == "fruit":
        bs = 40; base_lr_ft_backbone = 7e-6;  freeze_bn = True
    elif organ == "bark":
        bs = 40; base_lr_ft_backbone = 1.5e-5; freeze_bn = True

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        epochs_head=3,
        epochs_ft=5,
        bs=bs,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=base_lr_ft_backbone,
        num_workers=2,
        early_stop_patience=5,
        scheduler_patience=2,
        use_balanced_sampler=True,
        freeze_bn_after_warmup=freeze_bn,
    )

# ========= 替换结束 =========



🎯 训练器官 [bark] —— 样本=6630, 物种=345


[bark] Warmup 1/3: 100%|██████████| 133/133 [02:49<00:00,  1.27s/it]


[bark] Warmup 1/3 | Train Acc: 0.0918


[bark] Warmup 2/3: 100%|██████████| 133/133 [02:53<00:00,  1.31s/it]


[bark] Warmup 2/3 | Train Acc: 0.1992


...

[leaf] Finetune 4/5 | Train Acc: 0.2395 | Val Top1: 0.3747 | Top3: 0.5657 | MacroF1: 0.3088
[leaf] 🔥 Update BEST Top1=0.3747


[leaf] Eval (TTA=Y, MC=Y): 100%|██████████| 25041/25041 [1:00:30<00:00,  6.90it/s]


[leaf] Finetune 5/5 | Train Acc: 0.2449 | Val Top1: 0.3685 | Top3: 0.5595 | MacroF1: 0.3054


[leaf] Eval (TTA=Y, MC=Y): 100%|██████████| 25041/25041 [1:01:03<00:00,  6.84it/s]


✅ [leaf] Final Test | Top1: 0.3747 | Top3: 0.5657 | MacroF1: 0.3088
                                precision    recall  f1-score   support

            Acacia spectabilis       0.20      0.22      0.21         9
            Acrocomia aculeata       0.29      0.04      0.06        57
             Acrocomia emensis       0.00      0.00      0.00        45
            Acrocomia hassleri       0.00      0.00      0.00        34
         Acrocomia intumescens       0.09      0.02      0.04        46
               Acrocomia totai       0.14      0.28      0.18        47
            Adonidia merrillii       0.31      0.70      0.43        53
          Aglaonema commutatum       0.69      0.49      0.57        96
            Aglaonema modestum       0.51      0.40      0.45        75
              Aglaonema pictum       0.36      0.11      0.16        38
               Albizia elegans       0.18      0.31      0.23        26
              Alibertia edulis       0.50      0.16      0.24      

In [ ]:
# ====================== 新增：species_new 准备 ======================
#SPECIES_NEW_DIR = "/mnt/e/code/plants-classification-conda/species_new"
#(SPECIES_NEW_DIR / "splits").mkdir(parents=True, exist_ok=True)

from copy import deepcopy

# 记录每个器官的验证变换（评估/训练口径严格对齐）
VAL_TF_REGISTRY = {}


SPECIES_NEW_DIR = OUT_DIR / "species_new"
(SPECIES_NEW_DIR / "splits").mkdir(parents=True, exist_ok=True)

def build_new_species_maps_for_organ(df: pd.DataFrame, organ: str,
                                     col_organ: str, col_label: str, col_gid: str):
    """
    为给定器官重建'new_'前缀的局部物种映射：
    - 过滤掉在该器官下仅有 1 张图片的物种
    - 重新编号 local_id
    - 保存到 species_new/ 下（文件名前缀 new_）
    返回：gid2local, local2gid, label2local 以及被保留的全局 species_id 集合
    """
    organ = organ.lower().strip()
    sub = df[df[col_organ] == organ].copy()

    # 每物种（全局 species_id）在该器官下的样本数
    cnt = sub.groupby(col_gid)[col_gid].transform("count")
    sub = sub[cnt >= 2].copy()  # 只保留 >=2 的物种（确保可切分 train & val）

    # 若清理后为空或仅 1 类，则直接返回空映射
    uniq = sub[[col_label, col_gid]].drop_duplicates().sort_values([col_gid, col_label]).reset_index(drop=True)
    if len(uniq) < 2:
        return {}, {}, {}, set()

    uniq["local_id"] = range(len(uniq))
    label2local = {row[col_label]: int(row["local_id"])         for _, row in uniq.iterrows()}
    gid2local   = {str(int(row[col_gid])): int(row["local_id"]) for _, row in uniq.iterrows()}
    local2gid   = {str(int(row["local_id"])): int(row[col_gid]) for _, row in uniq.iterrows()}

    # 保存到 species_new/，带 new_ 前缀
    with open(SPECIES_NEW_DIR / f"new_species_local_map_{organ}.json", "w", encoding="utf-8") as f:
        json.dump(label2local, f, ensure_ascii=False, indent=2)
    with open(SPECIES_NEW_DIR / f"new_species_global2local_{organ}.json", "w", encoding="utf-8") as f:
        json.dump(gid2local, f, ensure_ascii=False, indent=2)
    with open(SPECIES_NEW_DIR / f"new_species_local2global_{organ}.json", "w", encoding="utf-8") as f:
        json.dump(local2gid, f, ensure_ascii=False, indent=2)

    print(f"✅ [{organ}] 已生成 new_* 映射：labels={len(label2local)}（已过滤仅1张的物种）")
    kept_gids = set(int(x) for x in sub[col_gid].unique())
    return gid2local, local2gid, label2local, kept_gids


def _load_fixed_split_or_fallback_NEW(
    df: pd.DataFrame,
    organ: str,
    col_organ: str,
    col_gid: str,
    new_gid2local: dict,
    val_ratio: float = 0.2,
    seed: int = 42,
):
    """
    保证 train/val 均覆盖所有“保留下来的”物种（local 类别）：
    - 每个类至少 1 张进入 val，且至少 1 张进入 train
    - 若某类样本 <2，之前就已被过滤，不会进入这里
    - 切分结果写入 species_new/splits 下
    """
    organ = organ.lower().strip()
    sp_dir = SPECIES_NEW_DIR / "splits"
    tr_csv = sp_dir / f"{organ}_train_split.csv"
    te_csv = sp_dir / f"{organ}_val_split.csv"

    if tr_csv.exists() and te_csv.exists():
        tr_df = pd.read_csv(tr_csv)
        te_df = pd.read_csv(te_csv)
        # 回填 __species_local__
        if "__species_local__" not in tr_df.columns:
            tr_df["__species_local__"] = tr_df[col_gid].astype(int).astype(str).map(new_gid2local).astype(int)
        if len(te_df) > 0 and "__species_local__" not in te_df.columns:
            te_df["__species_local__"] = te_df[col_gid].astype(int).astype(str).map(new_gid2local).astype(int)
        tr_df["__species_local__"] = tr_df["__species_local__"].astype(int)
        if len(te_df) > 0:
            te_df["__species_local__"] = te_df["__species_local__"].astype(int)
        return tr_df, te_df

    sub = df[df[col_organ] == organ].copy()
    sub["__species_local__"] = sub[col_gid].astype(int).astype(str).map(new_gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)

    # 分类别手动切分，确保每类 train/val 都有
    rng = np.random.RandomState(seed)
    tr_list, te_list = [], []
    for _, g in sub.groupby("__species_local__"):
        n = len(g)
        # 这里 n>=2 已由上游过滤保证
        n_val = max(1, int(round(n * val_ratio)))
        n_val = min(n_val, n - 1)  # 至少留1张给train
        idx = rng.permutation(n)
        val_idx = idx[:n_val]
        tr_idx  = idx[n_val:]
        tr_list.append(g.iloc[tr_idx])
        te_list.append(g.iloc[val_idx])

    tr_df = pd.concat(tr_list).reset_index(drop=True)
    te_df = pd.concat(te_list).reset_index(drop=True)

    sp_dir.mkdir(parents=True, exist_ok=True)
    tr_df.to_csv(tr_csv, index=False)
    te_df.to_csv(te_csv, index=False)
    print(f"[{organ}] 保存切分（NEW）：train={len(tr_df)}, val={len(te_df)}，"
          f"类覆盖(train={tr_df['__species_local__'].nunique()}, val={te_df['__species_local__'].nunique()})")
    return tr_df, te_df
# ====================== 新增结束 ======================

current_organ = None   # 仅用于评估函数内取 organ

# 器官自适应配置
ORGAN_SPEC = {
    "flower": {"img_size": 448, "epochs_head": 4, "epochs_ft": 12, "tta": True},
    "leaf":   {"img_size": 384, "epochs_head": 3, "epochs_ft": 10,  "tta": True},
    "fruit":  {"img_size": 448, "epochs_head": 5, "epochs_ft": 16,  "tta": True},
    "bark":   {"img_size": 512, "epochs_head": 5, "epochs_ft": 16,  "tta": True},
}

def make_species_transforms(organ: str, img_size: int = 512):
    organ = organ.lower()
    if organ == 'bark':
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.05)], p=0.5),
            transforms.RandomGrayscale(p=0.10),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.RandomPerspective(distortion_scale=0.05, p=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        val_tf = transforms.Compose([
            transforms.Resize(int(img_size*1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        val_tf = transforms.Compose([
            transforms.Resize(int(img_size*1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    return train_tf, val_tf


# === 关键修复：在 get_backbone_for_organ 内注册 VAL_TF_REGISTRY[organ] ===
def get_backbone_for_organ(organ: str, num_classes: int):
    organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    img_size = spec["img_size"]

    if organ == "flower":
        from torchvision.models import ConvNeXt_Small_Weights
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "leaf":
        weights = ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "fruit":
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "bark":
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_feat, num_classes))
    else:
        raise ValueError(f"未知器官: {organ}")

    train_tf, val_tf = make_species_transforms(organ, img_size)
    # ★ 注册验证变换（Warmup 与评估都会用到）
    VAL_TF_REGISTRY[organ] = val_tf
    return model.to(DEVICE), (train_tf, val_tf), img_size


from torch.utils.data._utils.collate import default_collate
def drop_corrupt_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), []
    return default_collate(batch)

class BasicImageDatasetNEW(Dataset):
    def __init__(self, df, image_col, label_col, label_lookup, tfm):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.label_lookup = label_lookup
        self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            return None
        img = self.tfm(img)
        label = self.label_lookup[int(row[self.label_col])]
        return img, label, path


# ============ 评估（保留，以验证训练是否正常）===========
from sklearn.metrics import accuracy_score, f1_score, classification_report
PRIOR_LOG_REGISTRY: Dict[str, torch.Tensor] = {}

@torch.no_grad()
def evaluate_with_multicrop(model, te_df, img_size, device,
                             use_tta=True, use_multicrop=True,
                             n_crops: int = 8, ratio_low: float = 0.85,
                             logit_adj_tau: float = 1.0):
    model.eval()
    organ = globals().get("current_organ", "unknown")
    val_tf = VAL_TF_REGISTRY.get(organ, None)
    assert val_tf is not None, f"[{organ}] 未找到验证变换，请检查 VAL_TF_REGISTRY 注册。"

    def _ensure_min_size(pil_img, min_side_hw):
        H, W = pil_img.size[1], pil_img.size[0]
        if H >= min_side_hw and W >= min_side_hw:
            return pil_img
        scale = max(min_side_hw / max(1, H), min_side_hw / max(1, W))
        new_w = max(min_side_hw, int(np.ceil(W * scale)))
        new_h = max(min_side_hw, int(np.ceil(H * scale)))
        return pil_img.resize((new_w, new_h), Image.BICUBIC)

    def apply_val_tf(img):
        return val_tf(img).unsqueeze(0).to(device, non_blocking=True)

    def make_crops(img):
        if not use_multicrop or n_crops <= 1:
            return []
        crops = []
        if organ in ("flower", "leaf"):
            img_big = _ensure_min_size(img, img_size)
            tc = transforms.TenCrop(img_size) if n_crops >= 10 else transforms.FiveCrop(img_size)
            out = tc(img_big)
            norm = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            ])
            for c in out:
                crops.append(norm(c if isinstance(c, Image.Image) else c).unsqueeze(0))
            return crops
        resize_side = int(img_size * 1.20)
        img_safe = _ensure_min_size(img, img_size)
        img_r = transforms.Resize(resize_side)(img_safe)
        W, H = img_r.size
        g = max(1, int(round(n_crops ** 0.5)))
        grid_rows, grid_cols = g, g
        sw = (W - img_size) // max(1, grid_cols - 1) if grid_cols > 1 else 0
        sh = (H - img_size) // max(1, grid_rows - 1) if grid_rows > 1 else 0
        norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
        for r in range(grid_rows):
            for c in range(grid_cols):
                left = min(c * sw, max(0, W - img_size))
                top  = min(r * sh, max(0, H - img_size))
                crop = img_r.crop((left, top, left + img_size, top + img_size))
                crops.append(norm(crop).unsqueeze(0))
        return crops

    log_prior = PRIOR_LOG_REGISTRY.get(organ, None)
    y_true, y_pred = [], []
    top1_hits, top3_hits = 0, 0

    for _, row in tqdm(te_df.iterrows(), total=len(te_df),
                       desc=f"[{organ}] Eval (TTA={'Y' if use_tta else 'N'}, MC={'Y' if use_multicrop else 'N'})"):
        path = row[COL_IMAGE]
        y = int(row["__species_local__"])
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            continue

        x = apply_val_tf(img)
        logits = model(x)
        if use_tta:
            logits = 0.5 * (logits + model(torch.flip(x, dims=[3])))

        crops = make_crops(img)
        if crops:
            for xx in crops:
                xx = xx.to(device, non_blocking=True)
                l = model(xx)
                if use_tta:
                    l = 0.5 * (l + model(torch.flip(xx, dims=[3])))
                logits += l
            logits = logits / (1 + len(crops))

        if log_prior is not None and logit_adj_tau is not None and logit_adj_tau > 0:
            logits = logits - logit_adj_tau * log_prior.view(1, -1).to(logits.device)

        probs = F.softmax(logits, dim=1)
        top3 = probs.topk(3, dim=1)
        pred1 = top3.indices[0, 0].item()
        top3_set = set(top3.indices[0].tolist())

        y_true.append(y); y_pred.append(pred1)
        if pred1 == y: top1_hits += 1
        if y in top3_set: top3_hits += 1

    if len(y_true) == 0:
        return 0.0, 0.0, 0.0, "N/A"

    top1 = top1_hits / len(y_true)
    top3 = top3_hits / len(y_true)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    labels_present = sorted(set(y_true) | set(y_pred))
    try:
        local2label = {int(k): v for v, k in json.load(
            open(OUT_DIR / f"species_local_map_{organ}.json", "r", encoding="utf-8")
        ).items()}
        target_names = [local2label.get(i, str(i)) for i in labels_present]
    except Exception:
        target_names = [str(i) for i in labels_present]

    report = classification_report(y_true, y_pred, labels=labels_present,
                                   target_names=target_names, zero_division=0)
    return top1, top3, macro_f1, report

# ============ 训练主函数（仅训练，不含推理）===========
class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, weight=None, reduction='mean', label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', label_smoothing=label_smoothing)
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce).clamp_min(1e-8)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss

def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0: return x, (y, y), 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, (y, y[idx]), lam

def mixup_criterion(crit, pred, targets, lam):
    y_a, y_b = targets
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

# === 下方：在你的训练函数里换成“new_映射 + NEW切分 + new_权重名” ===
def train_one_species_model_for_organ(
    df: pd.DataFrame,
    organ: str,
    epochs_head: int = None,
    epochs_ft: int = None,
    bs: int = 64,
    base_lr_head: float = 1e-3,
    base_lr_ft_head: float = 1e-4,
    base_lr_ft_backbone: float = 5e-6,
    num_workers: int = 2,
    early_stop_patience: int = 5,
    scheduler_patience: int = 2,
    use_balanced_sampler: bool = True,
    use_multicrop_eval: bool = True,
    freeze_bn_after_warmup: bool = True,
    use_mixup: bool = True,
    mixup_alpha: float = 0.2,
    use_ema: bool = True,
    ema_m: float = 0.999,
):
    # —— 保持你原有的器官自适应配置、骨干与评估逻辑不变 —— 
    global current_organ
    current_organ = organ = organ.lower().strip()

    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    epochs_head = spec["epochs_head"] if epochs_head is None else epochs_head
    epochs_ft   = spec["epochs_ft"]   if epochs_ft   is None else epochs_ft

    # ★★★ 关键改动1：为该器官构建 new_* 局部映射（会过滤掉该器官下仅1张图的物种）
    new_gid2local, new_local2gid, new_label2local, kept_gids = build_new_species_maps_for_organ(
        df, organ, COL_ORGAN_TXT, COL_LABEL, COL_SPECIESID
    )
    if len(new_local2gid) < 2:
        print(f"⚠️ 跳过 {organ}（过滤后可用物种数 < 2）")
        return

    num_classes = len(new_local2gid)

    # 模型 & 变换（沿用你已有的 get_backbone_for_organ，注意它会注册 VAL_TF）
    model, tf_pair, img_size = get_backbone_for_organ(organ, num_classes)
    train_tf, val_tf = tf_pair

    # ★★★ 关键改动2：使用 NEW 的切分逻辑，保证 train/val 全覆盖
    tr_df, te_df = _load_fixed_split_or_fallback_NEW(
        df=df, organ=organ, col_organ=COL_ORGAN_TXT, col_gid=COL_SPECIESID,
        new_gid2local=new_gid2local, val_ratio=0.2, seed=42
    )

    # 验证集类别必须包含于训练集
    assert set(te_df["__species_local__"].unique()).issubset(set(tr_df["__species_local__"].unique())), \
        f"[{organ}] (NEW) Val 出现了训练缺失的类别，请清理 species_new/splits。"

    # —— 下面训练细节完全沿用你的原逻辑（损失、DataLoader、warmup、微调、EMA、评估等）——
    # 类频与 class_w
    cnt = tr_df["__species_local__"].value_counts()
    class_w = torch.tensor(
        [1.0 / np.log(1.2 + cnt.get(i, 1)) for i in range(num_classes)],
        dtype=torch.float, device=DEVICE
    )

    local_lookup = {int(i): int(i) for i in range(num_classes)}
    tr_ds = BasicImageDatasetNEW(tr_df, COL_IMAGE, "__species_local__", local_lookup, train_tf)
    te_ds = BasicImageDatasetNEW(te_df, COL_IMAGE, "__species_local__", local_lookup, val_tf)

    if use_balanced_sampler:
        weights = tr_df["__species_local__"].map(lambda i: class_w[int(i)].item()).astype(float).values
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True
        )
        tr_ld = torch.utils.data.DataLoader(
            tr_ds, batch_size=bs, sampler=sampler,
            num_workers=num_workers, pin_memory=True, persistent_workers=False,
            collate_fn=drop_corrupt_collate)
    else:
        tr_ld = torch.utils.data.DataLoader(
            tr_ds, batch_size=bs, shuffle=True,
            num_workers=num_workers, pin_memory=True, persistent_workers=False,
            collate_fn=drop_corrupt_collate)

    te_ld = torch.utils.data.DataLoader(
        te_ds, batch_size=bs, shuffle=False,
        num_workers=max(1, num_workers//2), pin_memory=True, persistent_workers=False,
        collate_fn=drop_corrupt_collate)

    # 损失 & 训练两阶段（与你原脚本一致，这里略）……
    # 你原有的 Warmup / freeze BN / Finetune / EMA / 评估代码块直接保留即可
    # —— 省略 ——（把你现有训练主体粘过来，变量名不变即可）

    # …………（此处放你原来的训练循环与 evaluate_with_multicrop 调用）………… #

    # 损失
    if organ == "bark":
        crit = FocalLoss(gamma=1.5, label_smoothing=0.03); use_mixup = True
    elif organ == "fruit":
        crit = FocalLoss(gamma=1.6, label_smoothing=0.05); use_mixup = True
    else:
        crit = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05); use_mixup = True

    # ===== A) Warmup（只训头） =====
    for p in model.parameters(): p.requires_grad = False
    if organ == "flower":
        for p in model.classifier[2].parameters(): p.requires_grad = True
    elif organ == "leaf":
        for p in model.fc.parameters(): p.requires_grad = True
    elif organ == "fruit":
        for p in model.classifier[1].parameters(): p.requires_grad = True
    elif organ == "bark":
        for p in model.classifier[2].parameters(): p.requires_grad = True

    model = model.to(DEVICE)
    crit_warmup = nn.CrossEntropyLoss(label_smoothing=0.05)
    # Warmup 走 val_tf（稳定口径）
    warmup_loader = DataLoader(
        BasicImageDatasetNEW(tr_df, COL_IMAGE, "__species_local__", local_lookup, VAL_TF_REGISTRY[organ]),
        batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True, collate_fn=drop_corrupt_collate
    )
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=base_lr_head)

    # Sanity check：输出维度
    tmp_loader = DataLoader(
        BasicImageDatasetNEW(tr_df.sample(min(8, len(tr_df))), COL_IMAGE, "__species_local__", local_lookup, VAL_TF_REGISTRY[organ]),
        batch_size=min(8, bs), shuffle=True, num_workers=0, collate_fn=drop_corrupt_collate
    )
    x0, y0, _ = next(iter(tmp_loader))
    x0, y0 = x0.to(DEVICE), y0.to(DEVICE)
    with torch.no_grad():
        logits0 = model(x0)
    assert logits0.shape[1] == num_classes, f"[{organ}] 头部维度不等于类数：{logits0.shape[1]} vs {num_classes}"

    for ep in range(epochs_head):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(warmup_loader, total=len(warmup_loader), desc=f"[{organ}] Warmup {ep+1}/{epochs_head}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = crit_warmup(logits, y)
            opt_head.zero_grad(); loss.backward(); opt_head.step()
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
        print(f"[{organ}] Warmup {ep+1}/{epochs_head} | Train Acc: {correct/max(1,total):.4f}")

    # BN 冻结（小 batch 推荐）
    if freeze_bn_after_warmup:
        for m in model.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    # ===== B) 解冻高层微调（LLRD/两组LR + EMA + Mixup） =====
    for p in model.parameters(): p.requires_grad = False
    if organ == "leaf":
        for name, p in model.named_parameters():
            if name.startswith(("layer3","layer4","fc")): p.requires_grad = True
    elif organ in ["flower", "bark"]:
        for name, p in model.named_parameters():
            if any(name.startswith(k) for k in ("features.7","features.8","stages.2","stages.3","classifier")):
                p.requires_grad = True
        for p in getattr(model, "classifier").parameters(): p.requires_grad = True
    elif organ == "fruit":
        for name, p in model.named_parameters():
            if name.startswith(("features.6","features.7","classifier")): p.requires_grad = True
        for p in model.classifier.parameters(): p.requires_grad = True

    params_head, params_backbone = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (params_head if any(k in name for k in ["fc","classifier"]) else params_backbone).append(p)

    def build_llrd_for_convnext(m, base_lr_bb, lr_head):
        groups = []
        for name, p in m.named_parameters():
            if not p.requires_grad: 
                continue
            if "classifier" in name or name.startswith("fc"):
                groups.append({"params": [p], "lr": lr_head, "weight_decay": 5e-4})
            else:
                lr = base_lr_bb
                if any(k in name for k in ["features.8","stages.3"]):
                    lr = max(base_lr_bb * 8.0, 8e-5)
                elif any(k in name for k in ["features.7","stages.2"]):
                    lr = max(base_lr_bb * 4.0, 4e-5)
                groups.append({"params": [p], "lr": lr, "weight_decay": 3e-4})
        return groups

    if organ in ("bark","flower"):
        param_groups = build_llrd_for_convnext(model, base_lr_ft_backbone, base_lr_ft_head)
        opt_ft = torch.optim.AdamW(param_groups)
    else:
        opt_ft = torch.optim.AdamW([
            {"params": params_backbone, "lr": base_lr_ft_backbone, "weight_decay": 3e-4},
            {"params": params_head,     "lr": base_lr_ft_head,     "weight_decay": 5e-4},
        ])

    from torch.optim.lr_scheduler import ReduceLROnPlateau
    scheduler = ReduceLROnPlateau(opt_ft, mode="max", factor=0.5, patience=max(2, scheduler_patience), verbose=True)

    ema_model = deepcopy(model).to(DEVICE) if use_ema else None
    if use_ema:
        for p in ema_model.parameters(): p.requires_grad = False

    best_state = None
    best_top1  = -1.0
    bad = 0

    for ep in range(epochs_ft):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Finetune {ep+1}/{epochs_ft}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            if use_mixup:
                x, (ya, yb), lam = mixup_data(x, y, alpha=mixup_alpha)
                logits = model(x)
                loss = mixup_criterion(crit, logits, (ya, yb), lam)
            else:
                logits = model(x)
                loss = crit(logits, y)
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()
            if use_ema:
                with torch.no_grad():
                    for p_e, p in zip(ema_model.parameters(), model.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)
            pred = logits.argmax(1)
            correct += (pred == (y if not use_mixup else ya)).sum().item()
            total += y.size(0)
        tr_acc = correct / max(1,total)

        model_for_eval = ema_model if use_ema else model
        val_top1, val_top3, val_macro_f1, _ = evaluate_with_multicrop(
            model_for_eval, te_df, img_size, DEVICE,
            use_tta=True, use_multicrop=use_multicrop_eval
        )
        print(f"[{organ}] Finetune {ep+1}/{epochs_ft} | Train Acc: {tr_acc:.4f} | "
              f"Val Top1: {val_top1:.4f} | Top3: {val_top3:.4f} | MacroF1: {val_macro_f1:.4f}")

        scheduler.step(val_top1)
        if val_top1 > best_top1:
            best_top1 = val_top1
            bad = 0
            best_state = deepcopy(model_for_eval.state_dict())
            print(f"[{organ}] 🔥 Update BEST Top1={best_top1:.4f}")
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f"[{organ}] Early stopping at epoch {ep+1}.")
                break

    # 载入 BEST，最终验证一次并保存
    if best_state is not None:
        (ema_model if use_ema else model).load_state_dict(best_state, strict=False)

    best_model = (ema_model if use_ema else model)
    top1, top3, macro_f1, report = evaluate_with_multicrop(
        best_model, te_df, img_size, DEVICE, use_tta=True, use_multicrop=use_multicrop_eval
    )
    print(f"✅ [{organ}] Final Test | Top1: {top1:.4f} | Top3: {top3:.4f} | MacroF1: {macro_f1:.4f}")
    print(report)


    # 训练完保存 BEST 后：
    # ★★★ 关键改动3：保存到 species_new/，并用 new_ 前缀区分
    save_path = SPECIES_NEW_DIR / f"new_{organ}_species_model.pth"
    torch.save(best_model.state_dict(), save_path)
    print(f"✅ (NEW) 已保存: {save_path}")

# ===== 按器官开训（保持你原有的 batch 与 lr 设置）=====
for organ in ["bark","flower","fruit","leaf"]:
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    if organ == "flower":
        bs = 48; base_lr_ft_backbone = 1e-5;  freeze_bn = True
    elif organ == "leaf":
        bs = 48; base_lr_ft_backbone = 8e-6;  freeze_bn = True
    elif organ == "fruit":
        bs = 40; base_lr_ft_backbone = 7e-6;  freeze_bn = True
    elif organ == "bark":
        bs = 40; base_lr_ft_backbone = 1.5e-5; freeze_bn = True

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        epochs_head=3,
        epochs_ft=5,
        bs=bs,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=base_lr_ft_backbone,
        num_workers=2,
        early_stop_patience=5,
        scheduler_patience=2,
        use_balanced_sampler=True,
        freeze_bn_after_warmup=freeze_bn,
    )



🎯 训练器官 [bark] —— 样本=6630, 物种=345
✅ [bark] 已生成 new_* 映射：labels=281（已过滤仅1张的物种）


[bark] Warmup 1/3: 100%|██████████| 131/131 [02:46<00:00,  1.27s/it]


[bark] Warmup 1/3 | Train Acc: 0.0981


[bark] Warmup 2/3: 100%|██████████| 131/131 [02:47<00:00,  1.28s/it]


[bark] Warmup 2/3 | Train Acc: 0.1940


...

[flower] Warmup 2/3: 100%|██████████| 595/595 [15:13<00:00,  1.54s/it]


[flower] Warmup 2/3 | Train Acc: 0.5157


[flower] Warmup 3/3: 100%|██████████| 595/595 [15:10<00:00,  1.53s/it]
/home/jmy/miniconda3/envs/plant310/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[flower] Warmup 3/3 | Train Acc: 0.5674


[flower] Eval (TTA=Y, MC=Y):  67%|██████▋   | 4810/7147 [16:12<08:41,  4.49it/s]

### 以下是都舍弃版本

In [24]:
# ===== 修正版：导入补齐 =====
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import AutoAugment, AutoAugmentPolicy, RandomErasing
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm
from PIL import Image

# 复用你在上文已定义的全局变量/对象：
# OUT_DIR, DEVICE, df, COL_IMAGE, COL_ORGAN_TXT, COL_SPECIESID, ORGAN_SPEC, VAL_TF_REGISTRY, current_organ
# 若不存在请确保在此之前按你的代码2/3定义（这些在你的代码2/3里均已存在）【引用：代码2/3】
# --------------------------------------------------------------------------------------------

# ====== 仅负责生成并保存固定切分（不做 tr_csv/te_csv 存在性检查）======
def make_per_class_split(
    df: pd.DataFrame, organ: str,
    gid2local_json_dir: Path,
    COL_ORGAN_TXT: str, COL_SPECIESID: str,
    min_count_train: int = 3,   # <3 的类仅进训练，不进验证
    val_ratio: float = 0.2,
    seed: int = 42
):
    organ = organ.lower().strip()
    gid2local = json.load(open(gid2local_json_dir / f"species_global2local_{organ}.json", "r", encoding="utf-8"))

    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)

    tr_list, te_list = [], []
    rng = np.random.RandomState(seed)
    for cls, group in sub.groupby("__species_local__"):
        n = len(group)
        if n < min_count_train:
            tr_list.append(group)
            continue
        n_val = max(1, int(round(n * val_ratio)))
        n_val = min(n_val, n - 1)
        # 固定随机切分（不使用 stratify，逐类拆分更稳）
        idx = rng.permutation(n)
        val_idx = idx[:n_val]
        tr_idx  = idx[n_val:]
        tr_list.append(group.iloc[tr_idx]); te_list.append(group.iloc[val_idx])

    tr_df = pd.concat(tr_list).reset_index(drop=True)
    te_df = pd.concat(te_list).reset_index(drop=True) if te_list else pd.DataFrame(columns=sub.columns)

    out_dir = gid2local_json_dir / "splits"
    out_dir.mkdir(parents=True, exist_ok=True)
    tr_df.to_csv(out_dir / f"{organ}_train_split.csv", index=False)
    te_df.to_csv(out_dir / f"{organ}_val_split.csv", index=False)
    print(f"[{organ}] 保存切分：train={len(tr_df)}, val={len(te_df)}，"
          f"类覆盖(train={tr_df['__species_local__'].nunique()}, val={te_df['__species_local__'].nunique()})")
    return tr_df, te_df

# ====== 读取固定切分，若不存在则回退到原逻辑 ======
def _load_fixed_split_or_fallback(df, organ, OUT_DIR, COL_ORGAN_TXT, COL_SPECIESID, gid2local):
    sp_dir = OUT_DIR / "splits"
    tr_csv = sp_dir / f"{organ}_train_split.csv"
    te_csv = sp_dir / f"{organ}_val_split.csv"
    if tr_csv.exists() and te_csv.exists():
        tr_df = pd.read_csv(tr_csv)
        te_df = pd.read_csv(te_csv)
        if "__species_local__" not in tr_df.columns:
            tr_df["__species_local__"] = tr_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        if "__species_local__" not in te_df.columns and len(te_df) > 0:
            te_df["__species_local__"] = te_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        tr_df["__species_local__"] = tr_df["__species_local__"].astype(int)
        if len(te_df) > 0:
            te_df["__species_local__"] = te_df["__species_local__"].astype(int)
        return tr_df, te_df

    # 回退：器官内整体切分（尽量 stratify；不满足条件则 shuffle）
    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)
    counts = sub["__species_local__"].value_counts()
    if (counts.min() >= 2) and (counts.shape[0] >= 2) and (len(sub) >= 4):
        tr_df, te_df = train_test_split(sub, test_size=0.2, stratify=sub["__species_local__"], random_state=42)
    else:
        tr_df, te_df = train_test_split(sub, test_size=0.2, shuffle=True, random_state=42)
    return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)

# ====== 针对不同器官的 transform（训练/验证严格对齐注册） ======
def make_species_transforms(organ: str, img_size: int):
    organ = organ.lower().strip()

    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])

    if organ == "flower":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.15)),
            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0), ratio=(0.80, 1.20)),
            transforms.RandomHorizontalFlip(p=0.5),
            AutoAugment(AutoAugmentPolicy.IMAGENET),
            transforms.ColorJitter(0.20, 0.20, 0.12, 0.04),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            RandomErasing(p=0.20, scale=(0.02, 0.10), value='random'),
        ])
    elif organ == "leaf":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0), ratio=(0.85, 1.15)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.18, 0.18, 0.10, 0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            RandomErasing(p=0.25, scale=(0.02, 0.12), value='random'),
        ])
    elif organ == "fruit":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.65, 1.0), ratio=(0.90, 1.10)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(0.15, 0.15, 0.10, 0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
    else:  # bark
        center_path = transforms.Compose([
            transforms.Resize(int(img_size * 1.05)),
            transforms.CenterCrop(img_size),
        ])
        rand_path = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.80, 1.0), ratio=(0.98, 1.02)),
        ])
        class _RandOrCenter(transforms.RandomChoice):
            def __init__(self, p_center=0.4):
                super().__init__([center_path, rand_path]); self.p_center = p_center
            def __call__(self, img):
                return (center_path if random.random() < self.p_center else rand_path)(img)
        train_tf = transforms.Compose([
            _RandOrCenter(p_center=0.4),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.06, 0.06, 0.04, 0.02),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            RandomErasing(p=0.25, scale=(0.02, 0.08), value='random'),
        ])
    return train_tf, val_tf

# ====== Backbone 按器官选择（带 Dropout 头）并注册 val_tf 到 VAL_TF_REGISTRY ======
def get_backbone_for_organ(organ: str, num_classes: int):
    organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    img_size = spec["img_size"]

    if organ == "flower":
        weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "leaf":
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "fruit":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "bark":
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_feat, num_classes))
    else:
        raise ValueError(f"未知器官: {organ}")

    train_tf, val_tf = make_species_transforms(organ, img_size)
    VAL_TF_REGISTRY[organ] = val_tf
    return model.to(DEVICE), (train_tf, val_tf), img_size

# ====== collate：过滤坏图 ======
from torch.utils.data._utils.collate import default_collate
class BasicImageDataset(Dataset):
    def __init__(self, df, image_col, label_col, label_lookup, tfm):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.label_lookup = label_lookup
        self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            return None
        img = self.tfm(img)
        label = self.label_lookup[int(row[self.label_col])]
        return img, label, path
def drop_corrupt_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), []
    return default_collate(batch)

# ====== FocalLoss / Mixup（与你现有逻辑一致） ======
class FocalLoss(nn.Module):
    def __init__(self, gamma=1.6, weight=None, reduction='mean', label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', label_smoothing=label_smoothing)
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce).clamp_min(1e-8)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss

def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0: return x, (y, y), 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, (y, y[idx]), lam

def mixup_criterion(crit, pred, targets, lam):
    y_a, y_b = targets
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

# ====== 评估：与训练时 val_tf 严格对齐 + TTA + 多裁片 ======
@torch.no_grad()
def evaluate_with_multicrop(model, te_df, img_size, device, use_tta=True, use_multicrop=True):
    model.eval()
    organ = globals().get("current_organ", "unknown")
    val_tf = VAL_TF_REGISTRY.get(organ, transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ]))
    def apply_val_tf(img):
        return val_tf(img).unsqueeze(0).to(device, non_blocking=True)
    def make_crops(img):
        crops = []
        if organ in ("flower", "leaf"):
            base = transforms.Compose([transforms.Resize(int(img_size * 1.05)), transforms.FiveCrop(img_size)])
            out = base(img)
            norm = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
            for c in out: crops.append(norm(c).unsqueeze(0))
        else:
            resize_side = int(img_size * 1.2)
            img_r = transforms.Resize(resize_side)(img)
            W, H = img_r.size
            grid_rows = grid_cols = 3
            sw = (W - img_size) // max(1, grid_cols - 1) if grid_cols > 1 else 0
            sh = (H - img_size) // max(1, grid_rows - 1) if grid_rows > 1 else 0
            norm = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
            for r in range(grid_rows):
                for c in range(grid_cols):
                    left = min(c * sw, max(0, W - img_size))
                    top  = min(r * sh, max(0, H - img_size))
                    crop = img_r.crop((left, top, left + img_size, top + img_size))
                    crops.append(norm(crop).unsqueeze(0))
        return crops

    y_true, y_pred = [], []
    top1_hits, top3_hits = 0, 0
    for _, row in tqdm(te_df.iterrows(), total=len(te_df),
                       desc=f"[{organ}] Eval (TTA={'Y' if use_tta else 'N'}, MC={'Y' if use_multicrop else 'N'})"):
        path = row[COL_IMAGE]
        y = int(row["__species_local__"])
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            continue
        x = apply_val_tf(img)
        logits = model(x)
        if use_tta:
            logits = 0.5 * (logits + model(torch.flip(x, dims=[3])))
        logits_accum = logits
        if use_multicrop:
            crops = make_crops(img)
            for xx in crops:
                xx = xx.to(device, non_blocking=True)
                l = model(xx)
                if use_tta:
                    l = 0.5 * (l + model(torch.flip(xx, dims=[3])))
                logits_accum += l
            logits_accum = logits_accum / (1 + len(crops))
        probs = F.softmax(logits_accum, dim=1)
        top3 = probs.topk(3, dim=1)
        pred1 = top3.indices[0, 0].item()
        top3_set = set(top3.indices[0].tolist())
        y_true.append(y); y_pred.append(pred1)
        if pred1 == y: top1_hits += 1
        if y in top3_set: top3_hits += 1

    if len(y_true) == 0:
        return 0.0, 0.0, 0.0, "N/A"
    top1 = top1_hits / len(y_true)
    top3 = top3_hits / len(y_true)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    # 仅为可读性生成报告（只覆盖出现过的标签）
    labels_present = sorted(set(y_true) | set(y_pred))
    report = classification_report(y_true, y_pred, labels=labels_present, zero_division=0)
    return top1, top3, macro_f1, report

# ====== 主训练函数（均衡采样 + 两阶段微调 + EMA + 评估口径统一）======
def train_one_species_model_for_organ(
    df: pd.DataFrame,
    organ: str,
    epochs_head: int = None,
    epochs_ft: int = None,
    bs: int = 64,
    base_lr_head: float = 1e-3,
    base_lr_ft_head: float = 1e-4,
    base_lr_ft_backbone: float = 5e-6,
    num_workers: int = 2,
    early_stop_patience: int = 5,
    scheduler_patience: int = 2,
    use_balanced_sampler: bool = True,
    freeze_bn_after_warmup: bool = True,
    use_mixup: bool = True,
    mixup_alpha: float = 0.2,
    use_ema: bool = True,
    ema_m: float = 0.999,
):
    global current_organ
    current_organ = organ = organ.lower().strip()

    # 轮次
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    epochs_head = epochs_head if epochs_head is not None else spec["epochs_head"]
    epochs_ft   = epochs_ft   if epochs_ft   is not None else spec["epochs_ft"]

    # 映射
    gid2local = json.load(open(OUT_DIR / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    local2gid = json.load(open(OUT_DIR / f"species_local2global_{organ}.json", "r", encoding="utf-8"))
    num_classes = len(local2gid)

    # 模型 & tf
    model, tf_pair, img_size = get_backbone_for_organ(organ, num_classes)
    train_tf, val_tf = tf_pair

    # 固定切分优先
    tr_df, te_df = _load_fixed_split_or_fallback(
        df=df, organ=organ, OUT_DIR=OUT_DIR,
        COL_ORGAN_TXT=COL_ORGAN_TXT, COL_SPECIESID=COL_SPECIESID, gid2local=gid2local
    )

    # Dataset/Loader
    local_lookup = {int(i): int(i) for i in range(num_classes)}
    tr_ds = BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", local_lookup, train_tf)
    te_ds = BasicImageDataset(te_df, COL_IMAGE, "__species_local__", local_lookup, val_tf)

    if use_balanced_sampler:
        cnt = tr_df["__species_local__"].value_counts()
        class_w = 1.0 / cnt
        weights = tr_df["__species_local__"].map(class_w).astype(float).values
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True
        )
        tr_ld = DataLoader(tr_ds, batch_size=bs, sampler=sampler,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)
    else:
        tr_ld = DataLoader(tr_ds, batch_size=bs, shuffle=True,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)

    # 损失设定
    if organ == "fruit":
        crit = FocalLoss(gamma=1.6, label_smoothing=0.05)
        use_mixup = True
    else:
        crit = nn.CrossEntropyLoss(label_smoothing=0.10)
        if organ == "bark":
            use_mixup = False

    # A) 仅训练分类头
    def freeze_backbone_only_train_head(m: nn.Module, organ_name: str):
        for p in m.parameters(): p.requires_grad = False
        if organ_name == "flower":
            for p in m.classifier[2].parameters(): p.requires_grad = True
        elif organ_name == "leaf":
            for p in m.fc.parameters(): p.requires_grad = True
        elif organ_name == "fruit":
            for p in m.classifier[1].parameters(): p.requires_grad = True
        elif organ_name == "bark":
            for p in m.classifier[2].parameters(): p.requires_grad = True

    freeze_backbone_only_train_head(model, organ)
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=base_lr_head)

    model = model.to(DEVICE)
    for epoch in range(epochs_head):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Warmup {epoch+1}/{epochs_head}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = crit(logits, y)
            opt_head.zero_grad(); loss.backward(); opt_head.step()
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
        print(f"[{organ}] Warmup {epoch+1}/{epochs_head} | Train Acc: {correct/max(total,1):.4f}")

    if freeze_bn_after_warmup:
        for m in model.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    # B) 解冻高层微调 + EMA
    def unfreeze_high_layers(m: nn.Module, organ_name: str):
        for p in m.parameters(): p.requires_grad = False
        if organ_name == "leaf":
            for name, p in m.named_parameters():
                if name.startswith("layer3") or name.startswith("layer4") or name.startswith("fc"):
                    p.requires_grad = True
        elif organ_name == "flower":
            for name, p in m.named_parameters():
                if name.startswith("features.7") or name.startswith("stages.2") or name.startswith("classifier"):
                    p.requires_grad = True
            for p in m.classifier.parameters(): p.requires_grad = True
        elif organ_name == "fruit":
            for name, p in m.named_parameters():
                if name.startswith("features.6") or name.startswith("features.7") or name.startswith("classifier"):
                    p.requires_grad = True
            for p in m.classifier.parameters(): p.requires_grad = True
        elif organ_name == "bark":
            for name, p in m.named_parameters():
                if name.startswith("features.6") or name.startswith("features.7") or name.startswith("classifier"):
                    p.requires_grad = True
            for p in m.classifier.parameters(): p.requires_grad = True

    unfreeze_high_layers(model, organ)

    params_head, params_backbone = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            if any(k in name for k in ["fc", "classifier"]):
                params_head.append(p)
            else:
                params_backbone.append(p)

    wd_backbone = 3e-4 if organ in ("bark",) else 5e-2
    wd_head     = 5e-4 if organ in ("bark",) else 5e-2

    opt_ft = torch.optim.Adam([
        {"params": params_backbone, "lr": base_lr_ft_backbone, "weight_decay": wd_backbone},
        {"params": params_head,     "lr": base_lr_ft_head,     "weight_decay": wd_head},
    ])
    from torch.optim.lr_scheduler import ReduceLROnPlateau
    scheduler = ReduceLROnPlateau(opt_ft, mode="max", factor=0.5, patience=max(2, scheduler_patience), verbose=True)

    # EMA
    ema_model = deepcopy(model).to(DEVICE) if use_ema else None
    if use_ema:
        for p in ema_model.parameters(): p.requires_grad = False

    best_top1 = -1.0
    best_state = None

    for epoch in range(epochs_ft):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Finetune {epoch+1}/{epochs_ft}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            if use_mixup:
                x, (ya, yb), lam = mixup_data(x, y, alpha=mixup_alpha)
                logits = model(x)
                loss = mixup_criterion(crit, logits, (ya, yb), lam)
            else:
                logits = model(x)
                loss = crit(logits, y)
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()

            if use_ema:
                with torch.no_grad():
                    for p_e, p in zip(ema_model.parameters(), model.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)

            pred = logits.argmax(1)
            correct += (pred == (y if not use_mixup else ya)).sum().item()
            total += y.size(0)

        tr_acc = correct / max(total, 1)

        # 评估：EMA优先
        model_for_eval = ema_model if use_ema else model
        val_top1, val_top3, val_macro_f1, _ = evaluate_with_multicrop(
            model_for_eval, te_df, img_size, DEVICE, use_tta=True, use_multicrop=True
        )
        print(f"[{organ}] Finetune {epoch+1}/{epochs_ft} | Train Acc: {tr_acc:.4f} | "
              f"Val Top1: {val_top1:.4f} | Top3: {val_top3:.4f} | MacroF1: {val_macro_f1:.4f}")

        scheduler.step(val_top1)
        if val_top1 > best_top1:
            best_top1 = val_top1
            best_state = deepcopy(model_for_eval.state_dict())

    # 最优权重恢复
    if best_state is not None:
        (ema_model if use_ema else model).load_state_dict(best_state, strict=False)
    best_model = (ema_model if use_ema else model)

    top1, top3, macro_f1, report = evaluate_with_multicrop(
        best_model, te_df, img_size, DEVICE, use_tta=True, use_multicrop=True
    )
    print(f"✅ [{organ}] Final Test | Top1: {top1:.4f} | Top3: {top3:.4f} | MacroF1: {macro_f1:.4f}")
    print(report)

    torch.save(best_model.state_dict(), OUT_DIR / f"{organ}_species_model.pth")
    print(f"✅ 已保存: {OUT_DIR / f'{organ}_species_model.pth'}")

# ====== 先为每个器官生成固定切分（只需一次）======
make_per_class_split(df=df, organ="bark", gid2local_json_dir=OUT_DIR,
                     COL_ORGAN_TXT=COL_ORGAN_TXT, COL_SPECIESID=COL_SPECIESID,
                     min_count_train=2, val_ratio=0.2, seed=42)

for organ in ["flower","fruit","leaf"]:
    make_per_class_split(df=df, organ=organ, gid2local_json_dir=OUT_DIR,
                         COL_ORGAN_TXT=COL_ORGAN_TXT, COL_SPECIESID=COL_SPECIESID,
                         min_count_train=3, val_ratio=0.2, seed=42)

# ====== 训练每个器官的种类分类器（器官分类器已在代码2里训练好）======
for organ in ["bark","flower","fruit","leaf"]:
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    if organ == "flower":
        bs = 48; base_lr_ft_backbone = 1e-5;  freeze_bn = True
    elif organ == "leaf":
        bs = 48; base_lr_ft_backbone = 8e-6; freeze_bn = True
    elif organ == "fruit":
        bs = 40; base_lr_ft_backbone = 7e-6; freeze_bn = True
    else:  # bark
        bs = 40; base_lr_ft_backbone = 1.5e-5; freeze_bn = True

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        bs=bs,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=base_lr_ft_backbone,
        num_workers=2,
        early_stop_patience=5,
        scheduler_patience=2,
        use_balanced_sampler=True,
        freeze_bn_after_warmup=freeze_bn,
    )


[bark] 保存切分：train=5292, val=1338，类覆盖(train=345, val=281)
[flower] 保存切分：train=28577, val=7134，类覆盖(train=422, val=383)
[fruit] 保存切分：train=19975, val=4978，类覆盖(train=430, val=398)
[leaf] 保存切分：train=100159, val=25041，类覆盖(train=444, val=444)

🎯 训练器官 [bark] —— 样本=6630, 物种=345


[bark] Warmup 1/5:  41%|████▏     | 55/133 [01:21<01:55,  1.48s/it]


KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
"""
训练脚本：器官内物种分类（ArcFace/CosFace 版本）
- 固定切分（可重复）
- 均衡采样 + Class-Balanced Loss 权重
- Warmup(仅头) → 解冻高层微调（分组LR）
- EMA & ReduceLROnPlateau
- 评估双口径：Single-Crop(上线口径) / Multi-Crop+TTA(离线最优)
- 最优权重保存：{organ}_species_model.pth，包含 backbone 与 head
"""

VAL_TF_REGISTRY = {}
current_organ = "leaf"  # 仅用于评估标题显示

# ========= 读取数据 =========
df = pd.read_csv(DATA_CSV)

# ========= 工具 =========
def make_species_transforms(organ: str, img_size: int):
    organ = organ.lower().strip()
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])
    if organ == "flower":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.15)),
            transforms.RandomResizedCrop(img_size, scale=(0.70,1.0), ratio=(0.80,1.20)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.2,0.2,0.12,0.04),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            transforms.RandomErasing(p=0.2, scale=(0.02,0.10), value='random'),
        ])
    elif organ == "leaf":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.70,1.0), ratio=(0.85,1.15)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.18,0.18,0.10,0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            transforms.RandomErasing(p=0.25, scale=(0.02,0.12), value='random'),
        ])
    elif organ == "fruit":
        train_tf = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.65,1.0), ratio=(0.90,1.10)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(0.15,0.15,0.10,0.03),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
    else:  # bark
        center_path = transforms.Compose([
            transforms.Resize(int(img_size * 1.05)),
            transforms.CenterCrop(img_size),
        ])
        rand_path = transforms.Compose([
            transforms.Resize(int(img_size * 1.10)),
            transforms.RandomResizedCrop(img_size, scale=(0.80,1.0), ratio=(0.98,1.02)),
        ])
        class _RandOrCenter:
            def __init__(self, p_center=0.4):
                self.p_center=p_center
            def __call__(self, img):
                return (center_path if random.random()<self.p_center else rand_path)(img)
        train_tf = transforms.Compose([
            _RandOrCenter(p_center=0.4),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(0.06,0.06,0.04,0.02),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            transforms.RandomErasing(p=0.25, scale=(0.02,0.08), value='random'),
        ])
    return train_tf, val_tf

class BasicImageDataset(Dataset):
    def __init__(self, df, image_col, label_col, label_lookup, tfm):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.label_lookup = label_lookup
        self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            return None
        img = self.tfm(img)
        label = self.label_lookup[int(row[self.label_col])]
        return img, label, path

def drop_corrupt_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), []
    return default_collate(batch)

# ====== Margin 头 ======
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False):
        super().__init__()
        self.s = float(s); self.m = float(m)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.easy_margin = easy_margin
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, x, label=None):
        x = F.normalize(x); W = F.normalize(self.weight)
        cosine = F.linear(x, W)
        if label is None: return self.s * cosine
        sine = torch.sqrt((1.0 - torch.clamp(cosine**2, 0, 1)).clamp_min(1e-9))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin: phi = torch.where(cosine>0, phi, cosine)
        else:                phi = torch.where(cosine>self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine); one_hot.scatter_(1, label.view(-1,1), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return self.s * logits

class CosMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.35):
        super().__init__()
        self.s = float(s); self.m = float(m)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, x, label=None):
        x = F.normalize(x); W = F.normalize(self.weight)
        cosine = F.linear(x, W)
        if label is None: return self.s * cosine
        one_hot = torch.zeros_like(cosine); one_hot.scatter_(1, label.view(-1,1), 1.0)
        return self.s * (cosine - one_hot * self.m)

# ====== Backbone 工厂：输出特征 + Margin 头 ======
def get_backbone_for_organ(organ: str, num_classes: int, margin_head="arc", s=30.0, m=0.5):
    organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    img_size = spec["img_size"]
    if organ == "flower":
        weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1
        backbone = models.convnext_small(weights=weights)
        in_feat  = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
    elif organ == "leaf":
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        backbone = models.resnet50(weights=weights)
        in_feat  = backbone.fc.in_features
        backbone.fc = nn.Identity()
    elif organ == "fruit":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        backbone = models.efficientnet_b0(weights=weights)
        in_feat  = backbone.classifier[1].in_features
        backbone.classifier[1] = nn.Identity()
    else: # bark
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        backbone = models.convnext_tiny(weights=weights)
        in_feat  = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()

    head = ArcMarginProduct(in_feat, num_classes, s=s, m=m) if margin_head=="arc" \
           else CosMarginProduct(in_feat, num_classes, s=s, m=0.35)
    train_tf, val_tf = make_species_transforms(organ, img_size)
    VAL_TF_REGISTRY[organ] = val_tf
    return backbone.to(DEVICE), head.to(DEVICE), (train_tf, val_tf), img_size

# ====== Mixup & CB Loss 权重 ======
class FocalLoss(nn.Module):
    def __init__(self, gamma=1.6, weight=None, reduction='mean', label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', label_smoothing=label_smoothing)
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce).clamp_min(1e-8)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction=='mean' else (loss.sum() if self.reduction=='sum' else loss)

def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0: return x, (y, y), 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, (y, y[idx]), lam

def mixup_criterion(crit, pred, targets, lam):
    y_a, y_b = targets
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

def make_class_balanced_weight(tr_df, label_col="__species_local__", beta=0.9999, device="cuda"):
    cnt = tr_df[label_col].value_counts().sort_index()
    eff_num = (1.0 - np.power(beta, cnt.values)) / (1.0 - beta)
    cls_w = 1.0 / eff_num
    cls_w = cls_w / cls_w.mean()
    return torch.tensor(cls_w, dtype=torch.float32, device=device)

# ====== 切分：固定保存/读取 ======
def make_per_class_split(df, organ, gid2local_json_dir:Path, min_count_train=3, val_ratio=0.2, seed=42):
    organ = organ.lower().strip()
    gid2local = json.load(open(gid2local_json_dir / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    sub = df[df[COL_ORGAN_TXT]==organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)
    tr_list, te_list = [], []
    rng = np.random.RandomState(seed)
    for _, g in sub.groupby("__species_local__"):
        n = len(g)
        if n < min_count_train:
            tr_list.append(g); continue
        n_val = max(1, int(round(n*val_ratio))); n_val = min(n_val, n-1)
        idx = rng.permutation(n); val_idx = idx[:n_val]; tr_idx = idx[n_val:]
        tr_list.append(g.iloc[tr_idx]); te_list.append(g.iloc[val_idx])
    tr_df = pd.concat(tr_list).reset_index(drop=True)
    te_df = pd.concat(te_list).reset_index(drop=True) if te_list else pd.DataFrame(columns=sub.columns)
    spd = OUT_DIR / "splits"; spd.mkdir(parents=True, exist_ok=True)
    tr_df.to_csv(spd / f"{organ}_train_split.csv", index=False)
    te_df.to_csv(spd / f"{organ}_val_split.csv", index=False)
    print(f"[{organ}] 保存切分：train={len(tr_df)}, val={len(te_df)}，类覆盖(train={tr_df['__species_local__'].nunique()}, val={te_df['__species_local__'].nunique()})")
    return tr_df, te_df

def load_fixed_split_or_fallback(df, organ, gid2local_json_dir:Path):
    organ = organ.lower().strip()
    gid2local = json.load(open(gid2local_json_dir / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    spd = OUT_DIR / "splits"
    tr_csv, te_csv = spd / f"{organ}_train_split.csv", spd / f"{organ}_val_split.csv"
    if tr_csv.exists() and te_csv.exists():
        tr_df, te_df = pd.read_csv(tr_csv), pd.read_csv(te_csv)
        if "__species_local__" not in tr_df.columns:
            tr_df["__species_local__"] = tr_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        if len(te_df)>0 and "__species_local__" not in te_df.columns:
            te_df["__species_local__"] = te_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        return tr_df, te_df
    # fallback
    sub = df[df[COL_ORGAN_TXT]==organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)
    counts = sub["__species_local__"].value_counts()
    if (counts.min() >= 2) and (counts.shape[0] >= 2) and (len(sub) >= 4):
        tr_df, te_df = train_test_split(sub, test_size=0.2, stratify=sub["__species_local__"], random_state=42)
    else:
        tr_df, te_df = train_test_split(sub, test_size=0.2, shuffle=True, random_state=42)
    return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)

# ====== 评估（Single-Crop & Multi-Crop 双口径）======
@torch.no_grad()
def eval_single(backbone, head, te_df, img_size, device):
    backbone.eval(); head.eval()
    organ = globals().get("current_organ", "unknown")
    val_tf = VAL_TF_REGISTRY.get(organ, transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ]))
    y_true, y_pred = [], []
    for _, row in tqdm(te_df.iterrows(), total=len(te_df), desc=f"[{organ}] Eval(Single)"):
        p = row[COL_IMAGE]; y = int(row["__species_local__"])
        try:
            with Image.open(p) as im: img = im.convert("RGB")
        except: continue
        x = val_tf(img).unsqueeze(0).to(device)
        feat = backbone(x); logits = head(feat, None)
        pred = logits.argmax(1).item()
        y_true.append(y); y_pred.append(pred)
    if len(y_true)==0: return 0.0, "N/A"
    top1 = accuracy_score(y_true, y_pred)
    labels_present = sorted(set(y_true)|set(y_pred))
    report = classification_report(y_true, y_pred, labels=labels_present, zero_division=0)
    return top1, report

@torch.no_grad()
def eval_multicrop_tta(backbone, head, te_df, img_size, device, organ):
    backbone.eval(); head.eval()
    val_tf = VAL_TF_REGISTRY[organ]
    def make_crops(img):
        crops=[]
        if organ in ("flower","leaf"):
            base = transforms.Compose([transforms.Resize(int(img_size*1.05)), transforms.FiveCrop(img_size)])
            out = base(img)
            norm = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
            for c in out: crops.append(norm(c).unsqueeze(0))
        else:
            resize_side = int(img_size * 1.2); img_r = transforms.Resize(resize_side)(img)
            W, H = img_r.size; grid_rows = grid_cols = 3
            sw = (W - img_size) // max(1, grid_cols - 1) if grid_cols>1 else 0
            sh = (H - img_size) // max(1, grid_rows - 1) if grid_rows>1 else 0
            norm = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
            for r in range(grid_rows):
                for c in range(grid_cols):
                    left = min(c*sw, max(0, W-img_size))
                    top  = min(r*sh, max(0, H-img_size))
                    crop = img_r.crop((left, top, left+img_size, top+img_size))
                    crops.append(norm(crop).unsqueeze(0))
        return crops

    y_true, y_pred = [], []
    for _, row in tqdm(te_df.iterrows(), total=len(te_df), desc=f"[{organ}] Eval(MC+TTA)"):
        p = row[COL_IMAGE]; y = int(row["__species_local__"])
        try:
            with Image.open(p) as im: img = im.convert("RGB")
        except: continue
        # center
        x = val_tf(img).unsqueeze(0).to(device)
        feat = backbone(x); logits = head(feat, None)
        # hflip
        xh = torch.flip(x, dims=[3]); feat_h = backbone(xh); logits_h = head(feat_h, None)
        logits_accum = 0.5*(logits + logits_h)
        # multi-crop
        crops = make_crops(img)
        for xx in crops:
            xx = xx.to(device)
            f = backbone(xx); l = head(f, None)
            l = 0.5*(l + head(backbone(torch.flip(xx, dims=[3])), None))
            logits_accum += l
        logits_accum /= (1 + len(crops))
        pred = logits_accum.argmax(1).item()
        y_true.append(y); y_pred.append(pred)
    if len(y_true)==0: return 0.0, "N/A"
    top1 = accuracy_score(y_true, y_pred)
    labels_present = sorted(set(y_true)|set(y_pred))
    report = classification_report(y_true, y_pred, labels=labels_present, zero_division=0)
    return top1, report

# ====== 训练主函数 ======
def train_one_species_model_for_organ(
    df: pd.DataFrame, organ: str,
    bs: int, base_lr_head: float, base_lr_ft_head: float, base_lr_ft_backbone: float,
    epochs_head: int=None, epochs_ft: int=None,
    num_workers: int=2, freeze_bn_after_warmup: bool=True,
    use_balanced_sampler: bool=True, use_mixup: bool=True, mixup_alpha: float=0.2,
    use_ema: bool=True, ema_m: float=0.999, margin_head="arc", s=30.0, m=0.5
):
    global current_organ
    current_organ = organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    if epochs_head is None: epochs_head = spec["epochs_head"]
    if epochs_ft   is None: epochs_ft   = spec["epochs_ft"]

    gid2local = json.load(open(OUT_DIR / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    local2gid = json.load(open(OUT_DIR / f"species_local2global_{organ}.json", "r", encoding="utf-8"))
    num_classes = len(local2gid)

    backbone, head, tf_pair, img_size = get_backbone_for_organ(organ, num_classes, margin_head=margin_head, s=s, m=m)
    train_tf, val_tf = tf_pair

    tr_df, te_df = load_fixed_split_or_fallback(df, organ, OUT_DIR)
    local_lookup = {int(i): int(i) for i in range(num_classes)}
    tr_ds = BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", local_lookup, train_tf)
    te_ds = BasicImageDataset(te_df, COL_IMAGE, "__species_local__", local_lookup, val_tf)

    if use_balanced_sampler:
        cnt = tr_df["__species_local__"].value_counts()
        class_w = 1.0 / cnt
        weights = tr_df["__species_local__"].map(class_w).astype(float).values
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True
        )
        tr_ld = DataLoader(tr_ds, batch_size=bs, sampler=sampler, num_workers=num_workers,
                           pin_memory=True, persistent_workers=False, collate_fn=drop_corrupt_collate)
    else:
        tr_ld = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=num_workers,
                           pin_memory=True, persistent_workers=False, collate_fn=drop_corrupt_collate)

    # 损失：带 CB 权重
    cls_weight = make_class_balanced_weight(tr_df, "__species_local__", beta=0.9999, device=DEVICE)
    if organ == "fruit":
        crit = FocalLoss(gamma=1.6, weight=cls_weight, label_smoothing=0.05)
        use_mixup = True
    else:
        crit = nn.CrossEntropyLoss(weight=cls_weight, label_smoothing=0.10)
        if organ == "bark": use_mixup = False

    # 冻骨干仅训头
    for p in backbone.parameters(): p.requires_grad = False
    for p in head.parameters():     p.requires_grad = True
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, list(backbone.parameters())+list(head.parameters())),
                                lr=base_lr_head)

    for epoch in range(epochs_head):
        backbone.train(); head.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Warmup {epoch+1}/{epochs_head}"):
            if x.numel()==0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.no_grad():
                feat = backbone(x)
            logits = head(feat, y)
            loss = crit(logits, y)
            opt_head.zero_grad(); loss.backward(); opt_head.step()
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
        print(f"[{organ}] Warmup {epoch+1}/{epochs_head} | Train Acc: {correct/max(total,1):.4f}")

    if freeze_bn_after_warmup:
        for m in backbone.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    # 解冻高层
    for p in backbone.parameters(): p.requires_grad = False
    for p in head.parameters():     p.requires_grad = True
    if organ == "leaf":
        for name, p in backbone.named_parameters():
            if name.startswith("layer3") or name.startswith("layer4"):
                p.requires_grad = True
    elif organ == "flower":
        for name, p in backbone.named_parameters():
            if name.startswith("features.7") or name.startswith("stages.2"):
                p.requires_grad = True
    elif organ == "fruit":
        for name, p in backbone.named_parameters():
            if name.startswith("features.6") or name.startswith("features.7"):
                p.requires_grad = True
    else: # bark
        for name, p in backbone.named_parameters():
            if name.startswith("features.6") or name.startswith("features.7"):
                p.requires_grad = True

    params_backbone, params_head = [], []
    for name, p in backbone.named_parameters():
        if p.requires_grad: params_backbone.append(p)
    for p in head.parameters():
        if p.requires_grad: params_head.append(p)

    opt_ft = torch.optim.Adam([
        {"params": params_backbone, "lr": base_lr_ft_backbone, "weight_decay": 3e-4},
        {"params": params_head,     "lr": base_lr_ft_head,     "weight_decay": 5e-4},
    ])
    from torch.optim.lr_scheduler import ReduceLROnPlateau
    scheduler = ReduceLROnPlateau(opt_ft, mode="max", factor=0.5, patience=2, verbose=True, min_lr=3e-6)

    ema_backbone = deepcopy(backbone).to(DEVICE) if use_ema else None
    ema_head     = deepcopy(head).to(DEVICE)     if use_ema else None
    if use_ema:
        for p in ema_backbone.parameters(): p.requires_grad = False
        for p in ema_head.parameters():     p.requires_grad = False

    best_top1, best_state = -1.0, None

    for epoch in range(spec["epochs_ft"]):
        backbone.train(); head.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Finetune {epoch+1}/{spec['epochs_ft']}"):
            if x.numel()==0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            if use_mixup:
                x, (ya, yb), lam = mixup_data(x, y, alpha=mixup_alpha)
                feat = backbone(x); logits = head(feat, ya)
                loss = mixup_criterion(crit, logits, (ya, yb), lam)
                target_for_acc = ya
            else:
                feat = backbone(x); logits = head(feat, y)
                loss = crit(logits, y); target_for_acc = y
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()

            if use_ema:
                with torch.no_grad():
                    for p_e, p in zip(ema_backbone.parameters(), backbone.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)
                    for p_e, p in zip(ema_head.parameters(), head.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)

            pred = logits.argmax(1)
            correct += (pred == target_for_acc).sum().item(); total += y.size(0)
        tr_acc = correct / max(total,1)

        bb_eval = ema_backbone if use_ema else backbone
        hd_eval = ema_head     if use_ema else head

        top1_sc, _ = eval_single(bb_eval, hd_eval, te_df, img_size, DEVICE)
        top1_mc, _ = eval_multicrop_tta(bb_eval, hd_eval, te_df, img_size, DEVICE, organ)
        print(f"[{organ}] Epoch {epoch+1} | Train Acc: {tr_acc:.4f} | Val Single: {top1_sc:.4f} | Val MC+TTA: {top1_mc:.4f}")

        scheduler.step(top1_sc)  # 以“上线口径”单裁片指标做调度
        cur_select = top1_sc     # 以单裁片作为保存判据
        if cur_select > best_top1:
            best_top1 = cur_select
            best_state = {"backbone": deepcopy(bb_eval.state_dict()), "head": deepcopy(hd_eval.state_dict())}

    # final
    if best_state is not None:
        (ema_backbone if use_ema else backbone).load_state_dict(best_state["backbone"], strict=False)
        (ema_head     if use_ema else head).load_state_dict(best_state["head"], strict=False)
    bb_best = ema_backbone if use_ema else backbone
    hd_best = ema_head     if use_ema else head

    s1, rep1 = eval_single(bb_best, hd_best, te_df, img_size, DEVICE)
    s2, rep2 = eval_multicrop_tta(bb_best, hd_best, te_df, img_size, DEVICE, organ)
    print(f"✅ [{organ}] Final | Val Single: {s1:.4f}\n{rep1}")
    print(f"✅ [{organ}] Final | Val MC+TTA: {s2:.4f}\n{rep2}")

    torch.save({"backbone": bb_best.state_dict(), "head": hd_best.state_dict()},
               OUT_DIR / f"{organ}_species_model.pth")
    print(f"✅ 已保存: {OUT_DIR / f'{organ}_species_model.pth'}")

# ====== 入口：先生成固定切分，再训练全部器官 ======
if __name__ == "__main__":
    # 一次性生成固定切分（可按需提高 min_count_train）
    make_per_class_split(df, "bark",  OUT_DIR, min_count_train=3, val_ratio=0.2, seed=42)
    make_per_class_split(df, "fruit", OUT_DIR, min_count_train=3, val_ratio=0.2, seed=42)
    make_per_class_split(df, "flower",OUT_DIR, min_count_train=5, val_ratio=0.2, seed=42)
    make_per_class_split(df, "leaf",  OUT_DIR, min_count_train=5, val_ratio=0.2, seed=42)

    def pick(organ):
        if organ=="flower": return dict(bs=48, base_lr_ft_backbone=1e-5,  freeze_bn_after_warmup=True)
        if organ=="leaf":   return dict(bs=48, base_lr_ft_backbone=8e-6,  freeze_bn_after_warmup=True)
        if organ=="fruit":  return dict(bs=40, base_lr_ft_backbone=7e-6,  freeze_bn_after_warmup=True)
        if organ=="bark":   return dict(bs=36, base_lr_ft_backbone=1.5e-5, freeze_bn_after_warmup=True)
        return dict(bs=48, base_lr_ft_backbone=1e-5, freeze_bn_after_warmup=True)

    for organ in ["bark","fruit","flower","leaf"]:
        sub = df[df[COL_ORGAN_TXT]==organ]; n = len(sub); k = sub[COL_SPECIESID].nunique()
        if n<50 or k<2:
            print(f"⚠️ 跳过 {organ}（样本={n}, 物种={k}）"); continue
        hp = pick(organ)
        print(f"\n🎯 训练 [{organ}] —— 样本={n}, 物种={k}")
        train_one_species_model_for_organ(
            df=df, organ=organ,
            bs=hp["bs"],
            base_lr_head=1e-3, base_lr_ft_head=1e-4, base_lr_ft_backbone=hp["base_lr_ft_backbone"],
            freeze_bn_after_warmup=hp["freeze_bn_after_warmup"],
            use_balanced_sampler=True, use_mixup=True, mixup_alpha=0.2,
            use_ema=True, ema_m=0.999, margin_head="arc", s=30.0, m=0.5
        )


In [1]:
# ========================= 依赖 =========================
import json, math, random
from copy import deepcopy
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

from torchvision import transforms, models
from torchvision.models import (
    ResNet50_Weights,
    ConvNeXt_Tiny_Weights, ConvNeXt_Small_Weights,
    EfficientNet_B0_Weights,
)
from torch.utils.data import Dataset, DataLoader
from torch.utils.data._utils.collate import default_collate

# 你工程里已有的全局：OUT_DIR, df, organ_classes, COL_ORGAN_TXT, COL_SPECIESID, COL_IMAGE
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========================= 全局 =========================
VAL_TF_REGISTRY: Dict[str, transforms.Compose] = {}
current_organ: str = None
PRIOR_LOG_REGISTRY = {}  

ORGAN_SPEC = {
    "flower": {"img_size": 448, "epochs_head": 4, "epochs_ft": 12, "tta": True},
    "leaf":   {"img_size": 384, "epochs_head": 3, "epochs_ft": 10, "tta": True},
    "fruit":  {"img_size": 448, "epochs_head": 5, "epochs_ft": 16, "tta": True},
    "bark":   {"img_size": 512, "epochs_head": 5, "epochs_ft": 16, "tta": True},
}

# ========================= Transforms（保持你口径） =========================
def make_species_transforms(organ: str, img_size: int):
    organ = organ.lower().strip()
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])
    if organ == "bark":
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0), ratio=(0.90, 1.10)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.20,0.20,0.20,0.05)], p=0.5),
            transforms.RandomGrayscale(p=0.10),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1,1.5)),
            transforms.RandomPerspective(distortion_scale=0.05, p=0.05),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
    VAL_TF_REGISTRY[organ] = val_tf
    return train_tf, val_tf

# ========================= 骨干（不改动，只换最后线性层） =========================
def get_backbone_for_organ(organ: str, num_classes: int):
    organ = organ.lower().strip()
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    img_size = spec["img_size"]

    if organ == "flower":
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "leaf":
        weights = ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "fruit":
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))
    elif organ == "bark":
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_feat, num_classes))
    else:
        raise ValueError(f"未知器官: {organ}")

    train_tf, val_tf = make_species_transforms(organ, img_size)
    VAL_TF_REGISTRY[organ] = val_tf
    return model.to(DEVICE), (train_tf, val_tf), img_size

# ========================= Dataset =========================
class BasicImageDataset(Dataset):
    def __init__(self, df, image_col, label_col, tfm):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.tfm = tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            return None
        img = self.tfm(img)
        label = int(row[self.label_col])
        return img, label, path

def drop_corrupt_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.empty(0), torch.empty(0, dtype=torch.long), []
    return default_collate(batch)

# ========================= Class-Aware Batch Sampler =========================
class ClassAwareBatchSampler(torch.utils.data.Sampler[List[int]]):
    """
    每个 batch：均匀抽 K 个类，每类随机取样 ceil(B/K) 张，拼成 batch。
    适合长尾，显著缓解 train/val 口径偏移。
    """
    def __init__(self, labels: List[int], batch_size: int, num_classes: int, seed: int = 42):
        self.labels = np.array(labels, dtype=np.int64)
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.rng = np.random.RandomState(seed)

        self.idx_by_cls = {c: np.where(self.labels == c)[0].tolist() for c in range(num_classes)}
        for c in range(num_classes):
            if len(self.idx_by_cls[c]) == 0:
                self.idx_by_cls[c] = []  # 空类，虽然不应该出现

        self.classes = [c for c in range(num_classes) if len(self.idx_by_cls[c]) > 0]
        self.samples_per_class = max(1, batch_size // max(1, len(self.classes)))  # 至少1
        # 更合理：每 batch 均匀抽 k 类（k=min(batch_size, 类数)），每类取 floor(batch_size/k) or ceil
        self.k = min(self.batch_size, len(self.classes))

    def __iter__(self):
        # 随机打乱每类索引
        pointers = {c: 0 for c in self.classes}
        for c in self.classes:
            self.rng.shuffle(self.idx_by_cls[c])

        # 估算一个 epoch 里 batch 数：按总样本量近似
        total = len(self.labels)
        num_batches = max(1, total // self.batch_size)

        for _ in range(num_batches):
            chosen = self.rng.choice(self.classes, size=self.k, replace=False if self.k <= len(self.classes) else True)
            per_cls = math.ceil(self.batch_size / self.k)
            batch_idx = []
            for c in chosen:
                need = per_cls
                while need > 0:
                    if len(self.idx_by_cls[c]) == 0:
                        break
                    if pointers[c] >= len(self.idx_by_cls[c]):
                        self.rng.shuffle(self.idx_by_cls[c]); pointers[c] = 0
                    batch_idx.append(self.idx_by_cls[c][pointers[c]])
                    pointers[c] += 1; need -= 1
            if len(batch_idx) > self.batch_size:
                batch_idx = batch_idx[:self.batch_size]
            yield batch_idx

    def __len__(self):
        total = len(self.labels)
        return max(1, total // self.batch_size)

# ========================= Mixup =========================
def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0: return x, (y, y), 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, (y, y[idx]), lam

def mixup_criterion(crit, pred, targets, lam):
    y_a, y_b = targets
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

# ========================= LDAM + DRW（长尾核心） =========================
class LDAMLoss(nn.Module):
    """
    LDAM: logits - m_y，其中 m_y 与类频率成反比（小类更大 margin）。
    参考: "Learning Imbalanced Datasets with Label-Distribution-Aware Margin Loss" (NeurIPS 2019)
    """
    def __init__(self, cls_num_list: List[int], s: float = 30.0, max_m: float = 0.5, weight: Optional[torch.Tensor] = None):
        super().__init__()
        m_list = 1.0 / np.power(np.array(cls_num_list) + 1e-12, 0.25)
        m_list = m_list / m_list.max() * max_m
        self.m_list = torch.tensor(m_list, dtype=torch.float32)
        self.s = s
        self.weight = weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        # logits: [B, K], target: [B]
        if logits.dtype != torch.float32:
            logits = logits.float()
        m = self.m_list.to(logits.device)[target]  # [B]
        # 构造 one-hot
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, target.view(-1, 1), 1)
        # 对目标类减 margin
        logits_m = logits.clone()
        logits_m[index] = logits_m[index] - m
        # 缩放
        logits_m = self.s * logits_m
        return F.cross_entropy(logits_m, target, weight=self.weight, label_smoothing=0.0)

def make_drw_weights(cnt: pd.Series, beta: float = 0.9999, num_classes: int = None, device=DEVICE):
    # “有效样本数”权重（Class-Balanced Weight, Cui et al. CVPR'19）
    if num_classes is None:
        num_classes = int(cnt.index.max()) + 1
    cls_num_list = [int(cnt.get(i, 0)) for i in range(num_classes)]
    effective_num = 1.0 - np.power(beta, cls_num_list)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-8)
    weights = weights / (np.sum(weights) + 1e-8) * num_classes
    return torch.tensor(weights, dtype=torch.float32, device=device), cls_num_list

# ========================= 评估（TTA + Multi-crop） =========================
@torch.no_grad()
def evaluate_with_multicrop(model, te_df, img_size, device,
                             use_tta=True, use_multicrop=True,
                             n_crops: int = 8, ratio_low: float = 0.85,
                             logit_adj_tau: float = 1.0):
    """评估：安全地做 TTA + 多裁片（自动把小图放大到至少 img_size 再裁）"""
    from math import ceil
    model.eval()
    organ = globals().get("current_organ", "unknown")
    val_tf = VAL_TF_REGISTRY.get(organ)

    def _ensure_min_size(pil_img, min_side_hw):
        """把 PIL 图像等比放大，使得 H>=min_side_hw 且 W>=min_side_hw 后再返回。"""
        H, W = pil_img.size[1], pil_img.size[0]
        if H >= min_side_hw and W >= min_side_hw:
            return pil_img
        scale = max(min_side_hw / max(1, H), min_side_hw / max(1, W))
        new_w = max(min_side_hw, int(ceil(W * scale)))
        new_h = max(min_side_hw, int(ceil(H * scale)))
        return pil_img.resize((new_w, new_h), Image.BICUBIC)

    def apply_val_tf(img):
        return val_tf(img).unsqueeze(0).to(device, non_blocking=True)

    def make_crops(img):
        """确定性多裁片；在进入 Five/TenCrop 之前确保图像边长≥img_size。"""
        if not use_multicrop or n_crops <= 1:
            return []

        crops = []
        # flower/leaf 走 Five/TenCrop，先放大到至少 img_size
        if organ in ("flower", "leaf"):
            img_big = _ensure_min_size(img, img_size)
            if n_crops >= 10:
                tc = transforms.TenCrop(img_size)
                out = tc(img_big)
            else:
                fc = transforms.FiveCrop(img_size)
                out = fc(img_big)
            norm = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
            ])
            for c in out:
                # Five/TenCrop 可能返回 PIL 或 Tensor 的元组；都转成张量
                if isinstance(c, Image.Image):
                    crops.append(norm(c).unsqueeze(0))
                else:  # Tensor
                    crops.append(transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))(c).unsqueeze(0))
            return crops

        # fruit/bark：规则网格裁片（我们先把短边放大到 ~1.2*img_size，以覆盖更多区域）
        resize_side = int(img_size * 1.20)
        # 确保 >= img_size，再放到 1.2*img_size 的短边
        img_safe = _ensure_min_size(img, img_size)
        img_r = transforms.Resize(resize_side)(img_safe)
        W, H = img_r.size
        # g×g 网格，g≈sqrt(n_crops)
        g = max(1, int(round(n_crops ** 0.5)))
        grid_rows, grid_cols = g, g
        sw = (W - img_size) // max(1, grid_cols - 1) if grid_cols > 1 else 0
        sh = (H - img_size) // max(1, grid_rows - 1) if grid_rows > 1 else 0
        norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
        ])
        for r in range(grid_rows):
            for c in range(grid_cols):
                left = min(c * sw, max(0, W - img_size))
                top  = min(r * sh, max(0, H - img_size))
                crop = img_r.crop((left, top, left + img_size, top + img_size))
                crops.append(norm(crop).unsqueeze(0))
        return crops

    # 取该器官的 log π(y)（若你在别处填充了 PRIOR_LOG_REGISTRY）
    log_prior = PRIOR_LOG_REGISTRY.get(organ, None)

    y_true, y_pred = [], []
    top1_hits, top3_hits = 0, 0

    for _, row in tqdm(te_df.iterrows(), total=len(te_df),
                       desc=f"[{organ}] Eval (TTA={'Y' if use_tta else 'N'}, MC={'Y' if use_multicrop else 'N'})"):
        path = row[COL_IMAGE]
        y = int(row["__species_local__"])
        try:
            with Image.open(path) as im:
                img = im.convert("RGB")
        except Exception:
            continue

        # 单裁片：val_tf 里面已有 Resize(int(img_size*1.05))，不改
        x = apply_val_tf(img)
        logits = model(x)
        if use_tta:
            logits = 0.5 * (logits + model(torch.flip(x, dims=[3])))

        # 多裁片
        crops = make_crops(img)
        if crops:
            for xx in crops:
                xx = xx.to(device, non_blocking=True)
                l = model(xx)
                if use_tta:
                    l = 0.5 * (l + model(torch.flip(xx, dims=[3])))
                logits += l
            logits = logits / (1 + len(crops))

        # 仅评估用的 Logit-Adjusted（若有先验）
        if log_prior is not None and logit_adj_tau is not None and logit_adj_tau > 0:
            logits = logits - logit_adj_tau * log_prior.view(1, -1).to(logits.device)

        probs = F.softmax(logits, dim=1)
        top3 = probs.topk(3, dim=1)
        pred1 = top3.indices[0, 0].item()
        top3_set = set(top3.indices[0].tolist())

        y_true.append(y); y_pred.append(pred1)
        if pred1 == y: top1_hits += 1
        if y in top3_set: top3_hits += 1

    if len(y_true) == 0:
        return 0.0, 0.0, 0.0, "N/A"

    top1 = top1_hits / len(y_true)
    top3 = top3_hits / len(y_true)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    labels_present = sorted(set(y_true) | set(y_pred))
    try:
        local2label = {int(k): v for v, k in json.load(
            open(OUT_DIR / f"species_local_map_{organ}.json", "r", encoding="utf-8")
        ).items()}
        target_names = [local2label.get(i, str(i)) for i in labels_present]
    except Exception:
        target_names = [str(i) for i in labels_present]

    report = classification_report(y_true, y_pred, labels=labels_present,
                                   target_names=target_names, zero_division=0)
    return top1, top3, macro_f1, report


# ========================= 固定切分（与你一致） =========================
def _load_fixed_split_or_fallback(df, organ, OUT_DIR, COL_ORGAN_TXT, COL_SPECIESID, gid2local):
    sp_dir = OUT_DIR / "splits"
    tr_csv = sp_dir / f"{organ}_train_split.csv"
    te_csv = sp_dir / f"{organ}_val_split.csv"
    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)

    if tr_csv.exists() and te_csv.exists():
        tr_df = pd.read_csv(tr_csv)
        te_df = pd.read_csv(te_csv)
        if "__species_local__" not in tr_df.columns:
            tr_df["__species_local__"] = tr_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        if len(te_df) > 0 and "__species_local__" not in te_df.columns:
            te_df["__species_local__"] = te_df[COL_SPECIESID].astype(int).astype(str).map(gid2local).astype(int)
        tr_df["__species_local__"] = tr_df["__species_local__"].astype(int)
        if len(te_df) > 0:
            te_df["__species_local__"] = te_df["__species_local__"].astype(int)
        return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)

    counts = sub["__species_local__"].value_counts()
    if (counts.min() >= 2) and (counts.shape[0] >= 2) and (len(sub) >= 4):
        tr_df, te_df = train_test_split(sub, test_size=0.2, stratify=sub["__species_local__"], random_state=42)
    else:
        tr_df, te_df = train_test_split(sub, test_size=0.2, shuffle=True, random_state=42)
    return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)

# ========================= 训练主函数（CAB + LDAM-DRW + EMA + Mixup 衰减） =========================
def train_one_species_model_for_organ(
    df: pd.DataFrame,
    organ: str,
    epochs_head: int = None,
    epochs_ft: int = None,
    bs: int = 64,
    base_lr_head: float = 1e-3,
    base_lr_ft_head: float = 1e-4,
    base_lr_ft_backbone: float = 1.5e-5,
    num_workers: int = 2,
    early_stop_patience: int = 5,
    scheduler_patience: int = 2,
    use_balanced_sampler: bool = True,        # ★ 开：启用类感知采样 CAB
    use_multicrop_eval: bool = True,
    multicrop_n: int = 8,
    multicrop_ratio_low: float = 0.85,        # 兼容参数
    freeze_bn_after_warmup: bool = True,
    use_mixup: bool = True,
    mixup_alpha: float = 0.2,
    use_ema: bool = True,
    ema_m: float = 0.999,
):
    global current_organ
    current_organ = organ = organ.lower().strip()

    # 轮次
    spec = ORGAN_SPEC.get(organ, ORGAN_SPEC["leaf"])
    epochs_head = epochs_head if epochs_head is not None else spec["epochs_head"]
    epochs_ft   = epochs_ft   if epochs_ft   is not None else spec["epochs_ft"]

    # 映射
    gid2local = json.load(open(OUT_DIR / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    local2gid = json.load(open(OUT_DIR / f"species_local2global_{organ}.json", "r", encoding="utf-8"))
    num_classes = len(local2gid)

    # 模型 & tf
    model, tf_pair, img_size = get_backbone_for_organ(organ, num_classes)
    train_tf, val_tf = tf_pair

    # 切分
    tr_df, te_df = _load_fixed_split_or_fallback(df, organ, OUT_DIR, COL_ORGAN_TXT, COL_SPECIESID, gid2local)
    assert set(te_df["__species_local__"].unique()).issubset(set(tr_df["__species_local__"].unique())), \
        f"[{organ}] Val 出现了训练缺失的类别，请清理或重建 OUT_DIR/splits 下的固定切分。"

    # === 先验分布（Laplace 平滑）→ 评估时做 logit-adjusted，缓解长尾过拟合/偏置 ===
    global PRIOR_LOG_REGISTRY
    cnt_vec = tr_df["__species_local__"].value_counts().reindex(range(num_classes), fill_value=0).astype(float).values
    prior = (cnt_vec + 1.0) / (cnt_vec.sum() + num_classes)  # Laplace smoothing
    PRIOR_LOG_REGISTRY[organ] = torch.log(torch.tensor(prior, dtype=torch.float32, device=DEVICE))


    # 类频
    cnt = tr_df["__species_local__"].value_counts()
    cb_weight, cls_num_list = make_drw_weights(cnt, beta=0.9999, num_classes=num_classes, device=DEVICE)

    # Dataset/Loader（CAB）
    tr_ds = BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", train_tf)
    te_ds = BasicImageDataset(te_df, COL_IMAGE, "__species_local__", val_tf)

    if use_balanced_sampler:
        sampler = ClassAwareBatchSampler(tr_df["__species_local__"].tolist(), bs, num_classes, seed=42)
        tr_ld = DataLoader(tr_ds, batch_sampler=sampler,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)
    else:
        tr_ld = DataLoader(tr_ds, batch_size=bs, shuffle=True,
                           num_workers=num_workers, pin_memory=True, persistent_workers=False,
                           collate_fn=drop_corrupt_collate)

    # Warmup：仅训练分类头，用 val 口径 tf + CE
    def freeze_backbone_only_train_head(m: nn.Module, organ_name: str):
        for p in m.parameters(): p.requires_grad = False
        if organ_name == "flower":
            for p in m.classifier[2].parameters(): p.requires_grad = True
        elif organ_name == "leaf":
            for p in m.fc.parameters(): p.requires_grad = True
        elif organ_name == "fruit":
            for p in m.classifier[1].parameters(): p.requires_grad = True
        elif organ_name == "bark":
            for p in m.classifier[2].parameters(): p.requires_grad = True

    freeze_backbone_only_train_head(model, organ)
    crit_warmup = nn.CrossEntropyLoss(label_smoothing=0.05)
    warmup_loader = DataLoader(
        BasicImageDataset(tr_df, COL_IMAGE, "__species_local__", VAL_TF_REGISTRY[organ]),
        batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True, collate_fn=drop_corrupt_collate
    )
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-3)
    model = model.to(DEVICE)

    # Sanity
    tmp_loader = DataLoader(
        BasicImageDataset(tr_df.sample(min(8, len(tr_df))), COL_IMAGE, "__species_local__", VAL_TF_REGISTRY[organ]),
        batch_size=min(8, bs), shuffle=True, num_workers=0, collate_fn=drop_corrupt_collate
    )
    x0, y0, _ = next(iter(tmp_loader)); x0, y0 = x0.to(DEVICE), y0.to(DEVICE)
    with torch.no_grad():
        logits0 = model(x0)
    assert logits0.shape[1] == num_classes, f"[{organ}] 头部维度不等于类数：{logits0.shape[1]} vs {num_classes}"
    print(f"[{organ}] SanityCheck: batch_logits={tuple(logits0.shape)}, label_range=({tr_df['__species_local__'].min()}, {tr_df['__species_local__'].max()})")

    for epoch in range(epochs_head):
        model.train()
        total, correct = 0, 0
        for x, y, _ in tqdm(warmup_loader, total=len(warmup_loader), desc=f"[{organ}] Warmup {epoch+1}/{epochs_head}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = crit_warmup(logits, y)
            opt_head.zero_grad(); loss.backward(); opt_head.step()
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
        print(f"[{organ}] Warmup {epoch+1}/{epochs_head} | Train Acc: {correct/max(1,total):.4f}")

    if freeze_bn_after_warmup:
        for m in model.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

    # 解冻高层
    def unfreeze_high_layers(m: nn.Module, organ_name: str):
        for p in m.parameters(): p.requires_grad = False
        if organ_name == "leaf":
            for name, p in m.named_parameters():
                if name.startswith("layer3") or name.startswith("layer4") or name.startswith("fc"):
                    p.requires_grad = True
        elif organ_name in ["flower", "bark"]:
            for name, p in m.named_parameters():
                if (name.startswith("features.7") or name.startswith("features.8") or
                    name.startswith("stages.2")  or name.startswith("stages.3")  or
                    name.startswith("classifier")):
                    p.requires_grad = True
            for p in getattr(m, "classifier").parameters(): p.requires_grad = True
        elif organ_name == "fruit":
            for name, p in m.named_parameters():
                if name.startswith("features.6") or name.startswith("features.7") or name.startswith("classifier"):
                    p.requires_grad = True
            for p in m.classifier.parameters(): p.requires_grad = True

    unfreeze_high_layers(model, organ)

    # 优化器 + 调度
    params_head, params_backbone = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (params_head if any(k in name for k in ["fc","classifier"]) else params_backbone).append(p)

    if organ in ("bark","flower"):
        opt_ft = torch.optim.AdamW([
            {"params": params_backbone, "lr": max(base_lr_ft_backbone, 1.5e-5), "weight_decay": 3e-4},
            {"params": params_head,     "lr": base_lr_ft_head,                 "weight_decay": 5e-4},
        ])
    else:
        opt_ft = torch.optim.AdamW([
            {"params": params_backbone, "lr": base_lr_ft_backbone, "weight_decay": 3e-4},
            {"params": params_head,     "lr": base_lr_ft_head,     "weight_decay": 5e-4},
        ])

    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    scheduler = CosineAnnealingWarmRestarts(opt_ft, T_0=max(4, epochs_ft//4), T_mult=1)

    # LDAM-DRW 损失（后半程开启）
    ldam_loss = LDAMLoss(cls_num_list=cls_num_list, s=30.0, max_m=0.5, weight=None)
    ce_loss   = nn.CrossEntropyLoss(weight=None, label_smoothing=0.05)

    # EMA
    ema_model = deepcopy(model).to(DEVICE) if use_ema else None
    if use_ema:
        for p in ema_model.parameters(): p.requires_grad = False

    # EarlyStopping
    class EarlyStopping:
        def __init__(self, patience=5, mode="max"):
            self.patience = patience; self.mode = mode
            self.best = -float("inf") if mode == "max" else float("inf")
            self.bad = 0; self.best_state = None
        def step(self, score, model):
            improved = score > self.best if self.mode == "max" else score < self.best
            if improved:
                self.best = score; self.bad = 0
                self.best_state = deepcopy(model.state_dict())
            else:
                self.bad += 1
        def stop(self): return self.bad >= self.patience
    stopper = EarlyStopping(patience=early_stop_patience, mode="max")

    for epoch in range(epochs_ft):
        model.train()
        total, correct = 0, 0

        # DRW：前半程不用类权，后半程用 CB 权重
        use_drw = (epoch >= epochs_ft // 2)
        curr_ce  = nn.CrossEntropyLoss(weight=(cb_weight if use_drw else None), label_smoothing=0.05)
        curr_ldam = LDAMLoss(cls_num_list=cls_num_list, s=30.0, max_m=0.5,
                             weight=(cb_weight if use_drw else None))

        # Mixup 线性衰减（最后30%关闭）
        mixup_frac = epoch / max(1, epochs_ft - 1)
        alpha_now = 0.0 if mixup_frac >= 0.70 else mixup_alpha * (1.0 - mixup_frac / 0.70)

        for x, y, _ in tqdm(tr_ld, total=len(tr_ld), desc=f"[{organ}] Finetune {epoch+1}/{epochs_ft}"):
            if x.numel() == 0: continue
            x, y = x.to(DEVICE), y.to(DEVICE)

            if use_mixup and alpha_now > 0:
                x, (ya, yb), lam = mixup_data(x, y, alpha=alpha_now)
                logits = model(x)
                # 前半程：CE；后半程：LDAM
                base_loss = curr_ce if not use_drw else curr_ldam
                loss = mixup_criterion(base_loss, logits, (ya, yb), lam)
            else:
                logits = model(x)
                base_loss = curr_ce if not use_drw else curr_ldam
                loss = base_loss(logits, y)

            opt_ft.zero_grad(); loss.backward(); opt_ft.step()

            if use_ema:
                with torch.no_grad():
                    for p_e, p in zip(ema_model.parameters(), model.parameters()):
                        p_e.data.mul_(ema_m).add_(p.data, alpha=1 - ema_m)

            pred = logits.argmax(1)
            correct += (pred == (y if not (use_mixup and alpha_now > 0) else ya)).sum().item()
            total += y.size(0)

        tr_acc = correct / max(total, 1)

        # 评估
        model_for_eval = ema_model if use_ema else model
        val_top1, val_top3, val_macro_f1, _ = evaluate_with_multicrop(
            model_for_eval, te_df, img_size, DEVICE,
            use_tta=True, use_multicrop=use_multicrop_eval, n_crops=multicrop_n
        )
        print(f"[{organ}] Finetune {epoch+1}/{epochs_ft} | Train Acc: {tr_acc:.4f} | "
              f"Val Top1: {val_top1:.4f} | Top3: {val_top3:.4f} | MacroF1: {val_macro_f1:.4f}")

        # 训练分布 @ 验证口径（每类最多2张）
        tr_bal = (tr_df.groupby("__species_local__").head(2)
                         .sample(frac=1.0, random_state=epoch).reset_index(drop=True))
        tr_top1, tr_top3, tr_f1, _ = evaluate_with_multicrop(
            model_for_eval, tr_bal, img_size, DEVICE, use_tta=True, use_multicrop=True, n_crops=max(5, multicrop_n)
        )
        print(f"[{organ}] Train(Balanced-Eval) Top1={tr_top1:.4f} | Top3={tr_top3:.4f} | F1={tr_f1:.4f}")

        scheduler.step(epoch + 1)
        stopper.step(val_top1, model_for_eval)
        if stopper.stop():
            print(f"[{organ}] Early stopping at epoch {epoch+1}.")
            break

    best_model = ema_model if (use_ema and stopper.best_state is not None) else model
    if stopper.best_state is not None:
        best_model.load_state_dict(stopper.best_state, strict=False)

    top1, top3, macro_f1, report = evaluate_with_multicrop(
        best_model, te_df, img_size, DEVICE, use_tta=True, use_multicrop=True, n_crops=max(5, multicrop_n)
    )
    print(f"✅ [{organ}] Final Test | Top1: {top1:.4f} | Top3: {top3:.4f} | MacroF1: {macro_f1:.4f}")
    print(report)

    torch.save(best_model.state_dict(), OUT_DIR / f"{organ}_species_model.pth")
    print(f"✅ 已保存: {OUT_DIR / f'{organ}_species_model.pth'}")


In [40]:
# ================================================
# Step 5: 依次训练 flower/leaf/fruit/bark 四个模型
# ================================================
'''organ_classes = json.load(open(OUT_DIR / "organ_classes.json", "r", encoding="utf-8"))

for organ in organ_classes.keys():
    # 防止极端小数据器官训练
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        epochs_head=3,          # 可调
        epochs_ft=5,            # 可调
        bs=64,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=1e-5,
    )  # ======== 原版主函数调用 ========
'''
'''for organ in organ_classes.keys():
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        # 不传 epochs_head/epochs_ft 则按 ORGAN_SPEC 默认（已为 flower/bark 提高）
        bs=64,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=5e-6,
        num_workers=2,                 # 若走网络盘可先设 0，稳定后再升
        early_stop_patience=5,
        scheduler_patience=2,
        use_balanced_sampler=True,     # ✅ 第一优先：均衡采样
        use_multicrop_eval=True,       # ✅ 第二优先：多裁片
        multicrop_n=8,                 # 8~10 对 bark/fruit 很有用
        multicrop_ratio_low=0.85,
    )'''

organ_classes = json.load(open(OUT_DIR / "organ_classes.json", "r", encoding="utf-8"))

for organ in organ_classes.keys():
    sub = df[df[COL_ORGAN_TXT] == organ]
    uniq_species = sub[COL_SPECIESID].nunique()
    if len(sub) < 50 or uniq_species < 2:
        print(f"⚠️ 跳过 {organ}（样本={len(sub)}, 物种={uniq_species}）")
        continue

    # —— 器官专属策略 —— #
    if organ == "flower":
        bs = 48
        base_lr_ft_backbone = 1e-5   # 适度放大，帮助大分辨率收敛
        freeze_bn = True
        mc_n = 8
    elif organ == "leaf":
        bs = 48
        base_lr_ft_backbone = 8e-6
        freeze_bn = True
        mc_n = 8
    elif organ == "fruit":
        bs = 40               # 分辨率 512，适当降低 batch
        base_lr_ft_backbone = 7e-6    # 小步走，稳
        freeze_bn = True
        mc_n = 10             # ✅ 提升评估稳定性（Top-3 会更稳）
    else:  # bark
        bs = 40
        base_lr_ft_backbone = 1.5e-5 
        freeze_bn = False
        mc_n = 8

    print(f"\n🎯 训练器官 [{organ}] —— 样本={len(sub)}, 物种={uniq_species}")
    train_one_species_model_for_organ(
        df=df,
        organ=organ,
        bs=bs,
        base_lr_head=1e-3,
        base_lr_ft_head=1e-4,
        base_lr_ft_backbone=base_lr_ft_backbone,
        num_workers=2,
        early_stop_patience=5,
        scheduler_patience=2,
        use_balanced_sampler=False,      # ✅ 必开
        use_multicrop_eval=True,        # ✅ 必开
        multicrop_n=mc_n,
        multicrop_ratio_low=0.85,
        freeze_bn_after_warmup=freeze_bn,  # ✅ 小 batch/高分辨率更稳
    )




🎯 训练器官 [bark] —— 样本=6630, 物种=345
[bark] SanityCheck: batch_logits=(8, 345), label_range=(0, 344)


[bark] Warmup 1/5: 100%|██████████| 133/133 [03:01<00:00,  1.37s/it]


[bark] Warmup 1/5 | Train Acc: 0.1368


[bark] Warmup 2/5: 100%|██████████| 133/133 [02:54<00:00,  1.31s/it]


[bark] Warmup 2/5 | Train Acc: 0.2806


...

[leaf] Finetune 10/10 | Train Acc: 0.4118 | Val Top1: 0.0169 | Top3: 0.0352 | MacroF1: 0.0149


[leaf] Eval (TTA=Y, MC=Y): 100%|██████████| 888/888 [02:14<00:00,  6.59it/s]


[leaf] Train(Balanced-Eval) Top1=0.0248 | Top3=0.0462 | F1=0.0212
[leaf] Early stopping at epoch 10.


[leaf] Eval (TTA=Y, MC=Y):  73%|███████▎  | 18180/25041 [45:03<17:00,  6.73it/s] 


KeyboardInterrupt: 

In [185]:
# 复现与你训练函数一致的清洗与切分（无需训练）
import json
import pandas as pd
from sklearn.model_selection import train_test_split

def recreate_splits_for_organ(df: pd.DataFrame, organ: str,
                              COL_IMAGE="image_path",
                              COL_ORGAN_TXT="organ",
                              COL_SPECIESID="species_id",
                              OUT_DIR=None):
    organ = organ.lower().strip()
    assert OUT_DIR is not None, "请传入 OUT_DIR 路径对象或字符串"

    # 读取映射（与你训练时一致）
    gid2local = json.load(open(OUT_DIR / f"species_global2local_{organ}.json", "r", encoding="utf-8"))
    # 构造子集 & 清洗（与你训练函数一致）
    sub = df[df[COL_ORGAN_TXT] == organ].copy()
    sub["__species_local__"] = sub[COL_SPECIESID].astype(int).astype(str).map(gid2local)
    sub = sub.dropna(subset=["__species_local__"]).copy()
    sub["__species_local__"] = sub["__species_local__"].astype(int)

    # 与训练一致的“优先分层”切分逻辑
    counts = sub["__species_local__"].value_counts()
    min_count = counts.min() if len(counts) else 0
    if (min_count >= 2) and (counts.shape[0] >= 2) and (len(sub) >= 4):
        tr_df, te_df = train_test_split(sub, test_size=0.2,
                                        stratify=sub["__species_local__"], random_state=42)
    else:
        tr_df, te_df = train_test_split(sub, test_size=0.2,
                                        shuffle=True, random_state=42)
    return tr_df.reset_index(drop=True), te_df.reset_index(drop=True)

# ==== 用法：快速复现 bark 的 tr_df/te_df 并做检查 ====
# 假设你全局已经有 df, OUT_DIR, 以及常量 COL_*；若列名不同替换上面默认参数即可
tr_df, te_df = recreate_splits_for_organ(df, "bark",
                                         COL_IMAGE=COL_IMAGE,
                                         COL_ORGAN_TXT=COL_ORGAN_TXT,
                                         COL_SPECIESID=COL_SPECIESID,
                                         OUT_DIR=OUT_DIR)

# 1) 验证集是否都能映射到 local？
gid2local = json.load(open(OUT_DIR / "species_global2local_bark.json", "r", encoding="utf-8"))
miss = te_df[~te_df[COL_SPECIESID].astype(int).astype(str).isin(gid2local.keys())]
print("VAL 映射缺失样本数:", len(miss))

# 2) 训练/验证类集合是否差别很大？
tr_set = set(tr_df["__species_local__"].unique())
te_set = set(te_df["__species_local__"].unique())
print("仅训练有而验证没有的类数:", len(tr_set - te_set))
print("仅验证有而训练没有的类数:", len(te_set - tr_set))

# 3) （可选）随手看下验证集类分布
print("VAL 类分布（top10）:")
print(te_df["__species_local__"].value_counts().head(10))


VAL 映射缺失样本数: 0
仅训练有而验证没有的类数: 112
仅验证有而训练没有的类数: 11
VAL 类分布（top10）:
__species_local__
84     39
196    38
66     38
18     29
177    28
37     25
5      23
162    22
53     22
317    22
Name: count, dtype: int64


In [ ]:
val_counts = te_df['species'].value_counts()
print((val_counts <= 1).sum(), "classes have <=1 sample in val")


NameError: name 'tr_df' is not defined

In [11]:
tr_slice = tr_df.sample(frac=0.1, random_state=7)
print("slice 类计数:\n", tr_slice["__species_local__"].value_counts().head())


NameError: name 'tr_df' is not defined

In [19]:
####   重构step6，，，，之前step6 误删了一些函数，现在重新写一下、
####   但是保留上面的情况，因为是唯一可以运行的，因此要重新写，从头写


# =====================================================
# Step 6: 推理/上线：批量图片 → 器官 → 分器官种类 → 融合
# =====================================================
# === 使用代码1（OrgansHier）作为器官分类器 ===
# 放在 Step 6 同一位置，替换原 load_organ_classifier

import os, json
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torchvision.models import (
    ResNet50_Weights,
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
)

class OrgansHier(nn.Module):
    def __init__(self, backbone: str = 'convnext_small', pretrained: bool = True, drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.gate = nn.Linear(feat_dim, 2)     # 0: FL (flower+leaf), 1: FB (fruit+bark)
        self.head_fl = nn.Linear(feat_dim, 2)  # [flower, leaf]
        self.head_fb = nn.Linear(feat_dim, 2)  # [fruit, bark]

    def forward(self, x, temperature: float = 1.0):
        feats = self.backbone(x)
        gate_logits = self.gate(feats) / temperature
        fl_logits   = self.head_fl(feats) / temperature
        fb_logits   = self.head_fb(feats) / temperature
        g    = F.softmax(gate_logits, dim=1)[:, 0]  # P(FL)
        p_fl = F.softmax(fl_logits,   dim=1)        # [flower, leaf]
        p_fb = F.softmax(fb_logits,   dim=1)        # [fruit, bark]
        P = torch.stack([
            g * p_fl[:, 0],          # flower
            g * p_fl[:, 1],          # leaf
            (1 - g) * p_fb[:, 0],    # fruit
            (1 - g) * p_fb[:, 1],    # bark
        ], dim=1).clamp_min(1e-8)    # shape [B,4]
        return P  # 概率分布（已归一）


# ============ 仅软路由版本：器官TopK ============
def organ_predict_topk(
    image_paths,
    model,
    tfm,
    id2organ,
    temperature: float = 1.0,
    k: int = 2,
):
    """支持 str / Path / List[str]；仅软路由场景用到器官TopK概率"""
    # --- 输入归一化 ---
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]

        image_paths = [str(p) for p in image_paths]

    # --- 文件校验 ---
    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    # --- 预处理 ---
    xs = []
    for p in image_paths:
        with Image.open(p) as im:
            img = im.convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)

    # --- 设备对齐（保险）---
    if next(model.parameters()).device != batch.device:
        model = model.to(batch.device)

    with torch.no_grad():
        probs = model(batch, temperature=temperature)  # [B,4], 已归一
        topk = min(k, probs.size(1))
        confs_k, ids_k = torch.topk(probs, k=topk, dim=1)  # [B,k]

    # Python 化
    ids_k = ids_k.cpu().tolist()
    confs_k = confs_k.cpu().tolist()
    top1_org = [id2organ[row[0]] for row in ids_k]
    top1_conf = [row[0] for row in confs_k]
    return ids_k, confs_k, top1_org, top1_conf


def load_organ_classifier(path: Path = OUT_DIR / "organ_classification_mybest.pth"):
    """
    使用层级门控器官分类器；meta 中存放：backbone、img_size、T（温度，若做过温度标定）
    """
    # 1) 加载 pack
    pack = torch.load(path, map_location=DEVICE)

    # 2) 构建模型（要和保存时保持一致）
    model = OrgansHier(
        backbone=pack.get("backbone", "convnext_base"),
        pretrained=False,
        drop_path_rate=pack.get("drop_path_rate", 0.3)
    ).to(DEVICE)

    # 3) 加载权重（忽略额外的键）
    model.load_state_dict(pack["state_dict"], strict=False)
    model.eval()

    # 4) 恢复配置
    id2organ = {v: k for k, v in pack["organ2id"].items()}  # organ id -> 名称
    organs = pack["organs"]                                # ["bark","flower","fruit","leaf"]
    cfg = pack.get("cfg", None)                            # 训练时的配置字典
    temperature = pack.get("temperature", 1.0)             # 温度标定值

    # 5) 构建推理时 transform
    img_size = cfg["img_size"] if cfg and "img_size" in cfg else 384
    tfm = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    ])

    return model, id2organ, tfm, temperature



# === 全局缓存，避免重复load ===
_SPECIES_MODEL_CACHE = {}

def load_species_model_for_organ_cached(organ: str):
    organ = organ.lower().strip()
    if organ in _SPECIES_MODEL_CACHE:
        return _SPECIES_MODEL_CACHE[organ]

    # 读映射
    local2gid   = json.load(open(OUT_DIR / f"species_local2global_{organ}.json", "r", encoding="utf-8"))
    label2local = json.load(open(OUT_DIR / f"species_local_map_{organ}.json", "r", encoding="utf-8"))
    id2label = {int(v): k for k, v in label2local.items()}
    num_classes = len(local2gid)

    # 注意：get_backbone_for_organ 返回 (model, val_tf)
    model, tfm = get_backbone_for_organ(organ, num_classes)



    sd = torch.load(OUT_DIR / f"{organ}_species_model.pth", map_location=DEVICE)
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    model.load_state_dict(sd, strict=True)
    model = model.to(DEVICE).eval()

    _SPECIES_MODEL_CACHE[organ] = (model, tfm, id2label)
    return _SPECIES_MODEL_CACHE[organ]




# 返回：list[dict]，与 image_paths 等长，元素形如 {"label_probs": {species_name: prob, ...}}
@torch.inference_mode()
def species_predict_batch_dict(image_paths: List[str], organ: str) -> List[Dict[str, float]]:
    model, tfm, id2label = load_species_model_for_organ_cached(organ)
    xs = []
    for p in image_paths:
        img = Image.open(p).convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)
    logits = model(batch)
    probs = F.softmax(logits, dim=1).cpu().numpy()  # [B, K_local]

    out = []
    for row in probs:
        d = { id2label[i]: float(row[i]) for i in range(len(row)) }
        out.append(d)
    return out



# ============ 仅软路由版本：批量图片 → 融合物种 ============
def predict_species_for_batch_images_soft(
    image_paths,
    organ_topk: int = 2,
    tau_min: float = 0.10,         # 软路由：低置信度下限
    alpha_sharpen: float = 1.5,    # 软路由：幂次锐化
):
    """
    仅软路由：不走硬路由；支持 str / Path / List[str]
    依赖：load_organ_classifier、load_species_model_for_organ_cached、species_predict_batch_dict、DEVICE
    """
    # --- 输入归一化 ---
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]

        image_paths = [str(p) for p in image_paths]

    # --- 文件校验 ---
    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    # 1) 器官 Top-k（来自你上面的 load_organ_classifier）
    organ_model, id2organ, organ_tfm, organ_T = load_organ_classifier(OUT_DIR / "organ_classification_mybest.pth")
    ids_k, confs_k, top1_org_text, top1_conf = organ_predict_topk(
        image_paths, organ_model, organ_tfm, id2organ, temperature=organ_T, k=organ_topk
    )

    # 2) 仅软路由：对每张图计算器官权重（下限 + 幂次 + 归一化）
    per_image_route = []
    for i in range(len(image_paths)):
        organ_ids = ids_k[i]   # [k]
        organ_ps  = confs_k[i] # [k]
        w = np.array([max(p, tau_min) for p in organ_ps], dtype=np.float32)
        w = w ** alpha_sharpen
        w = w / (w.sum() + 1e-12)
        sel = list(zip(organ_ids, w.tolist()))  # [(organ_id, weight), ...]
        per_image_route.append(sel)

    # 3) 分桶：按“器官文本”聚合需要推断的图片索引，减少重复推断
    buckets = {}  # {organ_txt: [image_indices]}
    for i, sels in enumerate(per_image_route):
        for oid, _ in sels:
            organ_txt = id2organ[int(oid)]
            buckets.setdefault(organ_txt, []).append(i)

    # 4) 每个器官桶一次性跑种类头，得到 list[dict(物种->概率)]
    sp_probs = {}  # {organ_txt: list_of_dict_per_image}
    for organ_txt, idxs in buckets.items():
        paths = [image_paths[j] for j in idxs]
        # species_predict_batch_dict 内部自行做设备前向；如无请对齐设备
        sp_probs[organ_txt] = species_predict_batch_dict(paths, organ_txt)

    # 5) 融合每张图的物种概率：score(s) = sum_over_organs [ w_o * p(s|o) ]
    final = []
    for i, imgp in enumerate(image_paths):
        agg = {}
        for oid, w in per_image_route[i]:
            organ_txt = id2organ[int(oid)]
            # 在对应器官桶里找到当前图片的位置
            idx_in_bucket = buckets[organ_txt].index(i)
            d = sp_probs[organ_txt][idx_in_bucket]  # dict: species->prob
            for s, p in d.items():
                agg[s] = agg.get(s, 0.0) + w * p

        if not agg:
            final.append({
                "image": imgp,
                "organ_top1": top1_org_text[i],
                "organ_top1_conf": round(top1_conf[i], 4),
                "final_species": None,
                "final_conf": 0.0,
                "top5": []
            })
            continue

        # 归一化 & Top5
        z = sum(agg.values()) + 1e-12
        for k in agg: agg[k] /= z
        top = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:3]

        final.append({
            "image": imgp,
            "organ_top1": top1_org_text[i],
            "organ_top1_conf": round(top1_conf[i], 4),
            "final_species": top[0][0],
            "final_conf": round(float(top[0][1]), 4),
            "top3": [(k, round(float(v), 4)) for k, v in top]
        })

    # 6) 可选：全批“投票”
    vote = {}
    for item in final:
        if item["final_species"] is not None:
            vote[item["final_species"]] = vote.get(item["final_species"], 0.0) + item["final_conf"]
    tot = sum(vote.values()) + 1e-12
    vote = {k: round(v/tot, 4) for k, v in vote.items()}
    final_best = max(vote.items(), key=lambda x: x[1])[0] if vote else None

    return {
        "final_species": final_best,
        "vote_scores": vote,
        "details": final
    }



In [20]:
res = predict_species_for_batch_images_soft(
    "/mnt/e/code/plants-classification-conda/real_brazil_Carica papaya_fruit.png", 
    organ_topk=2,            # 建议 2；也可 3
    tau_min=0.10,            # 低置信度下限
    alpha_sharpen=1.5        # 器官权重幂次锐化
)
res

/tmp/ipykernel_3591/336660184.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pack = torch.load(path, map_location=DEVICE)


NameError: name 'get_backbone_for_organ' is not defined

### 重启内核后可以直接使用的推理模块
### 推理推理推理！！！！

In [17]:

# =====================
# 1) 器官层级门控分类器
# =====================

import os, json
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torchvision.models import (
    ResNet50_Weights,
    ConvNeXt_Small_Weights,
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
)
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("./real_data")
assert OUT_DIR.exists(), "缺少 ./outputs 目录，请确认训练产物是否已保存到此处。"


class OrgansHier(nn.Module):
    def __init__(self, backbone: str = 'convnext_small', pretrained: bool = True, drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.gate = nn.Linear(feat_dim, 2)     # 0: FL (flower+leaf), 1: FB (fruit+bark)
        self.head_fl = nn.Linear(feat_dim, 2)  # [flower, leaf]
        self.head_fb = nn.Linear(feat_dim, 2)  # [fruit, bark]

    def forward(self, x, temperature: float = 1.0):
        feats = self.backbone(x)
        gate_logits = self.gate(feats) / temperature
        fl_logits   = self.head_fl(feats) / temperature
        fb_logits   = self.head_fb(feats) / temperature
        g    = F.softmax(gate_logits, dim=1)[:, 0]  # P(FL)
        p_fl = F.softmax(fl_logits,   dim=1)        # [flower, leaf]
        p_fb = F.softmax(fb_logits,   dim=1)        # [fruit, bark]
        P = torch.stack([
            g * p_fl[:, 0],          # flower
            g * p_fl[:, 1],          # leaf
            (1 - g) * p_fb[:, 0],    # fruit
            (1 - g) * p_fb[:, 1],    # bark
        ], dim=1).clamp_min(1e-8)    # [B,4]
        return P

# =====================
# 2) 仅推理用 eval transform
# =====================

def build_eval_tf(organ: str, img_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    ])

# =====================
# 3) 推理骨干（只返回 model.eval, val_tf, img_size）
#    与训练的选择保持一致；如训练时有不同，请同步这里
# =====================

def get_infer_backbone_for_organ(organ: str, num_classes: int) -> Tuple[nn.Module, transforms.Compose, int]:
    organ = str(organ).lower().strip()
    img_size = 384

    if organ == "flower":
        # 升级为 ConvNeXt-Small（若环境不支持可改回 Tiny）
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_feat, num_classes)

    elif organ == "leaf":
        weights = ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        model.fc = nn.Linear(in_feat, num_classes)

    elif organ == "fruit":
        # 也可试 ConvNeXt-Tiny；先保留 B0 以兼顾速度
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_feat, num_classes)

    elif organ == "bark":
        # 关键改动：从 MobileNetV3-Small 换到 ConvNeXt-Tiny（或 Small）
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_feat, num_classes)
        
    else:
        raise ValueError(f"未知器官: {organ}")

    val_tf = build_eval_tf(organ, img_size)
    return model.to(DEVICE).eval(), val_tf, img_size

# =====================
# 4) 保护性断言：权重与头部维度匹配
# =====================

def _infer_last_linear(module: nn.Module) -> nn.Linear | None:
    # 在常见结构中找最后一个 Linear 头
    last = None
    for m in module.modules():
        if isinstance(m, nn.Linear):
            last = m
    return last


def assert_state_dict_compat(model: nn.Module, sd: Dict[str, torch.Tensor], num_classes: int):
    try:
        # 快速路径：根据权重里最后一个 Linear 的 out_features 判定
        keys = [k for k in sd.keys() if k.endswith("weight") and sd[k].dim() == 2]
        if keys:
            w = sd[keys[-1]]
            if w.size(0) != num_classes:
                raise AssertionError(f"state_dict 最后一层 out_features={w.size(0)} 与 num_classes={num_classes} 不一致")
    except Exception:
        # 回退：从 model 上拿最后线性层
        head = _infer_last_linear(model)
        if head is not None and getattr(head, 'out_features', None) != num_classes:
            raise AssertionError(
                f"模型头部 out_features={head.out_features} 与 num_classes={num_classes} 不一致")

# =====================
# 5) 器官分类器加载（pack 格式）
# =====================

def load_organ_classifier(path: Path | str = OUT_DIR / "organ_classification_mybest.pth"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"器官分类权重不存在: {path}")

    pack = torch.load(path, map_location=DEVICE)

    model = OrgansHier(
        backbone=pack.get("backbone", "convnext_base"),
        pretrained=False,
        drop_path_rate=pack.get("drop_path_rate", 0.3)
    ).to(DEVICE)
    model.load_state_dict(pack["state_dict"], strict=False)
    model.eval()

    id2organ = {v: k for k, v in pack["organ2id"].items()}
    cfg = pack.get("cfg", {})
    temperature = float(pack.get("temperature", 1.0))

    img_size = int(cfg.get("img_size", 384))
    tfm = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    ])
    return model, id2organ, tfm, temperature

# =====================
# 6) 器官 Top-K 预测（软路由用）
# =====================
@torch.no_grad()
def organ_predict_topk(
    image_paths,
    model: nn.Module,
    tfm: transforms.Compose,
    id2organ: Dict[int, str],
    temperature: float = 1.0,
    k: int = 2,
):
    # 归一化
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    image_paths = [str(p) for p in image_paths]

    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    xs = []
    for p in image_paths:
        with Image.open(p) as im:
            img = im.convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)

    if next(model.parameters()).device != batch.device:
        model = model.to(batch.device)

    probs = model(batch, temperature=temperature)  # [B,4]
    topk = min(k, probs.size(1))
    confs_k, ids_k = torch.topk(probs, k=topk, dim=1)
    ids_k   = ids_k.cpu().tolist()
    confs_k = confs_k.cpu().tolist()

    top1_org  = [id2organ[row[0]] for row in ids_k]
    top1_conf = [row[0] for row in confs_k]
    return ids_k, confs_k, top1_org, top1_conf

# =====================
# 7) 物种模型缓存 & 加载（推理）
# =====================
_SPECIES_MODEL_CACHE: Dict[str, Tuple[nn.Module, transforms.Compose, Dict[int,str]]] = {}

def load_species_model_for_organ_cached(organ: str):
    organ = organ.lower().strip()
    if organ in _SPECIES_MODEL_CACHE:
        return _SPECIES_MODEL_CACHE[organ]

    map_local2global = OUT_DIR / f"species_local2global_{organ}.json"
    map_label2local  = OUT_DIR / f"species_local_map_{organ}.json"
    if not map_local2global.exists() or not map_label2local.exists():
        raise FileNotFoundError(f"缺少映射：{map_local2global} 或 {map_label2local}")

    local2gid   = json.load(open(map_local2global, "r", encoding="utf-8"))
    label2local = json.load(open(map_label2local,  "r", encoding="utf-8"))
    id2label = {int(v): k for k, v in label2local.items()}
    num_classes = len(local2gid)

    model, val_tfm, _ = get_infer_backbone_for_organ(organ, num_classes)

    weight_path = OUT_DIR / f"{organ}_species_model.pth"
    if not weight_path.exists():
        raise FileNotFoundError(f"物种模型权重不存在: {weight_path}")
    sd = torch.load(weight_path, map_location=DEVICE)
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd

    # 保护性断言：头维度匹配
    assert_state_dict_compat(model, sd, num_classes)

    model.load_state_dict(sd, strict=True)
    model = model.to(DEVICE).eval()

    _SPECIES_MODEL_CACHE[organ] = (model, val_tfm, id2label)
    return _SPECIES_MODEL_CACHE[organ]

# =====================
# 8) 批量物种预测（返回：list[dict: species->prob]）
# =====================
@torch.inference_mode()
def species_predict_batch_dict(image_paths: List[str], organ: str) -> List[Dict[str, float]]:
    model, tfm, id2label = load_species_model_for_organ_cached(organ)

    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    xs = []
    for p in image_paths:
        with Image.open(p) as im:
            img = im.convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)
    logits = model(batch)
    probs = F.softmax(logits, dim=1).cpu().numpy()  # [B, K_local]

    out: List[Dict[str, float]] = []
    for row in probs:
        d = { id2label[i]: float(row[i]) for i in range(len(row)) }
        out.append(d)
    return out

# =====================
# 9) 软路由融合：批量图片 → 物种
# =====================

def predict_species_for_batch_images_soft(
    image_paths,
    organ_topk: int = 2,
    tau_min: float = 0.10,         # 软路由：低置信度下限
    alpha_sharpen: float = 1.5,    # 软路由：幂次锐化
):
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    image_paths = [str(p) for p in image_paths]

    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    # 1) 器官 Top-k
    organ_model, id2organ, organ_tfm, organ_T = load_organ_classifier(OUT_DIR / "organ_classification_mybest.pth")
    ids_k, confs_k, top1_org_text, top1_conf = organ_predict_topk(
        image_paths, organ_model, organ_tfm, id2organ, temperature=organ_T, k=organ_topk
    )

    # 2) 软路由权重（下限 + 幂次 + 归一化）
    per_image_route = []
    for i in range(len(image_paths)):
        organ_ids = ids_k[i]   # [k]
        organ_ps  = confs_k[i] # [k]
        w = np.array([max(float(p), tau_min) for p in organ_ps], dtype=np.float32)
        w = w ** float(alpha_sharpen)
        w = w / (w.sum() + 1e-12)
        sel = list(zip(organ_ids, w.tolist()))  # [(organ_id, weight), ...]
        per_image_route.append(sel)

    # 3) 分桶：按器官聚合需要推断的图片索引
    buckets: Dict[str, List[int]] = {}
    for i, sels in enumerate(per_image_route):
        for oid, _ in sels:
            organ_txt = id2organ[int(oid)]
            buckets.setdefault(organ_txt, []).append(i)

    # 4) 每个器官桶一次性跑种类头
    sp_probs: Dict[str, List[Dict[str,float]]] = {}
    for organ_txt, idxs in buckets.items():
        paths = [image_paths[j] for j in idxs]
        sp_probs[organ_txt] = species_predict_batch_dict(paths, organ_txt)

    # 5) 融合：score(s) = sum_o w_o * p(s|o)
    final = []
    for i, imgp in enumerate(image_paths):
        agg: Dict[str, float] = {}
        for oid, w in per_image_route[i]:
            organ_txt = id2organ[int(oid)]
            idx_in_bucket = buckets[organ_txt].index(i)
            d = sp_probs[organ_txt][idx_in_bucket]
            for s, p in d.items():
                agg[s] = agg.get(s, 0.0) + w * p

        if not agg:
            final.append({
                "image": imgp,
                "organ_top1": top1_org_text[i],
                "organ_top1_conf": round(float(top1_conf[i]), 4),
                "final_species": None,
                "final_conf": 0.0,
                "top3": []
            })
            continue

        # 归一化 & Top3
        z = sum(agg.values()) + 1e-12
        for k in list(agg.keys()):
            agg[k] = agg[k] / z
        top = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:3]

        final.append({
            "image": imgp,
            "organ_top1": top1_org_text[i],
            "organ_top1_conf": round(float(top1_conf[i]), 4),
            "final_species": top[0][0],
            "final_conf": round(float(top[0][1]), 4),
            "top3": [(k, round(float(v), 4)) for k, v in top]
        })

    # 6) 可选：全批“投票”
    vote: Dict[str, float] = {}
    for item in final:
        if item["final_species"] is not None:
            vote[item["final_species"]] = vote.get(item["final_species"], 0.0) + item["final_conf"]
    tot = sum(vote.values()) + 1e-12
    if tot > 0:
        vote = {k: round(v/tot, 4) for k, v in vote.items()}
        final_best = max(vote.items(), key=lambda x: x[1])[0]
    else:
        vote = {}
        final_best = None

    return {
        "final_species": final_best,
        "vote_scores": vote,
        "details": final
    }

# =====================
# 10) 单图包装（可选）
# =====================

def predict_species_for_single_image_soft(
    image_path: str | Path,
    organ_topk: int = 2,
    tau_min: float = 0.10,
    alpha_sharpen: float = 1.5,
):
    result = predict_species_for_batch_images_soft(
        image_paths=[str(image_path)],
        organ_topk=organ_topk,
        tau_min=tau_min,
        alpha_sharpen=alpha_sharpen,
    )
    # 返回更友好的单图结构
    details = result["details"][0] if result["details"] else {}
    return {
        "final_species": result.get("final_species"),
        "vote_scores": result.get("vote_scores"),
        "image": details.get("image"),
        "organ_top1": details.get("organ_top1"),
        "organ_top1_conf": details.get("organ_top1_conf"),
        "top3": details.get("top3", [])
    }



In [18]:
res = predict_species_for_single_image_soft(
    "/mnt/e/code/plants-classification-conda/real_brazil_Cariniana estrellensis_flower.png", 
    organ_topk=2,            # 建议 2；也可 3
    tau_min=0.10,            # 低置信度下限
    alpha_sharpen=1.5        # 器官权重幂次锐化
)
res

/tmp/ipykernel_6522/830194766.py:145: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pack = torch.load(path, map_location=DEVICE)
/tmp/ipykernel_6522/830194766.py:234: Future

RuntimeError: Error(s) in loading state_dict for ConvNeXt:
	Missing key(s) in state_dict: "classifier.2.weight", "classifier.2.bias". 
	Unexpected key(s) in state_dict: "classifier.2.1.weight", "classifier.2.1.bias". 

In [19]:
sd = torch.load('outputs/bark_species_model.pth')
keys = sd['state_dict'].keys() if 'state_dict' in sd else sd.keys()
print(next(iter(keys)))


/tmp/ipykernel_6522/3815276054.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load('outputs/bark_species_model.pth')


features.0.0.weight


## plant2文件的全流程
### 替换其中的器官+种类分类器
### 已成功替换！！！！
### 完整功能都要有：是否是植物、植物种类判断、是否在我数据集中、是否有病虫害
### 输出逻辑不变，格式不变


In [1]:
# ================================ 新模型测试 - Step3 ===============================

import os
import json
import torch
import pandas as pd
import numpy as np
import open_clip
from PIL import Image
from tqdm import tqdm
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ====== Step 1: 初始化设备与数据 ======
csv_path     = "/mnt/e/code/plants-classification-conda/real_data/plant_Brazil.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
df     = pd.read_csv(csv_path)



id_to_species = {v: k for k, v in df[['label', 'species']].drop_duplicates().values}

# ===== MODIFIED 1: 划分训练集和测试集 =====

# ====== Step 2: 定义 Transform ======
'''organ_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])
species_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])'''



# ====== Step 4: 加载 CLIP，用于植物/非植物 & 病虫害 ======
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14', pretrained='openai'
)
clip_model = clip_model.to(device).eval()



/home/jmy/miniconda3/envs/plant310/lib/python3.10/site-packages/open_clip/factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(


In [2]:
# 植物检测函数（保持不变）
def train_is_plant_clip(image_path):
    # 确认路径存在且不是目录
    if not os.path.exists(image_path):
        print(f"⚠️ 路径不存在: {image_path}")
        return False
    if os.path.isdir(image_path):
        print(f"⚠️ 路径是目录, 不是图片文件: {image_path}")
        return False

    labels = ["a photo of a plant", "a photo of an animal", "a photo of a person", "a photo of an object","a photo of a bottle",
    "a photo of an object","a logo or text on white background","a painting or cartoon of a plant","a green text on white background"]
    


    with torch.no_grad():
        image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        text_tokens = open_clip.tokenize(labels).to(device)
        img_feat = clip_model.encode_image(image)
        txt_feat = clip_model.encode_text(text_tokens)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
        logits = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1).squeeze()
    top_idx = logits.argmax().item()
    top_label = labels[top_idx]
    #print(f"🌱 是否植物判断：Top类 = {top_label}, 得分 = {logits[top_idx]:.4f}")
    return top_label == "a photo of a plant"

In [3]:
# ===== ORIGINAL: 病虫害检测函数（保持不变） =====
def is_diseased_train(image_paths, vote_threshold=0.7):
    # ... original voting prompts and code unchanged ...
    healthy_prompts = [
        "The leaves of healthy plants are usually bright green",
        "The leaves of healthy plants are usually full and shiny, with no obvious signs of disease or insect damage on the leaf surface",
        "Healthy plants usually have strong, straight stems that are able to support the weight of the plant",
        "Healthy plants will show vigorous growth, including sprouting new leaves, extending branches and blooming flowers",
        "Healthy plant leaves have clear veins and are not excessively curled or wrinkled",
        "A healthy plant has bright flowers with intact petals and no wilting, falling off, or diseased spots",
        "The fruit of a healthy plant is full and has no cracks, rot or lesions. The fruit skin is normal color"
    ]
    diseased_prompts = [
        "Unhealthy plant leaves or flowers will have spots or patches of different shapes, sizes and colors, such as round, oval, polygonal, wheel-shaped",
        "Unhealthy plants have curled, shrunken, twisted leaves and flowers, and misshapen and stunted flowers",
        "Tumor-like protrusions appear on the stem, such as rose cancer, and swelling occurs",
        'Soft rot, wet rot or dry rot on the stem',
        "Unhealthy plants may have holes, nicks, or signs of being eaten on their leaves and petals",
        "Unhealthy plants may have visible insects, such as aphids and spider mites. Some pests will leave spider web-like silk",
        "Leaves lose their normal green color, show yellowing symptoms, partially or completely die, and appear brown or black",
        "The petals may appear water-soaked, rotten, softened, or even completely rotten."
    ]
    feats = []
    for p in image_paths:
        try:
            img = preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
            with torch.no_grad():
                f = clip_model.encode_image(img)
                feats.append(f / f.norm(dim=-1, keepdim=True))
        except:
            continue
    if not feats:
        print("⚠️ 无有效图像特征，无法判断病虫害")
        return False
    img_feat = torch.mean(torch.stack(feats), dim=0, keepdim=True)
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    votes, total = 0, 0
    with torch.no_grad():
        for hp, dp in zip(healthy_prompts, diseased_prompts):
            toks = open_clip.tokenize([hp, dp]).to(device)
            txt_feat = clip_model.encode_text(toks)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            logit = (100 * img_feat @ txt_feat.T).softmax(dim=-1).squeeze()
            total += 1
            votes += (logit.argmax().item() == 1)
    return (votes / total) >= vote_threshold


In [7]:
# ----------organ-species-classification----------


# =====================
# 1) 器官层级门控分类器
# =====================

import os, json
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torchvision.models import (
    ResNet50_Weights,
    ConvNeXt_Small_Weights,
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Small_Weights,
)
import timm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path("./real_data")
assert OUT_DIR.exists(), "缺少 ./outputs 目录，请确认训练产物是否已保存到此处。"


class OrgansHier(nn.Module):
    def __init__(self, backbone: str = 'convnext_small', pretrained: bool = True, drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.gate = nn.Linear(feat_dim, 2)     # 0: FL (flower+leaf), 1: FB (fruit+bark)
        self.head_fl = nn.Linear(feat_dim, 2)  # [flower, leaf]
        self.head_fb = nn.Linear(feat_dim, 2)  # [fruit, bark]

    def forward(self, x, temperature: float = 1.0):
        feats = self.backbone(x)
        gate_logits = self.gate(feats) / temperature
        fl_logits   = self.head_fl(feats) / temperature
        fb_logits   = self.head_fb(feats) / temperature
        g    = F.softmax(gate_logits, dim=1)[:, 0]  # P(FL)
        p_fl = F.softmax(fl_logits,   dim=1)        # [flower, leaf]
        p_fb = F.softmax(fb_logits,   dim=1)        # [fruit, bark]
        P = torch.stack([
            g * p_fl[:, 0],          # flower
            g * p_fl[:, 1],          # leaf
            (1 - g) * p_fb[:, 0],    # fruit
            (1 - g) * p_fb[:, 1],    # bark
        ], dim=1).clamp_min(1e-8)    # [B,4]
        return P

# =====================
# 2) 仅推理用 eval transform
# =====================

def build_eval_tf(organ: str, img_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    ])

# =====================
# 3) 推理骨干（只返回 model.eval, val_tf, img_size）
#    与训练的选择保持一致；如训练时有不同，请同步这里
# =====================

def get_infer_backbone_for_organ(organ: str, num_classes: int) -> Tuple[nn.Module, transforms.Compose, int]:
    organ = str(organ).lower().strip()
    img_size = 384

    if organ == "flower":
        # 升级为 ConvNeXt-Small（若环境不支持可改回 Tiny）
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1
        model   = models.convnext_small(weights=weights)
        in_feat = model.classifier[2].in_features
        #model.classifier[2] = nn.Linear(in_feat, num_classes)
        model.classifier[2] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "leaf":
        weights = ResNet50_Weights.IMAGENET1K_V1
        model   = models.resnet50(weights=weights)
        in_feat = model.fc.in_features
        #model.fc = nn.Linear(in_feat, num_classes)
        model.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "fruit":
        # 也可试 ConvNeXt-Tiny；先保留 B0 以兼顾速度
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model   = models.efficientnet_b0(weights=weights)
        in_feat = model.classifier[1].in_features
        #model.classifier[1] = nn.Linear(in_feat, num_classes)
        model.classifier[1] = nn.Sequential(nn.Dropout(0.30), nn.Linear(in_feat, num_classes))

    elif organ == "bark":
        # 关键改动：从 MobileNetV3-Small 换到 ConvNeXt-Tiny（或 Small）
        weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model   = models.convnext_tiny(weights=weights)
        in_feat = model.classifier[2].in_features
        #model.classifier[2] = nn.Linear(in_feat, num_classes)
        model.classifier[2] = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_feat, num_classes))
        
    else:
        raise ValueError(f"未知器官: {organ}")

    val_tf = build_eval_tf(organ, img_size)
    return model.to(DEVICE).eval(), val_tf, img_size

# =====================
# 4) 保护性断言：权重与头部维度匹配
# =====================

def _infer_last_linear(module: nn.Module) -> nn.Linear | None:
    # 在常见结构中找最后一个 Linear 头
    last = None
    for m in module.modules():
        if isinstance(m, nn.Linear):
            last = m
    return last


def assert_state_dict_compat(model: nn.Module, sd: Dict[str, torch.Tensor], num_classes: int):
    try:
        # 快速路径：根据权重里最后一个 Linear 的 out_features 判定
        keys = [k for k in sd.keys() if k.endswith("weight") and sd[k].dim() == 2]
        if keys:
            w = sd[keys[-1]]
            if w.size(0) != num_classes:
                raise AssertionError(f"state_dict 最后一层 out_features={w.size(0)} 与 num_classes={num_classes} 不一致")
    except Exception:
        # 回退：从 model 上拿最后线性层
        head = _infer_last_linear(model)
        if head is not None and getattr(head, 'out_features', None) != num_classes:
            raise AssertionError(
                f"模型头部 out_features={head.out_features} 与 num_classes={num_classes} 不一致")

# =====================
# 5) 器官分类器加载（pack 格式）
# =====================

def load_organ_classifier(path: Path | str = OUT_DIR / "organ_classification_mybest.pth"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"器官分类权重不存在: {path}")

    pack = torch.load(path, map_location=DEVICE)

    model = OrgansHier(
        backbone=pack.get("backbone", "convnext_base"),
        pretrained=False,
        drop_path_rate=pack.get("drop_path_rate", 0.3)
    ).to(DEVICE)
    model.load_state_dict(pack["state_dict"], strict=False)
    model.eval()

    id2organ = {v: k for k, v in pack["organ2id"].items()}
    cfg = pack.get("cfg", {})
    temperature = float(pack.get("temperature", 1.0))

    img_size = int(cfg.get("img_size", 384))
    tfm = transforms.Compose([
        transforms.Resize(int(img_size * 1.05)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
    ])
    return model, id2organ, tfm, temperature

# =====================
# 6) 器官 Top-K 预测（软路由用）
# =====================
@torch.no_grad()
def organ_predict_topk(
    image_paths,
    model: nn.Module,
    tfm: transforms.Compose,
    id2organ: Dict[int, str],
    temperature: float = 1.0,
    k: int = 2,
):
    # 归一化
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    image_paths = [str(p) for p in image_paths]

    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    xs = []
    for p in image_paths:
        with Image.open(p) as im:
            img = im.convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)

    if next(model.parameters()).device != batch.device:
        model = model.to(batch.device)

    probs = model(batch, temperature=temperature)  # [B,4]
    topk = min(k, probs.size(1))
    confs_k, ids_k = torch.topk(probs, k=topk, dim=1)
    ids_k   = ids_k.cpu().tolist()
    confs_k = confs_k.cpu().tolist()

    top1_org  = [id2organ[row[0]] for row in ids_k]
    top1_conf = [row[0] for row in confs_k]
    return ids_k, confs_k, top1_org, top1_conf

# =====================
# 7) 物种模型缓存 & 加载（推理）
# =====================
_SPECIES_MODEL_CACHE: Dict[str, Tuple[nn.Module, transforms.Compose, Dict[int,str]]] = {}

def load_species_model_for_organ_cached(organ: str):
    organ = organ.lower().strip()
    if organ in _SPECIES_MODEL_CACHE:
        return _SPECIES_MODEL_CACHE[organ]

    map_local2global = OUT_DIR / f"species_local2global_{organ}.json"
    map_label2local  = OUT_DIR / f"species_local_map_{organ}.json"
    if not map_local2global.exists() or not map_label2local.exists():
        raise FileNotFoundError(f"缺少映射：{map_local2global} 或 {map_label2local}")

    local2gid   = json.load(open(map_local2global, "r", encoding="utf-8"))
    label2local = json.load(open(map_label2local,  "r", encoding="utf-8"))
    id2label = {int(v): k for k, v in label2local.items()}
    num_classes = len(local2gid)

    model, val_tfm, _ = get_infer_backbone_for_organ(organ, num_classes)

    weight_path = OUT_DIR / f"{organ}_species_model.pth"
    if not weight_path.exists():
        raise FileNotFoundError(f"物种模型权重不存在: {weight_path}")
    sd = torch.load(weight_path, map_location=DEVICE)
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd

    # 保护性断言：头维度匹配
    assert_state_dict_compat(model, sd, num_classes)

    model.load_state_dict(sd, strict=True)
    model = model.to(DEVICE).eval()

    _SPECIES_MODEL_CACHE[organ] = (model, val_tfm, id2label)
    return _SPECIES_MODEL_CACHE[organ]

# =====================
# 8) 批量物种预测（返回：list[dict: species->prob]）
# =====================
@torch.inference_mode()
def species_predict_batch_dict(image_paths: List[str], organ: str) -> List[Dict[str, float]]:
    model, tfm, id2label = load_species_model_for_organ_cached(organ)

    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    xs = []
    for p in image_paths:
        with Image.open(p) as im:
            img = im.convert("RGB")
        xs.append(tfm(img))
    batch = torch.stack(xs, dim=0).to(DEVICE)
    logits = model(batch)
    probs = F.softmax(logits, dim=1).cpu().numpy()  # [B, K_local]

    out: List[Dict[str, float]] = []
    for row in probs:
        d = { id2label[i]: float(row[i]) for i in range(len(row)) }
        out.append(d)
    return out

# =====================
# 9) 软路由融合：批量图片 → 物种
# =====================

def predict_species_for_batch_images_soft(
    image_paths,
    organ_topk: int = 2,
    tau_min: float = 0.10,         # 软路由：低置信度下限
    alpha_sharpen: float = 1.5,    # 软路由：幂次锐化
):
    if isinstance(image_paths, (str, Path)):
        image_paths = [str(image_paths)]
    image_paths = [str(p) for p in image_paths]

    for p in image_paths:
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Image not found: {p}")

    # 1) 器官 Top-k
    organ_model, id2organ, organ_tfm, organ_T = load_organ_classifier(OUT_DIR / "organ_classification_mybest.pth")
    ids_k, confs_k, top1_org_text, top1_conf = organ_predict_topk(
        image_paths, organ_model, organ_tfm, id2organ, temperature=organ_T, k=organ_topk
    )

    # 2) 软路由权重（下限 + 幂次 + 归一化）
    per_image_route = []
    for i in range(len(image_paths)):
        organ_ids = ids_k[i]   # [k]
        organ_ps  = confs_k[i] # [k]
        w = np.array([max(float(p), tau_min) for p in organ_ps], dtype=np.float32)
        w = w ** float(alpha_sharpen)
        w = w / (w.sum() + 1e-12)
        sel = list(zip(organ_ids, w.tolist()))  # [(organ_id, weight), ...]
        per_image_route.append(sel)

    # 3) 分桶：按器官聚合需要推断的图片索引
    buckets: Dict[str, List[int]] = {}
    for i, sels in enumerate(per_image_route):
        for oid, _ in sels:
            organ_txt = id2organ[int(oid)]
            buckets.setdefault(organ_txt, []).append(i)

    # 4) 每个器官桶一次性跑种类头
    sp_probs: Dict[str, List[Dict[str,float]]] = {}
    for organ_txt, idxs in buckets.items():
        paths = [image_paths[j] for j in idxs]
        sp_probs[organ_txt] = species_predict_batch_dict(paths, organ_txt)

    # 5) 融合：score(s) = sum_o w_o * p(s|o)
    final = []
    for i, imgp in enumerate(image_paths):
        agg: Dict[str, float] = {}
        for oid, w in per_image_route[i]:
            organ_txt = id2organ[int(oid)]
            idx_in_bucket = buckets[organ_txt].index(i)
            d = sp_probs[organ_txt][idx_in_bucket]
            for s, p in d.items():
                agg[s] = agg.get(s, 0.0) + w * p

        if not agg:
            final.append({
                "image": imgp,
                "organ_top1": top1_org_text[i],
                "organ_top1_conf": round(float(top1_conf[i]), 4),
                "final_species": None,
                "final_conf": 0.0,
                "top3": []
            })
            continue

        # 归一化 & Top3
        z = sum(agg.values()) + 1e-12
        for k in list(agg.keys()):
            agg[k] = agg[k] / z
        top = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:3]

        final.append({
            "image": imgp,
            "organ_top1": top1_org_text[i],
            "organ_top1_conf": round(float(top1_conf[i]), 4),
            "final_species": top[0][0],
            "final_conf": round(float(top[0][1]), 4),
            "top3": [(k, round(float(v), 4)) for k, v in top]
        })

    # 6) 可选：全批“投票”
    vote: Dict[str, float] = {}
    for item in final:
        if item["final_species"] is not None:
            vote[item["final_species"]] = vote.get(item["final_species"], 0.0) + item["final_conf"]
    tot = sum(vote.values()) + 1e-12
    if tot > 0:
        vote = {k: round(v/tot, 4) for k, v in vote.items()}
        final_best = max(vote.items(), key=lambda x: x[1])[0]
    else:
        vote = {}
        final_best = None

    return {
        "final_species": final_best,
        "vote_scores": vote,
        "details": final
    }


print("✅ 推理最小序列已就绪：重启后只需运行本文件即可开始预测。")


✅ 推理最小序列已就绪：重启后只需运行本文件即可开始预测。


In [10]:
# ====== 修正版 train_predict_species - 支持单张和批量输入 ======
def enhanced_train_predict_species(image_paths, prob_threshold=0.7, vote_threshold=0.7):
    """
    ResNet50植物识别函数 - 支持单张和批量输入，新增智能判断逻辑
    
    Args:
        image_paths: 单张图片路径(str) 或 多张图片路径列表(list)
        prob_threshold: 物种识别的置信度阈值
        vote_threshold: 病虫害判断的投票阈值
    
    新逻辑:
    - 单张输入：置信度>0.9输出top1，否则输出top3
    - 多张输入：如果top1都不同则按置信度排序，如果有相同则投票机制
    - 只要有植物图片就忽略非植物图片
    """
    
    # 统一转换为列表格式
    if isinstance(image_paths, str):
        image_paths = [image_paths]
        is_single_image = True
    else:
        is_single_image = False
    
    if not image_paths:
        print("❌ 输入图像列表为空")
        return
    
    # 限制最大批处理数量
    max_batch = 9
    image_paths = image_paths[:max_batch]

    
    valid_paths = []
    plant_results = []
    non_plant_count = 0
    
    #print(f"🔍 开始处理 {len(image_paths)} 张图像...")
    
    for i, path in enumerate(image_paths):
        print(f"\n📷 处理图像 {i+1}/{len(image_paths)}")
        
        # 路径检查
        if not os.path.exists(path):
            print(f"❌ 文件不存在: {path}")
            continue
        if os.path.isdir(path):
            print(f"❌ 路径是目录, 不是图片文件: {path}")
            continue
            
        # 1. 是否植物检测
        try:
            is_plant = train_is_plant_clip(path)
            if not is_plant:
                print(f"❌ 不是植物")
                #non_plant_count += 1
                continue
        except Exception as e:
            print(f"⚠️ 无法判断是否植物: {e}")
            continue
        
        valid_paths.append(path)
        
        try:
            # === 使用 Step 6 的软路由推理，一次只喂入当前图片 ===
            infer_out = predict_species_for_batch_images_soft(
                [path],                 # 单张走同一套软路由
                organ_topk=2,           # 与你在 Step 6 中一致
                tau_min=0.10,
                alpha_sharpen=1.5
            )
            det = infer_out["details"][0]if infer_out["details"] else {}   # 当前图片的融合结果

            # 器官名称与置信度（Top1 器官）
            organ_name = det.get("organ_top1", "unknown")
            organ_conf = det.get("organ_top1_conf", 0.0)

            # 物种 Top1 与 Top3（已是融合后的概率）
            species_pred = det.get("final_species", None)
            top3_list    = det.get("top3", [])   # 形如 [(name, prob), ...]
            top_prob     = det.get("final_conf", 0.0)

            # 置信度阈值判断（维持你原逻辑）
            '''if top_prob < prob_threshold or species_pred is None:
                print(f"⚠️ 无法识别（置信度过低: {top_prob:.4f}）")
                continue'''

            # 计算 species_id：优先用你现有的 id_to_species 反查
            species_id = None
            try:
                if 'id_to_species' in globals() and isinstance(id_to_species, dict):
                    # 反转成 {物种名: 全局ID}
                    species2id = {name: gid for gid, name in id_to_species.items()}
                    species_id = species2id.get(species_pred, None)
            except Exception:
                species_id = None
            if species_id is None:
                # 兜底：用 -1 表示未知（不改变后续流程）
                species_id = -1

            # 组装与原版一致的结果结构，并把 top3 透传
            result = {
                'path': path,
                'image_index': i + 1,          # 从 1 开始
                'organ': organ_name,
                'species': species_pred,
                'species_id': int(species_id),
                'confidence': float(top_prob),
                'top3': [(str(n), float(p)) for (n, p) in top3_list],
            }

            plant_results.append(result)
            print(f"✅ 植物识别成功（器官: {organ_name} | 物种: {species_pred} | 置信度: {top_prob:.4f}）")

        except Exception as e:
            print(f"❌ 处理失败: {e}")
            continue

    
    # 输出非植物图片统计
    if non_plant_count > 0:
        print(f"\n📊 跳过了 {non_plant_count} 张非植物图片")
    
    # 如果没有有效的植物识别结果
    if not plant_results:
        print("❌ 没有成功识别的植物图像")
        return 

        print(f"\n🌿 成功识别 {len(plant_results)} 张植物图像")
    
    #print(f"\n🌿 成功识别 {len(plant_results)} 张植物图像")
    
    # 🆕 新的判断逻辑
    if is_single_image:
        # 单张图片逻辑
        result = plant_results[0]
        print(f"\n🔍 识别为器官: {result['organ']}")
        
        if result['confidence'] > 0.7:
            # 置信度 > 0.9，只输出 top1
            print(f"🌿 植物种类: {result['species']}（置信度: {result['confidence']:.4f}）")
        elif 0.1 <= result['confidence'] < 0.7:
            # 置信度 <= 0.9，输出 top3
            print(f"🌿 植物种类候选（置信度 {result['confidence']:.4f}）:")
            for j, (name, conf) in enumerate(result['top3']):
                print(f"   Top{j+1}: {name} (置信度: {conf:.4f})")
                

    else:
        # 多张图片逻辑
        if len(plant_results) == 1:
            # 只有一张植物图片
            result = plant_results[0]
            print(f"\n🌿 唯一植物图片识别结果:")
            print(f"🔍 第{result['image_index']}张图片 - 器官: {result['organ']}")
            print(f"🌿 第{result['image_index']}张图片 - 植物种类: {result['species']}（置信度: {result['confidence']:.4f}）")
        else:
            # 多张植物图片
            print(f"\n🌼 批量识别结果分析:")
            
            # 检查是否有重复的 top1 物种
            top1_species = [r['species'] for r in plant_results]
            unique_species = list(set(top1_species))
            
            if len(unique_species) == len(plant_results):
                # 所有图片的 top1 都不同，按置信度排序输出
                print("🔍 每张图片识别为不同植物种类，按置信度排序:")
                sorted_results = sorted(plant_results, key=lambda x: x['confidence'], reverse=True)
                
                for rank, result in enumerate(sorted_results, 1):
                    print(f"  Top{rank}: 第{result['image_index']}张图片 - {result['species']}（置信度: {result['confidence']:.4f}）")
                    
            else:
                # 至少有两张图片的 top1 相同，使用投票机制
                print("🗳️ 检测到相同植物种类，使用投票机制输出Top3:")
                
                from collections import Counter
                species_votes = Counter([r['species'] for r in plant_results])
                species_confidence = {}
                species_images = {}
                
                # 计算每个物种的平均置信度和对应图片
                for species in species_votes.keys():
                    confidences = [r['confidence'] for r in plant_results if r['species'] == species]
                    images = [r['image_index'] for r in plant_results if r['species'] == species]
                    species_confidence[species] = sum(confidences) / len(confidences)
                    species_images[species] = images
                
                # 按票数排序，票数相同按置信度排序
                sorted_species = sorted(species_votes.items(), 
                                      key=lambda x: (x[1], species_confidence[x[0]]), 
                                      reverse=True)
                
                for rank, (species, votes) in enumerate(sorted_species[:3], 1):
                    avg_conf = species_confidence[species]
                    images = species_images[species]
                    images_str = "、".join([f"第{img}张" for img in images])
                    print(f"  Top{rank}: {species} - {votes}票 (置信度: {avg_conf:.4f}) [{images_str}图片]")
                
                # 使用票数最多的物种作为最终结果
                final_species = sorted_species[0][0]
                print(f"\n🏆 最终识别结果: {final_species}")
    
    # 病虫害检测（对所有有效图片）
    if valid_paths:
        try:
            is_diseased = is_diseased_train(valid_paths, vote_threshold)
            if is_single_image:
                print(f"🦠 病虫害判断: {'是' if is_diseased else '否'}")
            else:
                if len(plant_results) == 1:
                    species_name = plant_results[0]['species']
                else:
                    # 多张图片时使用最终结果
                    top1_species = [r['species'] for r in plant_results]
                    unique_species = list(set(top1_species))
                    if len(unique_species) == len(plant_results):
                        # 所有不同，使用置信度最高的
                        species_name = max(plant_results, key=lambda x: x['confidence'])['species']
                    else:
                        # 有相同的，使用投票最多的
                        from collections import Counter
                        species_votes = Counter([r['species'] for r in plant_results])
                        species_name = species_votes.most_common(1)[0][0]
                
                print(f"🦠 病虫害判断（{species_name}）: {'是' if is_diseased else '否'}（基于 {len(valid_paths)} 张植物图像）")
        except Exception as e:
            print(f"⚠️ 病虫害判断失败: {e}")
    
    #print(f"\n✅ 处理完成！共处理 {len(image_paths)} 张图像，识别出 {len(plant_results)} 张植物图像")
    #return plant_results


In [13]:

# 有非植物、2张在数据集得植物，但不同种类
enhanced_train_predict_species(['/mnt/e/code/plants-classification-conda/test_car.png', 
'/mnt/e/test/anethum-graveolens.webp', 
'test_997_Populus_tremula.png'],
)



📷 处理图像 1/3
❌ 不是植物

📷 处理图像 2/3


/tmp/ipykernel_7252/562704043.py:152: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pack = torch.load(path, map_location=DEVICE)


✅ 植物识别成功（器官: flower | 物种: Anethum graveolens | 置信度: 0.9376）

📷 处理图像 3/3
✅ 植物识别成功（器官: leaf | 物种: Ficus carica | 置信度: 0.5346）

🌼 批量识别结果分析:
🔍 每张图片识别为不同植物种类，按置信度排序:
  Top1: 第2张图片 - Anethum graveolens（置信度: 0.9376）
  Top2: 第3张图片 - Ficus carica（置信度: 0.5346）
🦠 病虫害判断（Anethum graveolens）: 否（基于 2 张植物图像）


✅ 创建species_new文件夹: real_data/species_new
